# Resonator Coil Characterisation — a real coil, outside any trap

**Standalone.** No imports from any other notebook, so it runs on a bare board.

Every stage from 8 upward measured a resonator whose answer was chosen in software. This one
measures a physical coil, where nothing is known in advance. Three safeguards replace that:

| Replacement | Catches |
|---|---|
| Self-test against the emulator, before and after | Any fault in the instrument. The emulator is a register bit, not a cable, so it can be switched in at any moment |
| The same width measured two ways | A fit that is wrong for a reason the fit cannot see — sloping baseline, stray tone, compression |
| A signal budget computed *and* measured | Coupling and loading problems, which on a bench look exactly like physics |

**Nothing is carried over between runs.** The amplifier's gain, the usable drive, the decimation,
the coarse search step, every plot limit and every feedback sweep span are measured or derived at
run time — a resonator behaves differently in the trap and out of it, warm and cold, so a
constant copied from a previous session is a wrong answer waiting to be believed.

**It cannot tell you anything that needs an ion**, and nothing from Stage 17.

**Before running:** Stage 12 first, while the emulator is still the authority. Then this once with
`COIL_CONNECTED = False`, to catalogue the board's own stray signals — searching a band you have
not surveyed is how a stray signal becomes a discovery. As root: everything reads `/dev/mem`.

| Step | Question |
|---|---|
| 0 | Is the right bitstream loaded, and does everything answer? |
| 1 | Does the instrument still return a known answer? |
| 2 | Are the levels sane, and what else is in the band? |
| 3 | Where is the resonance, and how sharp? |
| 4a–4d | How much gain, how much drive, how much signal, and does it match L and C? |
| 5 | Does a second, independent method agree? |
| 6 | Is the measurement linear? |
| 7 | Does anything move while I measure? |
| 8 | Does feedback change Q and centre as it should? |
| 9 | Summary, and the constants later notebooks need |


---
# Wiring

```
                 Red Pitaya STEMlab 125-14
      +--------------------------------------------+
      |   OUT1  (SMA)                   IN1  (SMA) |
      +-----|--------------------------------|-----+
            |                                |
      [ZFL-500LN+]                           |  short coax
       28 dB, +15 V                          |
            |  short coax                    |
         C_drive                          C_pick
         0.3 pF                           0.3 pF
            |    +----------------------+    |
            +----|   the coil / tank    |----+
                 +----------------------+
                            |
                     C_parallel 39 pF
                            |
                     board ground (SMA shells)
```

**OUT1** carries the drive and the feedback, already summed, through the amplifier; **IN1**
carries everything measured; **OUT2/IN2** are unused; the **E1/E2 headers** must never see the
coil — 3.3 V logic, no protection.

One output cable, because the loop closes inside the chip: `dac_a_sum = asg_dat[0] + pid_dat[0]`
sums the drive and the phase-shifted copy before the converter, saturating rather than wrapping.
The emulator is a multiplexer in front of the phase shifter, so Step 1's self-test works with the
coil still attached.

Input jumper **LV** (±1 V). The input is ~1 MΩ — do not terminate it, and keep the coax short:
every picofarad lands in `C_STRAY_EST`.

## The current limit — the reason this is safe

At 41 MHz a 0.3 pF capacitor is 12.9 kΩ, and OUT1 delivers at most ±1 V, so the power it can put
into *any* load — matched, mismatched, shorted, open — is bounded by

    P_max = V² / (4 X_c) = 1 / (4 × 12850) ≈ 19 µW

No setting of `DRIVE_FRAC`, no feedback gain and no software fault gets past that, because the
limit is the capacitor and not the code. The software guards protect the *measurement* — a
clipped front end still looks vaguely like a resonance. The coil is protected by 0.3 pF.

> **The one change that removes this.** Replace either capacitor with a direct connection, or
> anything in the nanofarad range, and OUT1 sees 50 Ω instead of 13 kΩ: 18 mA and 8 mW, about
> 500× more, and a fine-wire coil can care about that. Recompute `DRIVE_MAX` from the coil's own
> dissipation limit *before* the first run.

## The amplifier

**Mini-Circuits ZFL-500LN+**, in the **drive** path: 0.1–500 MHz, ~28 dB at 40 MHz, +5 dBm
output, +15 V at 60 mA. It sits there because the drive port is already fed from OUT1's 50 Ω
source, so a 50 Ω amplifier costs nothing; the pickup port is high-impedance (~260 Ω at 40 MHz)
and a 50 Ω input would throw away 14 dB before any gain arrived.

Two limits, and conflating them is a mistake: **damage** is set by the amplifier's *input* power
(derated 20 dB into an open load) and caps `DRIVE_MAX`; **compression** is set by its *output*
power and caps `DRIVE_LINEAR_MAX`. They differ by the gain — a factor of nine here.
`EXT_GAIN_DB` is a starting guess; Step 4a measures the gain and adopts it.


---
# Configuration

**The only cell meant to be edited**, and these are starting points — the notebook measures and
replaces most of them as it runs. `SEARCH_LO`, `SEARCH_HI` and `DRIVE_FRAC` decide whether Step 3
finds anything at all.


In [ ]:
# ============================ CONFIGURATION ==================================
#  The only cell meant to be edited. These are STARTING POINTS - the notebook
#  measures and replaces most of them as it runs.

COIL_CONNECTED = True     # False = input terminated. Step 2 becomes a survey of
                          #   the board's OWN stray signals and Steps 3-8 are
                          #   skipped. Run it that way ONCE before ever attaching
                          #   the coil.
SOURCE_NOTE    = 'bench coil, 0.3 pF coupling caps both ports, room temperature'

AUTO_CALIBRATE = True     # Step 4a adopts the measured amplifier gain; Step 4b
                          #   adopts its recommended drive and re-measures the
                          #   resonance at it. False = print advice only, which
                          #   means every later number belongs to a drive that
                          #   was chosen rather than measured.

# ---- where to look for the resonance ---------------------------------------
SEARCH_LO      = 20.0e6   # [Hz] bottom of the search band
SEARCH_HI      = 55.0e6   # [Hz] top. Must stay below 62.5 MHz (Nyquist)
SEARCH_STEP    = 100e3    # [Hz] coarse grid, and the parameter that decides
                          #   whether Step 3 finds anything. It MUST be finer
                          #   than the narrowest plausible linewidth or the
                          #   coarse pass steps straight over the line.
                          #   HALVED AUTOMATICALLY and retried if nothing is
                          #   found, so a colder, higher-Q resonator needs no
                          #   edit here.
SEARCH_RETRIES = 3        # how many halvings before giving up
SEARCH_MIN_DB  = 6.0      # contrast a candidate must clear. NOT sufficient on
                          #   its own - see confirm_candidate() in Step 3.
SEARCH_MIN_CORR = 0.50    # two scans of the candidate must correlate this well.
                          #   THIS is what separates a resonance from noise; the
                          #   contrast figure above cannot.
SEARCH_FIT_MAX_RESID = 0.10
                          # Step 3 refuses a fit worse than this rather than
                          #   handing a non-Lorentzian down to Steps 4-8.

# ---- drive ------------------------------------------------------------------
DRIVE_FRAC     = 0.0032   # generator amplitude, 0..1 of full scale. A starting
                          #   value only: Step 4b measures the usable window and
                          #   replaces this.
DRIVE_LIN_FRACS = [0.5, 0.71, 1.0]     # Step 6 ladder, as FRACTIONS of
                          #   DRIVE_FRAC. Absolute values are a trap: once an
                          #   amplifier is fitted DRIVE_FRAC drops by 40 dB and a
                          #   fixed list would drive it past its rating. Only
                          #   6 dB of span, because a rung 12 dB below an already
                          #   low working drive sits in the noise floor, where the
                          #   fitted "peak" is floor rather than signal.

# ---- the hard ceiling -------------------------------------------------------
DRIVE_MAX_BASE = 0.9      # ceiling with NOTHING fitted. Every amplitude write is
                          #   checked against the derived DRIVE_MAX and refused if
                          #   larger. Must be in (0, 1.0].
assert all(0.0 < x <= 1.0 for x in DRIVE_LIN_FRACS), \
    'DRIVE_LIN_FRACS are fractions of DRIVE_FRAC and must be in (0, 1]'

# ---- receiver ---------------------------------------------------------------
DEC_SEARCH     = 8        # log2(R) during the coarse search. R = 256, +-49 kHz
                          #   clean band, 488 kHz output rate. Wide and quick.
DEC_FIT        = 10       # log2(R) for the careful sweeps. Only a starting point:
                          #   choose_dec() picks the decimation from the measured
                          #   linewidth every time a fit is refined.
N_AVG          = 64       # PEEK register pairs averaged per measurement point

# ---- external amplifier in the DRIVE path -----------------------------------
G_PASSIVE      = 0.0158   # round-trip transfer with NO amplifier fitted. Step 4a
                          #   infers the amplifier's OUTPUT level from what
                          #   arrives at the input, so this must be right. With
                          #   no amplifier declared, Step 4a re-measures it and
                          #   writes it back.
EXT_GAIN_DB    = 33.7     # [dB] gain of ANYTHING between OUT1 and the drive
                          #   capacitor; 0.0 if nothing is fitted. Step 4a
                          #   measures it and adopts the measured value.
#  TWO SEPARATE LIMITS, and conflating them is a mistake:
#    DAMAGE is set by the amplifier's INPUT power  -> caps DRIVE_MAX
#    COMPRESSION is set by its OUTPUT power        -> caps DRIVE_LINEAR_MAX
#  They differ by the gain, so for a ZFL-500LN+ they are a factor of nine apart.
#  Steps that only need the drive to EXIST (the Step 1 self-test, where the
#  emulator replaces the input) may use the damage limit; steps that measure the
#  coil must stay under the other.
EXT_PAD_DB      = 0.0     # [dB] fixed attenuation between OUT1 and the amplifier.
                          #   It decides whether SELF-OSCILLATION can be explored:
                          #   an oscillating loop puts FULL SCALE (+10 dBm) into
                          #   the DAC and gen.amp()'s ceiling does not bound the
                          #   feedback path, so the amplifier sees 10 - EXT_PAD_DB
                          #   dBm, which must stay under AMP_IN_MAX_DBM.
AMP_IN_MAX_DBM  = -15.0   # [dBm] max input for NO DAMAGE. +5 dBm normally, but
                          #   the datasheets derate by 20 dB with no load, and a
                          #   0.3 pF capacitor IS an open load at 40 MHz. Use +5
                          #   if you fit a 50 ohm terminator at the output.
AMP_OUT_MAX_DBM = 0.0     # [dBm] linear output ceiling. 0 dBm for a ZFL-500LN+
                          #   into an open load: its +7.9 dBm compression figure
                          #   is into 50 ohm and an open load doubles the voltage.
AMP_TEST_START = 0.001    # first drive of the ladder. Safe for any gain up to
                          #   40 dB: OUT1 is -44 dBm here.
AMP_TEST_STEP  = 1.6      # multiplier per rung, about 4 dB

# ---- Step 4b: finding the drive window --------------------------------------
DRIVE_SCAN_N   = 7        # rungs in the drive-window scan
FIT_TRUST_TOL  = 0.02     # the emulator's Q must still be recovered this well for
                          #   a signal level to count as fittable

GUARD_NSAMP    = 256      # scope samples per input-level check. The guard runs at
                          #   every swept point, so it dominates a sweep's time.
SETTLE         = 0.02     # [s] wait after any retune, for the filter pipeline

# ---- the coupling and the tank ----------------------------------------------
C_COUPLE       = 0.3e-12  # [F] the coupling capacitors actually fitted
#  A transfer measurement determines (C_COUPLE + C_stray) and nothing else, so
#  the split below rests on C_COUPLE being its marked value and scales with it.
#  Step 4d prints the measurement that would settle it.
C_BOARD_IN     = 11.5e-12 # [F] the board's own input capacitance. Derived, not
                          #   measured - see above.
C_CABLE_PICKUP = 13.0e-12 # [F] the pickup cable. From the two-length experiment:
                          #   doubling it moved the transfer by -3.7 dB, which
                          #   requires 13.0 pF per length. ~13 cm at 100 pF/m.
C_STRAY_EST    = C_BOARD_IN + C_CABLE_PICKUP
                          # [F] the sum is what the transfer constrains.
L_COIL         = 423.5e-9   # [H] coil inductance
L_COIL_ERR     = 11.8e-9    # [H] its uncertainty
C_COIL         = 1.59e-12   # [F] the coil's own self-capacitance
C_COIL_ERR     = 0.06e-12   # [F] its uncertainty

C_PARALLEL     = 39.0e-12   # [F] the capacitor fitted ACROSS the tank to emulate
                            #   the trap's parasitic capacitance. Without it the
                            #   coil sits near 165 MHz, above Nyquist, and is
                            #   invisible to this board.
C_PARALLEL_ERR = 2.0e-12    # [F] its tolerance. A 5% part on 39 pF is +-1.95 pF
                            #   and it DOMINATES the predicted centre frequency -
                            #   25x the coil's own capacitance. If you know the
                            #   part is 1% or 2%, put that in: Step 4d's agreement
                            #   test is only as tight as this.

L_EST          = L_COIL   # [H] used by Step 4c's thermal-noise estimate. None to
                          #   skip that block rather than guess.
R_IN_BOARD     = 1.0e6    # [ohm] the board's input resistance
V_FULLSCALE    = 1.0      # [V] input full scale. 1.0 on LV, 20.0 on HV. Used only
                          #   to print volts instead of counts.
T_COIL         = 293.0    # [K] coil temperature, for the thermal-noise estimate

# ---- Step 7: drift -----------------------------------------------------------
DRIFT_MINUTES  = 10.0     # total observation time
DRIFT_PERIOD   = 20.0     # [s] between revisits

# ---- Step 8: feedback --------------------------------------------------------
FB_PHASES      = list(range(0, 360, 20))   # [deg] phase axis. Must stay
                          #   symmetric - every phase needs its opposite present,
                          #   because Step 8 subtracts the two.
FB_GAIN_MAX    = 1.90     # ceiling on the feedback coefficient. The register is
                          #   Q2.14 and A x 16384 must fit in int16: at A = 2.0 the
                          #   coefficient wraps to -32768 and the feedback SILENTLY
                          #   INVERTS. PhaseShift.set() refuses 1.99 and above.
FB_GAIN_PROBE  = 1.50     # gain used to FIND the two special phases. High on
                          #   purpose; the input level guard is still checked.
FB_GAIN_FRACS  = [0.15, 0.3, 0.45, 0.6, 0.75, 0.9, 1.0]
                          # gains for Step 8b as FRACTIONS of the usable ceiling.
                          #   Absolute values get filtered out entirely whenever
                          #   the coupling improves and the ceiling drops.
FB_NAVG        = 96       # averages per point in Step 8. Higher than anywhere
                          #   else: the signature is a few percent of Q, so the
                          #   fit noise has to go below it.

# ---- Step 8c: the gain/phase map --------------------------------------------
FB_MAP_PHASES  = 24       # phase steps across 360 degrees
FB_MAP_GAINS   = 12       # gain steps from zero to the usable ceiling
FB_MAP_NPTS    = 15       # frequency points per pixel
FB_MAP_NAVG    = 32       # averages per point. Lower than the 1D sweep: on the
                          #   map the effect is tens of percent, not one percent.
FB_MAP_SETTLE  = 0.010    # [s] per point
FB_OSC_LEVEL   = 0.10     # input fraction, DRIVE OFF, above which the loop is
                          #   judged to be self-oscillating. Those pixels are left
                          #   blank and drawn black, as in the paper.

# ---- Step 8d: the lineshape family, positive and negative feedback ----------
FB_SPEC_FRACS  = [-0.95, -0.70, -0.40, 0.0, 0.40, 0.70, 0.90, 0.95]
                          # fractions of the usable ceiling. NEGATIVE means the
                          #   opposite phase - the coefficient register cannot go
                          #   negative, but phi + 180 is the same thing.
FB_SPEC_NPTS   = 61       # points per lineshape
FB_SPEC_HALFSPAN = 3.5    # sweep half-span in linewidths OF THAT SETTING, not of
                          #   the unfed line. Feedback narrows the line by 1/(1-K),
                          #   so a fixed span puts three points across the
                          #   narrowest curve and the Lorentzian stops looking like
                          #   one. Also sets panel (a)'s x-limits.
FB_SPEC_ZOOM_LW = 2.5     # panel (b)'s half-width, in linewidths of the UNFED
                          #   line. That curve is the reference the whole family
                          #   is quoted against, so the zoom is built around it
                          #   rather than around whichever setting happened to
                          #   come out narrowest.

# ---- Step 8e: how far and how accurately the resonance can be moved ---------
FB_SHIFT_FRACS = [0.2, 0.4, 0.6, 0.8, 0.95]
FB_SHIFT_TESTS = 3        # target shifts to request and then verify

FB_LEVEL_GUARD = 0.80     # abort the point if the input exceeds this fraction of
                          #   full scale. THE REAL PATH HAS NO OVERLOAD FLAG - the
                          #   emulator's sticky CLIPPED bit has quietly been doing
                          #   that job for ten stages and it is gone here.

# ---- Step 1: instrument self-test -------------------------------------------
SELFTEST_F0    = 37.0e6   # deliberately NOT 40 MHz, so a hard-coded 40 MHz
SELFTEST_Q     = 250.0    #   anywhere in the analysis shows up as a failure
SELFTEST_TOL_F = 0.20     # tolerance: fraction of a linewidth on the centre
SELFTEST_TOL_Q = 0.05     # tolerance: fractional error on Q

RESULTS_JSON   = 'bench_coil_results.json'   # written by Step 9

FS_CLK         = 125e6    # the one clock everything on the board runs from [Hz]

# ---- derived ceilings -------------------------------------------------------
#  External gain sits outside anything the software can see, so fold it into the
#  ceiling here. With an amplifier fitted, DRIVE_MAX is set by what the AMPLIFIER
#  can survive, not by what the converter can. Step 4a recomputes both from the
#  MEASURED gain.
def derive_limits(gain_db, pad_db, in_max_dbm, out_max_dbm, base=None):
    """(DRIVE_MAX, DRIVE_LINEAR_MAX) for a declared external chain.

    A pad in front of the amplifier raises BOTH limits by its value, and it is
    the only way to give the converter a decent number of counts: the output
    ceiling caps the delivered power either way, so without a pad the DAC ends up
    running at a couple of dozen counts."""
    base = DRIVE_MAX_BASE if base is None else base
    if gain_db <= 0.0:
        return base, base
    dmax = min(base, 10.0 ** ((in_max_dbm + pad_db - 10.0) / 20.0))
    dlin = 10.0 ** ((out_max_dbm - gain_db + pad_db - 10.0) / 20.0)
    return dmax, dlin


DRIVE_MAX, DRIVE_LINEAR_MAX = derive_limits(
    EXT_GAIN_DB, EXT_PAD_DB, AMP_IN_MAX_DBM, AMP_OUT_MAX_DBM)
if EXT_GAIN_DB > 0.0:
    print('external gain %.1f dB declared: DRIVE_MAX %.4f (damage), linear limit '
          '%.5f (compression)' % (EXT_GAIN_DB, DRIVE_MAX, DRIVE_LINEAR_MAX))
    if DRIVE_FRAC > DRIVE_LINEAR_MAX:
        print('  NOTE: DRIVE_FRAC %.5f is above the linear limit. Safe, but the'
              % DRIVE_FRAC)
        print('  amplifier will compress. Step 4b measures the usable window and')
        print('  replaces it, so this is a starting value rather than a problem.')

# The Step 1 self-test drives the EMULATOR, which replaces the digitised input,
# so the amplifier plays no part in what it measures - only the damage limit
# applies. It needs a large signal to hit its 5% tolerance on Q.
SELFTEST_DRIVE = min(0.3, 0.9 * DRIVE_MAX)

assert 0.0 < DRIVE_MAX <= 1.0, 'DRIVE_MAX must be in (0, 1.0]'
assert 0.0 <= DRIVE_FRAC <= DRIVE_MAX, (
    'DRIVE_FRAC %.5f exceeds DRIVE_MAX %.5f. With %.1f dB of external gain the '
    'amplifier would be driven far past its rating. Use about %.5f.'
    % (DRIVE_FRAC, DRIVE_MAX, EXT_GAIN_DB, 0.5 * DRIVE_MAX))
assert 0.0 < FB_GAIN_MAX < 1.99, 'FB_GAIN_MAX must be below the 1.99 register limit'
assert 0.0 <= FB_GAIN_PROBE <= FB_GAIN_MAX, 'FB_GAIN_PROBE exceeds FB_GAIN_MAX'
assert all(0.0 < fr <= 1.0 for fr in FB_GAIN_FRACS), \
    'FB_GAIN_FRACS are fractions of the usable ceiling and must be in (0, 1]'
assert len(FB_PHASES) % 2 == 0 and all(((p + 180) % 360) in FB_PHASES for p in FB_PHASES), \
    'FB_PHASES must contain the opposite of every phase it contains'

print('configuration loaded. coil connected: %s' % COIL_CONNECTED)
print('search band %.1f - %.1f MHz, coarse step %.0f kHz (halved up to %d times '
      'if nothing is found)'
      % (SEARCH_LO / 1e6, SEARCH_HI / 1e6, SEARCH_STEP / 1e3, SEARCH_RETRIES))
print('auto-calibration: %s' % ('ON - measured gain and drive are adopted'
                                if AUTO_CALIBRATE else 'OFF - advice only'))


---
# Imports, and the one rule that must not be broken

**Every register access is a single 32-bit read or write.** A bulk `mmap` slice becomes a burst
the bridge to the register bus cannot acknowledge: it locks the bus and reboots the board — no
traceback, no log. The loops below look inefficient on purpose.

DDR is different — bulk reads *there* are fine and fast, which is why the capture routine reads
its records in one slice while the register drivers never do.

The two shims exist because this board carries older numpy and matplotlib than a desktop.


In [ ]:
import os, mmap, struct, time, json
import numpy as np
import matplotlib
import matplotlib.pyplot as plt


def s32(v):
    """Reinterpret a 32-bit register value as a signed integer."""
    return v - (1 << 32) if v & 0x80000000 else v


class Bus(object):
    """One register block, accessed strictly one 32-bit word at a time."""

    def __init__(self, base, span=0x1000):
        self.base = base
        self.fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
        self.m = mmap.mmap(self.fd, span, offset=base)

    def w(self, off, val):
        self.m[off:off + 4] = struct.pack('<I', val & 0xFFFFFFFF)

    def r(self, off):
        return struct.unpack('<I', self.m[off:off + 4])[0]

    def rs(self, off):
        return s32(self.r(off))


# =============================================================================
#  PLOTTING COMPATIBILITY  -  this board runs matplotlib 1.x
# =============================================================================
#  The 'C0'..'C9' colour shorthand arrived in matplotlib 2.0. On 1.x it is not
#  rejected when you pass it - it is accepted, stored, and then raises
#
#      ValueError: to_rgba: Invalid rgba arg "C3"
#
#  from inside the draw, at which point the figure is already built. What you see
#  is a traceback followed by a bare <matplotlib.figure.Figure at 0x...> instead
#  of a plot, and every OTHER cell still runs, so it looks like a plotting
#  nuisance rather than a version problem. Every colour in this notebook is
#  therefore written out in full.
CLR_BLUE   = '#1f77b4'
CLR_ORANGE = '#ff7f0e'
CLR_GREEN  = '#2ca02c'
CLR_RED    = '#d62728'
CLR_GREY   = '0.5'

# The paper's figures use the modern perceptual colour maps; matplotlib 1.x has
# none of them, so viridis is rebuilt here from its own anchors - identical
# output on any version.
VIRIDIS_ANCHORS = [
    (0.2670, 0.0049, 0.3294), (0.2823, 0.0950, 0.4173), (0.2788, 0.1755, 0.4834),
    (0.2590, 0.2515, 0.5247), (0.2297, 0.3224, 0.5457), (0.1994, 0.3876, 0.5546),
    (0.1727, 0.4488, 0.5579), (0.1490, 0.5081, 0.5573), (0.1276, 0.5669, 0.5506),
    (0.1206, 0.6258, 0.5335), (0.1579, 0.6838, 0.5017), (0.2461, 0.7389, 0.4520),
    (0.3692, 0.7889, 0.3829), (0.5160, 0.8312, 0.2943), (0.6785, 0.8637, 0.1895),
    (0.8456, 0.8873, 0.0997), (0.9932, 0.9062, 0.1439)]
CMAP_SEQ = matplotlib.colors.LinearSegmentedColormap.from_list(
    'viridis_local', VIRIDIS_ANCHORS, N=256)
CMAP_DIV = matplotlib.colors.LinearSegmentedColormap.from_list(
    'coolwarm_local',
    [(0.230, 0.299, 0.754), (0.553, 0.690, 0.996), (0.866, 0.866, 0.866),
     (0.968, 0.718, 0.612), (0.706, 0.016, 0.150)], N=256)
CMAP_SEQ.set_bad('black')
CMAP_DIV.set_bad('black')

# One place for the look of every figure. Setting it here means no cell needs to
# repeat grid, font and layout calls.
plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 110, 'figure.autolayout': True,
    'font.size': 9, 'axes.titlesize': 10, 'axes.labelsize': 9,
    'legend.fontsize': 8, 'legend.framealpha': 0.9,
    'xtick.labelsize': 8, 'ytick.labelsize': 8,
    'axes.grid': True, 'grid.color': '0.88', 'grid.linewidth': 0.6,
    'axes.axisbelow': True, 'lines.linewidth': 1.2, 'lines.markersize': 4,
})

try:                       # inline figures when running under Jupyter...
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:          # ...and a file-only backend when running as a script
    matplotlib.use('Agg')

print('numpy %s, matplotlib %s' % (np.__version__, matplotlib.__version__))
if int(matplotlib.__version__.split('.')[0]) < 2:
    print('  matplotlib 1.x detected - using explicit hex colours (see above)')


---
# The four hardware blocks

| Block | Base | Role here |
|---|---|---|
| **Acquisition** | `0x40600000` | The receiver: a narrowband voltmeter (registers) and a recorder (memory) |
| **Generator** | `0x40200000` | The stimulus: a stored sine replayed at a chosen rate |
| **Phase shifter** | `0x40300000` | The feedback controller, Step 8 only |
| **Digital resonator** | `0x40400000` | The emulator — a known answer, in Steps 1, 4b and 9a |
| **Scope** | `0x40100000` | Full-rate capture of the raw input, for level checks |

Full register maps are in `acquisition_custom/README.md` and in each Verilog header.

**The trick that makes every sweep simple:** tune the receiver's local oscillator to *exactly*
the drive frequency, and the signal lands at 0 Hz where the in-phase and quadrature values are
steady numbers readable from two registers at leisure. At any other offset they rotate, and two
register reads land on different samples — an artefact that once convincingly imitated filter
droop in this project. "Exactly" is doable rather than approximate: the generator's frequency
word has 30 bits, the oscillator's 32, and both count the same 125 MHz clock, so an oscillator
word of `4 × step` is the same frequency identically, not to within a rounding error.


In [ ]:
# =============================================================================
#  ACQUISITION SYSTEM - base 0x40600000
# =============================================================================
#  VOLTMETER: tune the local oscillator to the drive, read PEEK_I/PEEK_Q.
#  RECORDER : stream to memory for spectra.
#  ORDER MATTERS: disable -> write settings -> reset -> enable. The decimator
#  holds a running total that assumes a fixed R. Retuning the oscillator is one
#  register write and is safe at any time.
# =============================================================================
class Acq(Bus):

    def __init__(self):
        Bus.__init__(self, 0x40600000)
        self.dec = None

    # ---- setup ----
    def start(self, dec_log2, src=0):
        """Bring the chain up at a given decimation. Safe to call repeatedly."""
        self.w(0x00, 0)                 # disable while settings change
        self.w(0x08, dec_log2)          # decimation R = 2**dec_log2
        self.w(0x0C, src)               # 0 = IN1, 1 = IN2
        self.w(0x00, 1 | 2)             # enable + soft reset (self-clearing)
        self.w(0x00, 1)                 # running
        self.dec = dec_log2

    def stop(self):
        self.w(0x00, 0)

    # ---- tuning ----
    def tune_word(self, finc):
        """Set the local oscillator by its raw 32-bit frequency word."""
        self.w(0x04, int(finc) & 0xFFFFFFFF)

    def tune(self, f):
        """Set the local oscillator to the nearest word to f hertz."""
        self.tune_word(int(round(f / FS_CLK * (1 << 32))))

    # ---- voltmeter ----
    def measure(self, navg=None):
        """Average navg in-phase/quadrature pairs; return one complex number.

        At DC both are steady, so this is coherent averaging: a real signal adds
        linearly while noise averages down as the square root of navg."""
        navg = N_AVG if navg is None else navg
        zi = 0.0
        zq = 0.0
        for _ in range(navg):
            zi += self.rs(0x18)
            zq += self.rs(0x1C)
        return complex(zi / navg, zq / navg)

    # ---- housekeeping ----
    def f_out(self):
        """Output sample rate at the current decimation [Hz]."""
        return FS_CLK / (1 << self.dec)

    def window_hz(self):
        """Half-width of the CLEAN part of the receiver window [Hz].

        The decimator rejects by better than 75 dB only over the middle fifth of
        its output band, so the usable half-width is 0.1 x the output rate."""
        return 0.10 * self.f_out()

    def build_id(self):
        return self.r(0x48)

    def status2(self):
        return self.r(0x38)


acq = Acq()
print('acquisition mapped. BUILD_ID = 0x%08X' % acq.build_id())


In [ ]:
# =============================================================================
#  SIGNAL GENERATOR - base 0x40200000, stock Red Pitaya module
# =============================================================================
#  Replays a 16384-entry table; frequency is one register write.
#      f = step x 125e6 / 2**30      (about 0.116 Hz per step)
#  Registers: 0x04 amplitude (0x2000 = x1.0), 0x08 table size, 0x0C start,
#  0x10 step, table at 0x10000. Control 0x00: 0x11 free-run, 0xC0 held at zero.
# =============================================================================
class Generator(Bus):

    def __init__(self):
        Bus.__init__(self, 0x40200000, span=0x20000)
        self._step = None

    def load_sine(self):
        """Write one cycle of a sine into the table. Takes a second or two:
        16384 single-word writes, and single-word is not negotiable here."""
        for i in range(16384):
            v = int(round(8191.0 * np.sin(2.0 * np.pi * i / 16384.0)))
            self.w(0x10000 + 4 * i, v & 0x3FFF)
        self.w(0x08, (16384 << 16) - 1)     # table size, fixed point, inclusive
        self.w(0x0C, 0)                     # start replaying from entry 0

    def freq_word(self, f):
        """Nearest 30-bit frequency word to f hertz."""
        return int(round(f / FS_CLK * (1 << 30)))

    def set_word(self, step):
        """Set frequency by raw word; return the frequency actually produced."""
        self._step = int(step)
        self.w(0x10, self._step)
        return self._step * FS_CLK / float(1 << 30)

    def freq(self, f):
        """Set frequency in hertz; return the frequency actually produced."""
        return self.set_word(self.freq_word(f))

    def lo_word(self):
        """The 32-bit local-oscillator word that lands this signal EXACTLY at
        0 Hz. The generator counts in units of 2**-30 of the clock and the local
        oscillator in units of 2**-32, so the conversion is a factor of four and
        carries no rounding error at all."""
        return 4 * self._step

    def amp(self, frac):
        """Output amplitude as a fraction of full scale. CLAMPED, deliberately.

        The register is Q1.13 with 0x2000 = x1.0, and it happily accepts up to
        0x3FFF = x1.9999. The write below is masked with & 0x3FFF, so before this
        guard existed the mask turned mistakes into silent, WRONG drive levels:

            frac = -0.5  ->  0x3000  ->  x1.5   (a sign slip drives 50% HARDER)
            frac =  2.0  ->  0x0000  ->  x0.0   (silence, and no complaint)
            frac =  1.5  ->  0x3000  ->  x1.5   (clipped square wave, not a sine)

        Every one of those is silent on the bench. Anything outside
        [0, DRIVE_MAX] is now refused instead, so the only way to drive the coil
        harder is to raise DRIVE_MAX in the configuration cell on purpose."""
        frac = float(frac)
        if not np.isfinite(frac):
            raise ValueError('drive amplitude is not a finite number: %r' % frac)
        if frac < 0.0:
            raise ValueError(
                'negative drive amplitude (%.4f). A sign has no meaning here, and '
                'the register would silently wrap it to x%.4f.'
                % (frac, (int(round(frac * 8192)) & 0x3FFF) / 8192.0))
        if frac > DRIVE_MAX:
            raise ValueError(
                'drive %.4f exceeds DRIVE_MAX = %.4f. Nothing in this notebook '
                'may drive harder than the ceiling; raise it in the configuration '
                'cell if that is really what you want.' % (frac, DRIVE_MAX))
        self.w(0x04, int(round(frac * 8192)) & 0x3FFF)

    def run(self):
        self.w(0x00, 0x40)      # reset the read pointer
        self.w(0x00, 0x11)      # free-run, wrap at the end of the table

    def off(self):
        self.amp(0.0)
        self.w(0x00, 0xC0)      # held in reset, output forced to zero


gen = Generator()
gen.off()
gen.load_sine()
print('generator ready, table loaded, output off')


In [ ]:
# =============================================================================
#  PHASE SHIFTER - base 0x40300000   (Step 8 only)
# =============================================================================
#  Output = A cos(phi) x I + A sin(phi) x Q, Q being the input delayed a quarter
#  period. Two corrections, both from the module header:
#   1. The quarter-period delay is under one clock at 40 MHz, so add whole signal
#      periods and pick the k whose fraction sits nearest 0 or 1.
#   2. Fixed THREE-sample latency (L = 3 since FIX 6; one sample is 115 deg at
#      40 MHz, so a stale value rotates the whole axis). Subtracted from the
#      request so the delivered phase is the requested one.
#  Both describe the DIGITAL path only. Step 8 finds its phases by measurement.
#  "Feedback off" is coefficients (0, 0), NOT the bypass bit.
# =============================================================================
class PhaseShift(Bus):

    LATENCY = 3      # datapath latency in samples (L = 3 since FIX 6; see above)

    def __init__(self):
        Bus.__init__(self, 0x40300000)
        self.f_ref = None

    def config_delay(self, f):
        """Programme the quarter-period delay line for frequency f."""
        best = None
        for k in range(0, 9):
            D = FS_CLK / (4.0 * f) + k * FS_CLK / f
            if D < 1.0 or D > 1000.0:
                continue
            frac = D - np.floor(D)
            score = min(frac, 1.0 - frac)       # interpolator error peaks at 0.5
            if best is None or score < best[0]:
                best = (score, int(np.floor(D)), frac)
        if best is None:
            raise ValueError('no usable quarter-period delay at %.3f MHz' % (f / 1e6))
        self.w(0x08, best[1])                       # whole clocks
        self.w(0x0C, int(round(best[2] * 16384)))   # fraction, Q0.14
        self.w(0x10, 0)                             # bypass OFF
        self.f_ref = f
        return best[1], best[2]

    def set(self, A, phi_deg):
        """Apply gain A and phase phi_deg (degrees)."""
        if self.f_ref is None:
            raise RuntimeError('call config_delay() first')
        if not (0.0 <= A < 1.99):
            # A x 16384 must stay inside int16. At A = 2.0 the coefficient is
            # 32768, which wraps to -32768 and silently INVERTS the feedback.
            raise ValueError('gain must be in [0, 1.99)')
        phi = phi_deg - 360.0 * self.LATENCY * self.f_ref / FS_CLK
        self.w(0x00, int(round(A * np.cos(np.radians(phi)) * 16384)) & 0xFFFF)
        self.w(0x04, int(round(A * np.sin(np.radians(phi)) * 16384)) & 0xFFFF)

    def off(self):
        self.w(0x00, 0)
        self.w(0x04, 0)


ps = PhaseShift()
ps.off()
print('phase shifter mapped, feedback off')


In [ ]:
# =============================================================================
#  DIGITAL RESONATOR (THE EMULATOR) - base 0x40400000
# =============================================================================
#  Used only in Step 1. Enabled, it REPLACES the digitised input, so the coil can
#  stay connected and simply be ignored - a known answer is available at any
#  moment without recabling.
#  Coefficients c = r cos(w0) x 2**17, s = r sin(w0) x 2**17, w0 = 2 pi f0/125e6,
#  r = 1 - pi (f0/125e6)/Q. Rounding moves the pole, so the QUANTISED values
#  recovered below are the real known answer.
# =============================================================================
class Emulator(Bus):

    def __init__(self):
        Bus.__init__(self, 0x40400000)
        self.f0_q = None
        self.q_q = None

    def config(self, f0, Q, delay=13, oshift=7):
        """Set the emulated resonance. FOUR coefficients, since 2026-08-07.

        The recursion advances two samples per update - two-way look-ahead, see
        the module header - so the pole it actually realises is the one in
        COEF_C2/COEF_S2: r squared, turned by 2*w0. COEF_C and COEF_S are still
        required, because they carry the drive term, so all four are written.

        The quantised answer returned therefore comes from c2 and s2, since that
        is the pole the hardware has. Comparing the Step 1 self-test against
        anything else would build a permanent error into its tolerance.

        `delay` is two lower than before: the module's own pipeline is now two
        clocks longer, so 13 + 2 = the board's real 15."""
        w0 = 2.0 * np.pi * f0 / FS_CLK
        r = 1.0 - np.pi * (f0 / FS_CLK) / Q
        ci = int(round(r * np.cos(w0) * 131072))
        cs = int(round(r * np.sin(w0) * 131072))
        c2 = int(round(r * r * np.cos(2.0 * w0) * 131072))
        s2 = int(round(r * r * np.sin(2.0 * w0) * 131072))
        self.w(0x04, ci & 0x3FFFF)
        self.w(0x08, cs & 0x3FFFF)
        self.w(0x28, c2 & 0x3FFFF)
        self.w(0x2C, s2 & 0x3FFFF)
        self.w(0x0C, delay)
        self.w(0x10, oshift)
        # arctan2 gives 2*w0 modulo 2*pi, and 2*w0 passes pi above 31.25 MHz,
        # so lift it onto the branch nearest the frequency asked for - which is
        # known here, because it is what produced the coefficients.
        r2 = np.hypot(c2, s2) / 131072.0
        two_w0 = np.arctan2(s2, c2)
        two_w0 += 2.0 * np.pi * round((2.0 * w0 - two_w0) / (2.0 * np.pi))
        self.f0_q = (two_w0 / 2.0) / (2.0 * np.pi) * FS_CLK
        self.q_q = np.pi * (self.f0_q / FS_CLK) / (1.0 - np.sqrt(r2))
        #  Prove the write took. COEF_C2 exists only in the four-coefficient
        #  bitstream; an older one does not decode 0x28 and returns zero, and
        #  then the recursion has no pole at all - the emulator becomes a
        #  pass-through with a flat response, which a Lorentzian fitter turns
        #  into a plausible-looking wrong answer rather than an error.
        if ((self.r(0x28) & 0x3FFFF) != (c2 & 0x3FFFF) or
                (self.r(0x2C) & 0x3FFFF) != (s2 & 0x3FFFF)):
            raise RuntimeError(
                'COEF_C2/COEF_S2 did not read back. This notebook writes the '
                'four-coefficient emulator (two-way look-ahead, 2026-08-07); '
                'the bitstream on the board does not have it. Rebuild and '
                'reload. NOTE the opposite mistake is SILENT: an old notebook '
                'on new hardware leaves c2 = s2 = 0, the recursion has no pole, '
                'and Step 1 reports a fitted Q near 1 with a centre outside the '
                'swept band.')
        return self.f0_q, self.q_q

    def enable(self):
        self.w(0x00, 1)

    def disable(self):
        self.w(0x00, 0)

    def clear(self):
        self.w(0x00, 3)          # zero the state and the sticky clip flag

    def enabled(self):
        return bool(self.r(0x00) & 1)

    def clipped(self):
        return bool(self.r(0x14) & 1)


emu = Emulator()
emu.disable()
print('emulator mapped and DISABLED (the board is listening to its real input)')


In [ ]:
# =============================================================================
#  INTERNAL SCOPE - base 0x40100000
# =============================================================================
#  16384 raw samples at up to 125 MSa/s. The only overload guard on the real
#  path: the emulator's sticky clip flag is gone once a real coil is measured.
#  Sees the same signal as the rest of the chain, so with the emulator enabled it
#  shows the emulator.
# =============================================================================
SC_BASE = 0x40100000
SC_SPAN = 0x40000
SC_BUF_A = 0x10000
SC_BUF_B = 0x20000
SC_NMAX = 1 << 14


class Scope(Bus):

    def __init__(self):
        Bus.__init__(self, SC_BASE, span=SC_SPAN)

    def _buf(self, off, n):
        """Read n samples, one 32-bit word at a time (this is a register block,
        not DDR). The buffer holds 14-bit two's-complement values."""
        ba = bytearray(4 * n)
        m = self.m
        for i in range(n):
            o = off + 4 * i
            ba[4 * i:4 * i + 4] = m[o:o + 4]
        v = np.frombuffer(bytes(ba), dtype='<u4').astype(np.int64) & 0x3FFF
        return np.where(v & 0x2000, v - 0x4000, v).astype(float)

    def acquire(self, dec=1, n=SC_NMAX, chan='A'):
        """Capture and return (samples, sample_rate). dec = 1 is every sample."""
        self.w(0x14, dec)
        self.w(0x10, n - 1)
        self.w(0x00, 2)        # reset the capture engine
        self.w(0x00, 1)        # arm
        self.w(0x04, 1)        # trigger now, from software
        t0 = time.time()
        while self.r(0x04) != 0:
            if time.time() - t0 > 1.0:
                break
        time.sleep(0.005)
        off = SC_BUF_A if chan == 'A' else SC_BUF_B
        return self._buf(off, n), FS_CLK / float(dec)

    def peak_frac(self, n=4096):
        """TOTAL peak excursion as a fraction of full scale. 1.0 means clipping.

        Deliberately includes everything - the converter's DC offset, broadband
        noise, out-of-band pickup - because all of it counts towards clipping.
        That makes it the right guard and the WRONG way to measure the size of a
        tone. Use tone_frac() for that."""
        x, _ = self.acquire(dec=1, n=n)
        return float(np.max(np.abs(x))) / 8191.0

    def tone_frac(self, f, n=4096):
        """Amplitude of the component at f alone, as a fraction of full scale.

        WHY THIS EXISTS. peak_frac() answers "how close am I to clipping". It was
        also being used to answer "how much 40 MHz is arriving", and those are
        not the same question. max|x| is inflated by the converter's DC offset
        and by every other thing in the record, so on this board it read about
        four times the true tone amplitude - which then over-predicted the
        feedback loop gain by the same factor.

        A least-squares projection onto cos and sin at the known frequency fixes
        it: the mean is removed first, so DC cannot contribute, and anything not
        at f is orthogonal to both basis vectors and averages away. No window and
        no scalloping loss, because f is known exactly rather than searched."""
        x, fs = self.acquire(dec=1, n=n)
        x = x - np.mean(x)                       # the converter's DC offset
        t = np.arange(len(x)) / fs
        basis = np.column_stack([np.cos(2.0 * np.pi * f * t),
                                 np.sin(2.0 * np.pi * f * t)])
        coef, _, _, _ = np.linalg.lstsq(basis, x, rcond=-1)
        return float(np.hypot(coef[0], coef[1])) / 8191.0


sc = Scope()
print('scope mapped, %d samples per capture' % SC_NMAX)


# =============================================================================
#  ALL_OFF  -  the one call that makes the hardware harmless
# =============================================================================
#  Every step below that raises the drive does so inside a try/finally that ends
#  here, so an exception, a failed fit or an interrupted cell cannot leave the
#  coil being driven with the feedback loop closed. It is also registered to run
#  when the kernel shuts down.
#
#  Safe to call at any time, including twice, including when nothing is running.
#  If anything ever looks wrong on the bench, run all_off() in a fresh cell.
# =============================================================================
def all_off():
    """Silence the drive, open the feedback loop, disable the emulator."""
    for what, fn in (('generator', lambda: gen.off()),
                     ('feedback', lambda: ps.off()),
                     ('emulator', lambda: emu.disable())):
        try:
            fn()
        except Exception as exc:
            # One block failing to switch off must never stop the others being
            # switched off - the generator is the one that matters.
            print('all_off: could not switch off the %s (%s)' % (what, exc))


import atexit
atexit.register(all_off)
all_off()
print('all_off() defined and armed - drive, feedback and emulator are all off')


---
# Recording to memory

The voltmeter reports one number per tuning; spectra need time records, and those need memory.
This is the Stage 6/7 transport with every safeguard kept:

- **The region comes from the device tree**, not from a constant.
- **The bounds are read back before arming.** The writer is a DMA master: if the stop bound did
  not take, it runs upward through the memory the operating system is using. That is not an
  exception with a traceback — it is a dead board. Two register reads turn the worst failure
  mode here into a sentence.
- **The region is pre-filled with a poison value**, so a torn record cannot masquerade as data.
- **Settle first, record second.** Arming and then sleeping lets the opening milliseconds catch
  the pipeline still flushing the *previous* tuning, which poisons each sweep point with its
  predecessor's power — most visibly the first point, which has maximum leverage on the fit.
- **Wait for the writer.** The sample counter reports words handed *to* it, not words landed.


In [ ]:
# =============================================================================
#  MEMORY CAPTURE
# =============================================================================
def _be32(b, i=0):
    return struct.unpack('>I', b[i * 4:i * 4 + 4])[0]


def _dt_read(path):
    for root in ('/sys/firmware/devicetree/base', '/proc/device-tree'):
        try:
            with open(os.path.join(root, path), 'rb') as fh:
                return fh.read()
        except (IOError, OSError):
            pass
    return None


_blob = _dt_read('memory/reg') or _dt_read('memory@0/reg')
assert _blob and len(_blob) >= 8, 'cannot read the device tree memory node'
_declared_end = _be32(_blob, 0) + _be32(_blob, 1)
assert _declared_end < 512 * 1024 ** 2, \
    'no memory above the declared end - see Test Stage 6 Step 1'

MEM_START = (_declared_end + 0xFFF) & ~0xFFF
MEM_SIZE = 0x00800000                        # 8 MB is far more than needed
MEM_STOP = MEM_START + (MEM_SIZE & ~7) - 8   # LAST address, INCLUSIVE
POISON = 0xDEADBEEF

print('capture region 0x%08X .. 0x%08X inclusive (%d MB)'
      % (MEM_START, MEM_STOP + 7, MEM_SIZE // (1024 * 1024)))


def _poison(nbytes):
    fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
    mm = mmap.mmap(fd, ((nbytes + 4095) // 4096) * 4096, offset=MEM_START)
    try:
        mm[0:nbytes] = np.full(nbytes // 4, POISON, dtype='<u4').tobytes()
    finally:
        mm.close()
        os.close(fd)


def _read_region(nbytes):
    """Bulk read of DDR. try/finally because everything here can raise, and a
    leaked mapping inside a scan loop surfaces much later as an unrelated-
    looking out-of-memory error."""
    fd = mm = None
    try:
        fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
        mm = mmap.mmap(fd, ((nbytes + 8191) // 4096) * 4096, offset=MEM_START)
        return mm[0:nbytes]
    finally:
        if mm is not None:
            mm.close()
        if fd is not None:
            os.close(fd)


def capture(lo_word, dec_log2, n_samples, settle=None):
    """Record n_samples of the complex stream. Returns a complex numpy array.

    lo_word is the RAW 32-bit local-oscillator word, so a caller that wants an
    exact zero-frequency landing can pass gen.lo_word() and get it. Raises on
    any transport fault rather than returning plausible garbage."""
    settle = SETTLE if settle is None else settle
    if n_samples * 8 > (MEM_STOP + 8 - MEM_START):
        raise ValueError('record of %d samples exceeds the capture region' % n_samples)

    _poison(n_samples * 8)

    acq.w(0x00, 0)                       # disable while settings change
    acq.w(0x08, dec_log2)
    acq.w(0x0C, 0)
    acq.dec = dec_log2
    acq.tune_word(lo_word)
    acq.w(0x28, MEM_START)
    acq.w(0x2C, MEM_STOP)

    lo_rb, hi_rb = acq.r(0x28), acq.r(0x2C)
    if lo_rb != MEM_START or hi_rb != MEM_STOP:
        acq.w(0x00, 0)
        raise RuntimeError(
            'capture bounds did not take: wrote 0x%08X..0x%08X, read back '
            '0x%08X..0x%08X. Refusing to arm an unbounded memory writer.'
            % (MEM_START, MEM_STOP, lo_rb, hi_rb))

    acq.w(0x00, 1 | 2)                   # run at the new tuning...
    acq.w(0x00, 1)
    time.sleep(settle)                   # ...let the pipeline flush...
    acq.w(0x00, 1 | 16 | 128)            # ...then arm with a fresh writer
    acq.w(0x00, 1 | 16)

    t_record = n_samples * (1 << dec_log2) / FS_CLK
    deadline = time.time() + t_record + 0.5
    while int(acq.r(0x34)) < n_samples and time.time() < deadline:
        time.sleep(0.001)
    acq.w(0x00, 1)                       # disarm
    time.sleep(0.002)                    # let the last burst land

    st2 = acq.r(0x38)
    if st2 & 8:
        raise RuntimeError('memory refused writes - run check_system.py')
    got = min(int(acq.r(0x34)), n_samples)
    if got < n_samples:
        raise RuntimeError('short record: asked %d, got %d' % (n_samples, got))

    buf = _read_region(got * 8)
    u32 = np.frombuffer(buf, dtype='<u4')
    limit = max(8, 0.001 * u32.size)
    if int(np.count_nonzero(u32 == POISON)) > limit:
        # A word still inside the writer's burst buffer is not a lost word. If
        # it is genuinely lost it will still be poison in 20 ms.
        time.sleep(0.02)
        u32 = np.frombuffer(_read_region(got * 8), dtype='<u4')
        if int(np.count_nonzero(u32 == POISON)) > limit:
            raise RuntimeError('unwritten words in the record - transport fault')

    iq = u32.astype(np.int64)
    i = np.where(iq[0::2] & 0x80000000, iq[0::2] - (1 << 32), iq[0::2])
    q = np.where(iq[1::2] & 0x80000000, iq[1::2] - (1 << 32), iq[1::2])
    return i.astype(float) + 1j * q.astype(float)


---
# Analysis

**`sweep()`** steps the generator across a list of frequencies, keeps the receiver pinned exactly
on the drive, and reads a complex number at each point. **Only the magnitude is used** — see
Step 5 for why the phase between points is not a measurement on this board.

**`fit_lorentzian()`** turns `|z|²` against frequency into a centre and a width, with no fitting
library: the *reciprocal* of a Lorentzian power curve is a parabola, so a weighted quadratic fit
recovers everything. It also fits an **additive power floor**, found by search rather than
assumed — a floor absorbed into the parabola inflates the width, and the floor under a coil
behind 0.3 pF is not small.

**`measure_transfer()`** returns both the total input excursion (the clipping guard) and the
component at the drive frequency alone. Different questions; only the second is a transfer.

**`refine_and_fit()`** picks a decimation from a linewidth estimate, sweeps densely and fits.
Step 3 calls it, and Step 4b calls it again once the drive is adopted, so the headline numbers
always belong to the drive the rest of the notebook uses.


In [ ]:
# =============================================================================
#  ANALYSIS
# =============================================================================
def sweep(freqs, drive_frac=None, navg=None, settle=None, level_guard=None):
    """Drive at each frequency and read the complex response at DC.

    Returns (f_actual, z): f_actual is what the generator really produced (its
    word is quantised) and z is complex. Only |z| is a measurement - see Step 5.

    The receiver is tuned by WORD, not by hertz, so the signal lands at exactly
    zero frequency and the two register reads cannot land on different phases of
    a rotating vector."""
    drive_frac = DRIVE_FRAC if drive_frac is None else drive_frac
    navg = N_AVG if navg is None else navg
    settle = SETTLE if settle is None else settle

    gen.amp(drive_frac)                  # refused outright if above DRIVE_MAX
    gen.run()
    f_act = np.zeros(len(freqs))
    z = np.zeros(len(freqs), dtype=complex)
    try:
        for k, f in enumerate(freqs):
            f_act[k] = gen.freq(f)
            acq.tune_word(gen.lo_word())
            time.sleep(settle)
            if level_guard is not None and sc.peak_frac(GUARD_NSAMP) > level_guard:
                raise RuntimeError('input exceeded %.0f%% of full scale at %.6f MHz'
                                   % (100 * level_guard, f_act[k] / 1e6))
            z[k] = acq.measure(navg)
    finally:
        # Whatever ended this sweep - the guard above, a bad register read, a
        # Ctrl-C halfway through a 350-point pass - the drive comes off. Every
        # measurement goes through here, so no failure path can walk away
        # leaving the coil driven.
        gen.amp(0.0)
    return f_act, z


def preflight_drive(f_probe, target=None):
    """Approach the working drive in two steps instead of jumping to it.

    THE FAILSAFE THAT MATTERS. Every other guard assumes EXT_GAIN_DB is right; if
    it is understated, DRIVE_MAX is too permissive and the first full-drive write
    is the one that does the damage. So measure at a tenth of the drive,
    extrapolate, and refuse before the converter ever sees the full value."""
    target = DRIVE_FRAC if target is None else target
    lo = max(target / 10.0, 2.0e-4)
    try:
        gen.freq(f_probe); gen.amp(lo); gen.run(); time.sleep(0.05)
        small = sc.peak_frac(GUARD_NSAMP)
    finally:
        gen.off()
    pred = small * target / lo
    print('preflight: %.4f of full scale at drive %.5f -> %.3f predicted at %.5f'
          % (small, lo, pred, target))
    if pred > 0.90:
        raise RuntimeError(
            'a drive of %.5f would put %.2f of full scale into the input. '
            'Something delivers far more than expected - check EXT_GAIN_DB '
            '(declared %.1f dB) and that the amplifier is in the DRIVE path, '
            'not the pickup path.' % (target, pred, EXT_GAIN_DB))
    if small > 0.60:
        raise RuntimeError('input already at %.2f of full scale at a tenth of '
                           'the working drive - stop and investigate.' % small)
    return pred


def fit_lorentzian(f, p, guard=0.05, minpts=5):
    """Centre, linewidth, peak and baseline from power p against frequency f.

    1/p is a parabola in f for a PURE Lorentzian. Any additive floor is absorbed
    into that parabola and widens the answer, so the floor is found by search:
    for each trial floor, subtract it, fit, rebuild the full model, and keep the
    trial that reproduces the measured points best. Two passes, coarse then fine.

    Points below `guard` of the peak are excluded - their reciprocals are all
    noise - and the rest are weighted by their power, so the well-measured points
    near the top dominate."""
    f = np.asarray(f, dtype=float)
    p = np.asarray(p, dtype=float)
    pmax = float(np.max(p))
    u = f - f.mean()

    def _try(b):
        q = p - b
        keep = q > guard * q.max()
        if int(np.count_nonzero(keep)) < minpts:
            return None
        A, B, C = np.polyfit(u[keep], 1.0 / q[keep], 2, w=q[keep])
        if A <= 0.0:
            return None                              # parabola opens the wrong way
        u0 = -B / (2.0 * A)
        lw = 2.0 * np.sqrt(max(C / A - u0 ** 2, 1e-30))
        p0 = 1.0 / max(C - A * u0 ** 2, 1e-300)
        f0 = f.mean() + u0
        model = p0 / (1.0 + (2.0 * (f - f0) / lw) ** 2) + b
        rms = float(np.sqrt(np.mean(((p - model) / pmax) ** 2)))
        return rms, (f0, lw, p0, b)

    best, best_b = _try(0.0), 0.0
    hi = 0.999 * float(np.min(p))
    if hi > 0.0:
        for b in np.linspace(0.0, hi, 17):
            got = _try(b)
            if got is not None and (best is None or got[0] < best[0]):
                best, best_b = got, b
        w = hi / 16.0
        for b in np.linspace(max(best_b - w, 0.0), min(best_b + w, hi), 17):
            got = _try(b)
            if got is not None and (best is None or got[0] < best[0]):
                best, best_b = got, b
    if best is None:
        _n = int(np.count_nonzero(p > guard * p.max()))
        if _n < minpts:
            raise ValueError('fit failed: only %d points above the %.0f%% guard'
                             % (_n, 100 * guard))
        raise ValueError('fit failed: %d points available but no trial baseline '
                         'gave a peak-shaped curve - this is not a resonance '
                         '(peak %.1f counts)' % (_n, np.sqrt(max(p.max(), 0.0))))
    f0, lw, p0, base = best[1]
    return dict(f0=f0, lw=lw, peak=p0, base=base, resid=best[0], Q=f0 / lw)


def lorentzian(f, fit):
    """The fitted model evaluated anywhere - used by every plot, so no cell has
    to rebuild it and none can rebuild it differently."""
    return fit['peak'] / (1.0 + (2.0 * (np.asarray(f, dtype=float) - fit['f0'])
                                 / fit['lw']) ** 2) + fit['base']


def choose_dec(lw):
    """log2(decimation) small enough that the receiver window does not broaden a
    line of width lw.

    Aim: clean half-window <= lw/10, i.e. 0.1 x output_rate <= lw/10, i.e.
    output_rate <= lw. Clamped to 8..12 - below R = 256 the droop corrector
    bypasses itself, above R = 4096 the hardware stops."""
    d = int(np.ceil(np.log2(FS_CLK / max(lw, 1.0))))
    return int(min(12, max(8, d)))


def half_power_width(f, p):
    """Crude width straight off the data: the span above half the peak.

    Deliberately not a fit. Used only to decide how wide the next sweep should
    be, where being fooled by a fit would be worse than being approximate."""
    p = np.asarray(p, dtype=float)
    base = float(np.median(p))
    half = base + 0.5 * (float(np.max(p)) - base)
    above = np.where(p >= half)[0]
    if len(above) < 2:
        return None
    return float(f[above[-1]] - f[above[0]])


def refine_and_fit(f_centre, lw_hint, drive, npts=61, span_lw=2.5, verbose=True):
    """Dense sweep at a decimation chosen from lw_hint, then fit.

    Returns (dec_log2, f, p, fit). Called by Step 3 and AGAIN by Step 4b once the
    drive has been adopted, so the headline numbers always belong to the drive the
    rest of the notebook uses. Nothing about it is specific to this coil, which is
    the point: the same call works warm, cold, in the trap or out of it."""
    dec = choose_dec(lw_hint)
    acq.start(dec)
    if verbose:
        smear = acq.window_hz() / lw_hint
        print('  decimation R = %d, output rate %.1f kHz, clean window +-%.1f kHz '
              '(%.1f%% of a linewidth)'
              % (1 << dec, acq.f_out() / 1e3, acq.window_hz() / 1e3, 100 * smear))
        if smear > 0.25:
            print('  WARNING: the window is a sizeable fraction of the line and')
            print('  will broaden the measured width.')
    grid = f_centre + np.linspace(-span_lw, span_lw, npts) * lw_hint
    f, z = sweep(grid, drive_frac=drive, navg=N_AVG, settle=SETTLE, level_guard=0.95)
    gen.amp(0.0)
    p = np.abs(z) ** 2
    return dec, f, p, fit_lorentzian(f, p)


def measure_transfer(f0, drive):
    """One-way DAC -> coil -> ADC transfer at f0.

    Returns (raw_peak, tone, G_fwd). raw_peak is the TOTAL excursion, inflated by
    the converter's DC offset and everything else in the record - that is the
    right clipping guard and the WRONG way to size a tone. tone is the component
    at f0 alone, and only that is a transfer."""
    try:
        gen.freq(f0); gen.amp(drive); gen.run(); time.sleep(0.05)
        raw = sc.peak_frac(SC_NMAX)
        tone = sc.tone_frac(f0)
    finally:
        gen.off()
    return raw, tone, tone / max(drive, 1e-12)


print('analysis helpers defined')


---
# Step 0 — Is the right thing loaded, and does everything answer?

**Loading an old bitstream looks exactly like success:** the light blinks, registers answer,
measurements look plausible. The build identifier at `0x48` is a hash of the Verilog it was built
from, and zero means `build_project.tcl` did not produce it.

The emulator probe matters for a subtler reason — an older bitstream has a *stub* at that address
that answers every read with zero and looks politely dead, and Step 1 would then compare against
nothing at all and pass.


In [ ]:
# =============================================================================
#  STEP 0 - PREFLIGHT
# =============================================================================
print('=' * 68)
print('STEP 0 - preflight')
print('=' * 68)

fail = []

# --- 0a. build identifier ----------------------------------------------------
bid = acq.build_id()
print('build id                 : 0x%08X' % bid)
if bid == 0:
    fail.append('BUILD_ID is zero - this bitstream was not built by build_project.tcl')
else:
    print('  compare this against bitstream_id.txt, or run check_system.py')

# --- 0b. do the acquisition registers respond? -------------------------------
probe = 0x5A5A5A5A
acq.w(0x04, probe)
back = acq.r(0x04)
print('local oscillator readback: wrote 0x%08X, read 0x%08X' % (probe, back))
if back != probe:
    fail.append('acquisition registers do not read back')

# --- 0c. is the emulator present, or is this a stub? -------------------------
probe18 = 0x15555                     # alternating bits, positive in 18 bits
emu.w(0x04, probe18)
back18 = emu.r(0x04) & 0x3FFFF
print('emulator readback        : wrote 0x%05X, read 0x%05X' % (probe18, back18))
if back18 != probe18:
    fail.append('the emulator is absent or stubbed - Step 1 cannot run')

#  Does it have the FOUR-coefficient map? COEF_C2 at 0x28 exists only in the
#  two-way look-ahead build (2026-08-07). Without it the recursion has no pole
#  and the emulator passes the drive straight through - a silent failure that
#  looks like a broken fit rather than a mismatched bitstream.
emu.w(0x28, probe18)
back28 = emu.r(0x28) & 0x3FFFF
print('emulator COEF_C2 readback: wrote 0x%05X, read 0x%05X' % (probe18, back28))
if back28 != probe18:
    fail.append('no COEF_C2 at 0x28 - this bitstream predates the two-way '
                'look-ahead resonator, but this notebook writes four '
                'coefficients. Rebuild and reload.')
emu.w(0x04, 0)
emu.w(0x28, 0)                 # leave the coefficients as they were found

# --- 0d. is the chain actually producing samples? ----------------------------
acq.start(DEC_FIT)
n0 = acq.r(0x20)
time.sleep(0.10)
n1 = acq.r(0x20)
rate = (n1 - n0) / 0.10
expect = FS_CLK / (1 << DEC_FIT)
print('output rate              : %.0f samples/s (expected %.0f, %+.1f%%)'
      % (rate, expect, 100.0 * (rate / expect - 1.0)))
if not (0.8 * expect < rate < 1.25 * expect):
    fail.append('output sample rate is wrong - check DEC_LOG2 and the clock')

st2 = acq.status2()
print('status2                  : 0x%02X  (overflow=%d corr_overrun=%d '
      'corr_bypassed=%d write_err=%d)'
      % (st2, st2 & 1, (st2 >> 1) & 1, (st2 >> 2) & 1, (st2 >> 3) & 1))
if (st2 >> 2) & 1:
    print('  NOTE: droop corrector bypassed - expected below R = 256 only')

print('-' * 68)
if fail:
    for f in fail:
        print('FAIL: %s' % f)
    raise SystemExit('Step 0 failed - fix these before continuing')
print('PASS - hardware answers and the chain is running')


---
# Step 1 — Does the instrument still return a known answer?

Enabled, the emulator **replaces the digitised input entirely**, so the coil can stay connected
and irrelevant. This check is therefore not a one-way door: it is available at any moment,
including between two real measurements that disagree.

The test sets a centre and a Q the analysis is never told, sweeps, fits, and demands them back.
The comparison is against the **quantised** values — the coefficients are 18-bit integers, so the
emulator's true centre and Q are the ones recovered from the rounded coefficients, and comparing
against the requested numbers would build a permanent error into the tolerance.

`SELFTEST_F0` is deliberately not 40 MHz: a 40 MHz constant baked into the analysis would fail
here rather than quietly agree.


In [ ]:
# =============================================================================
#  STEP 1 - INSTRUMENT SELF-TEST AGAINST A KNOWN ANSWER
# =============================================================================
def instrument_selftest(verbose=True):
    """Switch the emulator in, measure it, and demand the answer it was given.

    Returns a dict. Leaves the emulator DISABLED whatever happens."""
    try:
        f0_q, q_q = emu.config(SELFTEST_F0, SELFTEST_Q, delay=13, oshift=7)
        lw_q = f0_q / q_q
        emu.clear()
        emu.enable()

        acq.start(DEC_FIT)
        ps.off()

        # +-2.5 linewidths is the whole curve, not just its top. SELFTEST_DRIVE
        # is capped at 0.3 to keep the emulator's 25-bit state clear of
        # saturation, and at the amplifier's damage limit when one is fitted.
        grid = f0_q + np.linspace(-2.5, 2.5, 25) * lw_q
        f, z = sweep(grid, drive_frac=SELFTEST_DRIVE)
        gen.amp(0.0)

        if emu.clipped():
            raise RuntimeError('the emulator saturated - lower the self-test drive')

        fit = fit_lorentzian(f, np.abs(z) ** 2)
        err_f = (fit['f0'] - f0_q) / lw_q            # in linewidths
        err_q = fit['Q'] / q_q - 1.0

        ok = (abs(err_f) <= SELFTEST_TOL_F) and (abs(err_q) <= SELFTEST_TOL_Q)
        if verbose:
            print('known answer (quantised) : f0 = %.6f MHz, Q = %.1f, lw = %.1f kHz'
                  % (f0_q / 1e6, q_q, lw_q / 1e3))
            print('measured                 : f0 = %.6f MHz, Q = %.1f'
                  % (fit['f0'] / 1e6, fit['Q']))
            print('error                    : centre %+.3f linewidths, Q %+.2f%%'
                  % (err_f, 100 * err_q))
            print('fit residual             : %.4f of peak' % fit['resid'])
            print('-' * 68)
            print('%s - the instrument %s reproduce an answer it was given'
                  % ('PASS' if ok else 'FAIL', 'can' if ok else 'CANNOT'))
        return dict(ok=ok, f0_known=f0_q, q_known=q_q, f0_meas=fit['f0'],
                    q_meas=fit['Q'], err_f_lw=err_f, err_q_frac=err_q,
                    resid=fit['resid'], f=f, z=z, fit=fit)
    finally:
        emu.disable()
        gen.off()


print('=' * 68)
print('STEP 1 - instrument self-test (emulator in, coil ignored)')
print('=' * 68)
st1_before = instrument_selftest()
print('emulator disabled - the board is listening to its real input again')


In [ ]:
# --- Step 1 picture: the known answer and what came back ---------------------
_f = st1_before['f'] / 1e6
_p = np.abs(st1_before['z']) ** 2
_ft = st1_before['fit']
_fine = np.linspace(_f.min(), _f.max(), 400)
_model = _ft['peak'] / (1.0 + (2.0 * (_fine * 1e6 - _ft['f0']) / _ft['lw']) ** 2) + _ft['base']

fig, ax = plt.subplots(figsize=(7.0, 3.6))
ax.plot(_f, _p, 'o', ms=4, label='measured')
ax.plot(_fine, _model, '-', lw=1.4, label='fit')
ax.axvline(st1_before['f0_known'] / 1e6, ls='--', lw=1.0, color='0.4',
           label='known centre')
ax.set_xlabel('drive frequency / MHz')
ax.set_ylabel('power (arb.)')
ax.set_title('Step 1 - emulator with a known answer  (Q asked %.0f, got %.0f)'
             % (st1_before['q_known'], st1_before['q_meas']))
ax.legend(fontsize=8)
plt.show()


---
# Step 2 — Levels, and what else is in the band

| | Why it matters |
|---|---|
| Are the levels sane? | An overloaded input produces a distorted shape that still looks vaguely like a resonance. This project's signature failure |
| What else is in the band? | A clock harmonic near the coil would be found by Step 3 and fitted as a spectacular high-Q resonance |
| How quiet is the input? | The floor decides whether a resonance is findable at all |

**Run this with the input terminated first.** That catalogues the board's *own* stray signals;
run it again with the coil attached and anything new belongs to the coil or the room. Without the
first run there is no way to tell those apart.


In [ ]:
# =============================================================================
#  STEP 2a - INPUT LEVELS (the only overload guard on the real path)
# =============================================================================
print('=' * 68)
print('STEP 2 - input health   [%s]' % SOURCE_NOTE)
print('=' * 68)

gen.off()
ps.off()
emu.disable()

quiet_frac = sc.peak_frac(SC_NMAX)
print('input peak, generator OFF : %.4f of full scale (%.0f counts)'
      % (quiet_frac, quiet_frac * 8191))

if COIL_CONNECTED:
    try:
        gen.freq(0.5 * (SEARCH_LO + SEARCH_HI))
        gen.amp(DRIVE_FRAC)
        gen.run()
        time.sleep(0.05)
        drive_frac_seen = sc.peak_frac(SC_NMAX)
    finally:
        gen.off()
    print('input peak, generator ON  : %.4f of full scale (%.0f counts)'
          % (drive_frac_seen, drive_frac_seen * 8191))
else:
    drive_frac_seen = float('nan')

print('-' * 68)
if quiet_frac > 0.5:
    print('WARNING: the input is more than half full scale with NO drive.')
    print('         Something large is arriving. Find it before going on -')
    print('         everything downstream assumes a linear front end.')
elif quiet_frac > 0.05:
    print('NOTE: %.1f%% of full scale with no drive. Not dangerous, but the'
          % (100 * quiet_frac))
    print('      survey below should show what it is.')
else:
    print('levels look sane')


In [ ]:
# =============================================================================
#  STEP 2b - BAND SURVEY, generator off
# =============================================================================
#  Step the receiver across the search band and record the power in each
#  window. The step is a fraction of the clean window width so nothing can hide
#  in a gap between windows - the same discipline the adaptive sweep uses.
# =============================================================================
acq.start(DEC_SEARCH)
f_out_s = acq.f_out()
step_s = 0.15 * f_out_s                       # windows overlap generously
survey_f = np.arange(SEARCH_LO, SEARCH_HI + step_s, step_s)

gen.off()
survey_p = np.zeros(len(survey_f))
t0 = time.time()
for k, f in enumerate(survey_f):
    z = capture(int(round(f / FS_CLK * (1 << 32))), DEC_SEARCH, 1024, settle=0.004)
    survey_p[k] = float(np.mean(np.abs(z) ** 2))
print('surveyed %d windows across %.1f - %.1f MHz in %.0f s'
      % (len(survey_f), SEARCH_LO / 1e6, SEARCH_HI / 1e6, time.time() - t0))

# --- report the strongest features, as decibels above the median window ------
med = float(np.median(survey_p))
excess_db = 10.0 * np.log10(np.maximum(survey_p, 1e-300) / max(med, 1e-300))
order = np.argsort(excess_db)[::-1]

print('')
print('median window power      : %.3e  (this is the noise floor)' % med)
print('strongest windows (dB above the median):')
print('    frequency / MHz    excess / dB')
shown = 0
for idx in order:
    if excess_db[idx] < 6.0 or shown >= 10:
        break
    print('    %14.4f     %+8.1f' % (survey_f[idx] / 1e6, excess_db[idx]))
    shown += 1
if shown == 0:
    print('    (nothing more than 6 dB above the median - a clean band)')

fig, ax = plt.subplots(figsize=(9.0, 3.2))
ax.plot(survey_f / 1e6, excess_db, '-', lw=1.0)
ax.axhline(6.0, ls='--', lw=0.9, color='0.5', label='6 dB reporting threshold')
ax.set_xlabel('receiver centre / MHz')
ax.set_ylabel('dB above median')
ax.set_title('Step 2 - band survey, generator off   [%s]' % SOURCE_NOTE)
ax.legend(fontsize=8)
plt.show()

print('')
print('KEEP THIS PLOT. With the input terminated it is the board\'s own spectrum;')
print('with the coil attached, anything NEW is the coil or the room.')


---
# Step 3 — Find the resonance without being told where it is

| Pass | What it does | Why |
|---|---|---|
| **Look around** | Step across the band, few averages, no fitting | It only has to find the neighbourhood, and nothing is fitted, so there is nothing to be fooled by |
| **Confirm** | Scan the candidate's neighbourhood twice and correlate | Contrast alone cannot tell a resonance from noise — see below |
| **Bracket** | Widen until the curve comes back down on *both* sides | A fit that interpolates between points is trustworthy; one that extrapolates past them is not |
| **Refine** | Dense sweep at a decimation chosen from the width just estimated, then fit | The receiver window must be small next to the linewidth, or the window smears the curve |

**The coarse pass cannot find a line narrower than its own step** — it walks straight over it. So
`SEARCH_STEP` is **halved automatically** and the pass repeated until a candidate confirms, which
is what a colder, higher-Q resonator needs and is exactly the parameter nobody remembers to
change when the coil moves into a cryostat.

**Why contrast is not enough to trigger that.** `|z|²` on an empty band is exponentially
distributed, so the largest of a few hundred points sits about 8 dB above the median *whatever*
the averaging — the ratio is scale-free. A 6 dB threshold therefore passes pure noise, the
halving never fires, and the search reports the biggest noise excursion in the band as a
discovery. The confirmation step scans the neighbourhood twice and **correlates the two scans**: a
resonance puts the same shape on the grid both times, while two realisations of noise are
independent and correlate at zero give or take 1/√N. Correlation rather than "do the two maxima
agree", because the top of a broad line is flat and its argmax wanders by a good fraction of a
linewidth while a narrow line's does not — one threshold cannot serve both. Correlation is
scale-free and works at any Q.

The fit at the end is checked against `SEARCH_FIT_MAX_RESID`, but only warns: it is taken at the
starting drive, and a poor residual there is exactly what Step 4b exists to fix. **Step 4b
enforces it** once the drive has been chosen by measurement.


In [ ]:
# =============================================================================
#  STEP 3 - FIND THE RESONANCE
# =============================================================================
if not COIL_CONNECTED:
    raise SystemExit('COIL_CONNECTED is False - Steps 3 onwards need the coil. '
                     'Stop here; the survey above is the deliverable for this run.')

print('=' * 68)
print('STEP 3 - find the resonance')
print('=' * 68)

emu.disable()
ps.off()
preflight_drive(0.5 * (SEARCH_LO + SEARCH_HI))


def confirm_candidate(f_cand, step, drive, band_floor):
    """Does the coarse pass's best point survive a second look?

    THE CONTRAST TEST ALONE IS NOT ENOUGH, and this is the trap it walks into:
    |z|^2 on an empty band is exponentially distributed, so the largest of a few
    hundred points sits roughly 8 dB above the median whatever the averaging -
    the ratio is scale-free, so more averaging does not help. A 6 dB threshold
    therefore passes pure noise, and the search reports the biggest noise
    excursion in the band as a discovery. That is exactly what a narrow line
    looks like from outside: the coarse grid steps over the real resonance,
    finds nothing, and the noise wins by default.

    So scan the neighbourhood TWICE and CORRELATE the two scans. A resonance
    puts the same shape on the grid both times; two realisations of noise are
    independent, and their correlation sits at zero give or take 1/sqrt(N).
    Correlation is used rather than "do the two maxima agree", because the top
    of a broad line is flat and its argmax wanders by a good fraction of a
    linewidth, while a narrow line's does not - the same threshold cannot serve
    both. Correlation is scale-free and works at any Q.

    `band_floor` is the median power of the COARSE pass, not of this zoomed
    window: on a line wider than the window the local median is most of the way
    up the peak, and a perfectly good resonance then measures 2 dB of "contrast"
    against itself.
    """
    span = 2.0 * step
    fine = np.linspace(f_cand - span, f_cand + span, 81)
    fine = fine[(fine > 1.0e6) & (fine < 0.49 * FS_CLK)]
    ps = []
    for _ in range(2):
        f_, z_ = sweep(fine, drive_frac=drive, navg=32, settle=0.006,
                       level_guard=0.95)
        gen.amp(0.0)
        ps.append(np.abs(z_) ** 2)
    a = ps[0] - ps[0].mean()
    b = ps[1] - ps[1].mean()
    denom = np.sqrt(float(np.dot(a, a)) * float(np.dot(b, b)))
    corr = float(np.dot(a, b) / denom) if denom > 0 else 0.0
    p = 0.5 * (ps[0] + ps[1])
    contrast = 10.0 * np.log10(p.max() / max(band_floor, 1e-300))
    lw = half_power_width(fine, p)
    ok = bool(corr >= SEARCH_MIN_CORR and contrast >= SEARCH_MIN_DB and lw)
    return dict(ok=ok, f0=float(fine[int(np.argmax(p))]), lw=lw, corr=corr,
                contrast=contrast, f=fine, p=p)


# --- pass 1: look around, then confirm --------------------------------------
#  The coarse pass cannot find a line narrower than its own step, so the step is
#  halved and the pass repeated until a candidate CONFIRMS. That is the one
#  parameter that has to change when the coil gets colder and narrower, and it
#  is the one nobody remembers to change.
acq.start(DEC_SEARCH)
SEARCH_STEP_USED = float(SEARCH_STEP)
FOUND = None
for _attempt in range(SEARCH_RETRIES + 1):
    grid1 = np.arange(SEARCH_LO, SEARCH_HI + SEARCH_STEP_USED, SEARCH_STEP_USED)
    t0 = time.time()
    f1, z1 = sweep(grid1, drive_frac=DRIVE_FRAC, navg=16, settle=0.006,
                   level_guard=0.95)
    gen.amp(0.0)
    p1 = np.abs(z1) ** 2
    k1 = int(np.argmax(p1))
    contrast_db = 10.0 * np.log10(p1[k1] / max(np.median(p1), 1e-300))
    print('pass 1, step %6.1f kHz : %4d points in %3.0f s, best %.4f MHz, '
          '%.1f dB above the median, %.0f counts'
          % (SEARCH_STEP_USED / 1e3, len(grid1), time.time() - t0,
             f1[k1] / 1e6, contrast_db, np.sqrt(p1[k1])))

    conf = confirm_candidate(f1[k1], SEARCH_STEP_USED, DRIVE_FRAC,
                             float(np.median(p1)))
    print('        confirming : two scans correlate %+.3f (need %.2f), contrast '
          '%.1f dB, width %s'
          % (conf['corr'], SEARCH_MIN_CORR, conf['contrast'],
             '%.1f kHz' % (conf['lw'] / 1e3) if conf['lw'] else 'unresolved'))
    if conf['ok']:
        FOUND = conf
        print('        CONFIRMED at %.4f MHz' % (conf['f0'] / 1e6))
        break
    if _attempt == SEARCH_RETRIES:
        raise RuntimeError(
            'no candidate survived a second look after %d halvings, down to a '
            '%.2f kHz step. The coarse pass keeps landing on noise, which means '
            'either the resonance is narrower than %.2f kHz, or it is outside '
            '%.0f-%.0f MHz, or the coil is not coupled at this drive (%.5f). '
            'Raise SEARCH_RETRIES, widen the band, or check the coupling.'
            % (SEARCH_RETRIES, SEARCH_STEP_USED / 1e3, SEARCH_STEP_USED / 1e3,
               SEARCH_LO / 1e6, SEARCH_HI / 1e6, DRIVE_FRAC))
    SEARCH_STEP_USED *= 0.5
    print('        NOT confirmed - halving the step to %.2f kHz and retrying'
          % (SEARCH_STEP_USED / 1e3))

if np.sqrt(p1[k1]) < 20.0:
    print('        WARNING: under 20 counts at the coarse peak. Check the drive,')
    print('        the coupling, and whether an external amplifier is declared in')
    print('        EXT_GAIN_DB but not actually fitted.')


In [ ]:
# --- pass 2: bracket ---------------------------------------------------------
#  Widen until the curve comes down on BOTH sides. Stopping only once both
#  flanks are below half the peak is what makes the later fit an interpolation
#  rather than an extrapolation.
span = max(6.0 * FOUND['lw'], 0.5 * SEARCH_STEP_USED)
f_c = FOUND['f0']
lw_guess = FOUND['lw']
for attempt in range(6):
    grid2 = f_c + np.linspace(-span, span, 41)
    grid2 = grid2[(grid2 > 1.0e6) & (grid2 < 0.49 * FS_CLK)]
    f2, z2 = sweep(grid2, drive_frac=DRIVE_FRAC, navg=24, settle=0.008,
                   level_guard=0.95)
    gen.amp(0.0)
    p2 = np.abs(z2) ** 2
    base2 = float(np.median(p2))
    half = base2 + 0.5 * (float(np.max(p2)) - base2)
    down_both = (p2[0] < half) and (p2[-1] < half)
    f_c = float(f2[int(np.argmax(p2))])
    lw2 = half_power_width(f2, p2)
    print('pass 2 attempt %d: span +-%8.2f kHz, centre %.4f MHz, flanks down '
          'both sides: %s' % (attempt + 1, span / 1e3, f_c / 1e6, down_both))
    if down_both and lw2:
        lw_guess = lw2
        break
    span *= 2.0

print('        rough linewidth %.2f kHz  ->  rough Q %.0f'
      % (lw_guess / 1e3, f_c / lw_guess))

# --- pass 3: refine and fit --------------------------------------------------
print('pass 3: refine')
DEC_USE, f3, p3, FIT = refine_and_fit(f_c, lw_guess, DRIVE_FRAC)
F0 = FIT['f0']
LW = FIT['lw']
Q_SWEEP = FIT['Q']
FLOOR_FRAC = float(np.sqrt(FIT['base'] / max(FIT['peak'], 1e-300)))

print('')
print('  centre frequency       : %.6f MHz' % (F0 / 1e6))
print('  linewidth              : %.2f kHz' % (LW / 1e3))
print('  quality factor         : %.1f' % Q_SWEEP)
print('  off-resonance floor    : %.1f%% of the peak amplitude' % (100 * FLOOR_FRAC))
print('  fit residual           : %.4f of peak' % FIT['resid'])

#  A fit that does not describe the data is not a resonance. It is only a
#  WARNING here, because this fit is taken at the starting drive and Step 4b has
#  not yet had its say - a poor residual at a drive that turns out to be
#  noise-limited is exactly what Step 4b exists to fix. Step 4b enforces it.
if FIT['resid'] > SEARCH_FIT_MAX_RESID:
    print('  WARNING: residual above the %.2f limit - this is not yet a clean'
          % SEARCH_FIT_MAX_RESID)
    print('  Lorentzian. Step 4b will re-measure at a drive chosen by')
    print('  measurement, and will refuse to continue if it is still this bad.')
if FLOOR_FRAC > 0.5:
    print('  NOTE: a large floor costs precision - it is what the fit has to')
    print('  separate the line from - but it does not bias the width, provided it')
    print('  adds in POWER rather than coherently. Step 5 tests that.')
print('')
print('  Provisional: Step 4b measures the usable drive window and re-runs this')
print('  fit at the drive it recommends.')


In [ ]:
# --- Step 3 picture ----------------------------------------------------------
#  Three panels, all scaled from this run's own data: the whole band, the
#  confirmation scan that decided the candidate was real, and the fitted line.
_fine = np.linspace(f3.min(), f3.max(), 600)

fig, (a1, a2, a3) = plt.subplots(3, 1, figsize=(8.0, 7.4))

a1.plot(f1 / 1e6, 10 * np.log10(np.maximum(p1, 1e-300) / max(np.median(p1), 1e-300)),
        '-', lw=0.9, color=CLR_BLUE)
a1.axvline(F0 / 1e6, ls='--', lw=1.0, color=CLR_RED)
a1.axhline(SEARCH_MIN_DB, ls=':', lw=0.9, color=CLR_GREY)
a1.set_xlabel('drive frequency / MHz')
a1.set_ylabel('dB above median')
a1.set_title('pass 1 - the whole search band at a %.2f kHz step'
             % (SEARCH_STEP_USED / 1e3))

a2.plot((FOUND['f'] - F0) / 1e3, FOUND['p'], '-', lw=1.0, color=CLR_GREEN)
a2.set_xlabel('offset from the fitted centre / kHz')
a2.set_ylabel('power (arb.)')
a2.set_title('confirmation - the two scans correlate %+.3f, %.1f dB contrast'
             % (FOUND['corr'], FOUND['contrast']))

a3.plot((f3 - F0) / 1e3, p3, 'o', ms=3.5, color=CLR_BLUE, label='measured')
a3.plot((_fine - F0) / 1e3, lorentzian(_fine, FIT), '-', lw=1.4, color=CLR_ORANGE,
        label='Lorentzian fit')
a3.axhline(FIT['base'], ls=':', lw=1.0, color=CLR_GREY, label='fitted floor')
a3.set_xlabel('offset from centre / kHz')
a3.set_ylabel('power (arb.)')
a3.set_title('pass 3 - f0 = %.6f MHz, linewidth = %.2f kHz, Q = %.1f, '
             'residual %.4f' % (F0 / 1e6, LW / 1e3, Q_SWEEP, FIT['resid']))
a3.legend(fontsize=8)

plt.show()


---
# Step 4a — How much gain is the amplifier actually giving?

A ladder of drive levels, climbing until the amplifier is near its rating. **It never needs to
know the gain** — it infers the output level from what arrives at the input and the passive
transfer measured without it, so each rung refuses the next if that would cross
`AMP_OUT_MAX_DBM`. Safe whatever is in the box.

**The measured gain is then adopted**, so the ceilings the rest of the notebook obeys come from a
measurement rather than a declaration. The one case that still aborts is a measured gain well
*above* the declared one: `DRIVE_MAX` was derived from the declaration, so every measurement
already taken ran under a ceiling that was too permissive, and that has to be re-run rather than
patched. With no amplifier declared, the ladder's round trip *is* `G_PASSIVE` and is adopted as
such — which is how that calibration gets measured instead of remembered.


In [ ]:
# =============================================================================
#  STEP 4a - HOW MUCH GAIN IS THE AMPLIFIER ACTUALLY GIVING?
# =============================================================================
print('=' * 68)
print('STEP 4a - amplifier gain ladder')
print('=' * 68)
print('passive transfer without the amplifier : %.5f' % G_PASSIVE)
print('stopping when the amplifier would pass : %+.1f dBm' % AMP_OUT_MAX_DBM)
print('')
print('%9s %8s %10s %9s %10s' % ('drive', 'tone', 'G total', 'gain dB', 'amp dBm'))

amp_D, amp_G, amp_dbm = [], [], []
AMP_STOP_REASON = 'reached the amplifier rating'
try:
    D = AMP_TEST_START
    while D <= DRIVE_MAX:
        _raw, tone, Gt = measure_transfer(F0, D)
        if tone <= 0:
            AMP_STOP_REASON = 'no signal at the input'; break
        out = 10.0 + 20.0 * np.log10(max(tone / G_PASSIVE, 1e-12))
        amp_D.append(D); amp_G.append(Gt); amp_dbm.append(out)
        print('%9.5f %8.5f %10.4f %+9.1f %+10.1f'
              % (D, tone, Gt, 20 * np.log10(Gt / G_PASSIVE), out))
        if tone > 0.5:
            AMP_STOP_REASON = 'input approaching full scale'; break
        if len(amp_G) > 2 and Gt < 0.79 * max(amp_G):
            AMP_STOP_REASON = 'gain has fallen 2 dB - the amplifier is compressing'
            break
        if out + 20.0 * np.log10(AMP_TEST_STEP) > AMP_OUT_MAX_DBM:
            break
        D *= AMP_TEST_STEP
        if D > DRIVE_MAX:
            AMP_STOP_REASON = 'DRIVE_MAX reached before the amplifier rating'
finally:
    gen.off()

print('')
print('stopped: %s' % AMP_STOP_REASON)
if len(amp_G) < 2:
    raise RuntimeError('ladder too short to measure a gain - check the wiring')

amp_D = np.array(amp_D); amp_G = np.array(amp_G); amp_dbm = np.array(amp_dbm)
# The linear region: rungs within 10% of the best transfer. Never fewer than two
# - a "gain" from a single rung is a reading, not a measurement.
_lin_mask = amp_G > 0.90 * amp_G.max()
if int(np.count_nonzero(_lin_mask)) < 2:
    _lin_mask = np.zeros_like(amp_G, dtype=bool)
    _lin_mask[np.argsort(amp_G)[-2:]] = True
    print('  (only one rung stayed within 10% - the gain is provisional)')
G_WITH_AMP = float(np.mean(amp_G[_lin_mask]))
AMP_GAIN_DB = float(20 * np.log10(G_WITH_AMP / G_PASSIVE))
DRIVE_SUGGEST = float(min(amp_D[int(np.argmax(amp_G))],
                          amp_D[-1] * 10 ** ((AMP_OUT_MAX_DBM - 3.0 - amp_dbm[-1]) / 20.0)))

print('  amplifier gain          : %+.1f dB  (%.0fx), from %d linear rungs'
      % (AMP_GAIN_DB, 10 ** (AMP_GAIN_DB / 20), int(np.count_nonzero(_lin_mask))))
print('  round-trip transfer     : %.4f  (was %.4f)' % (G_WITH_AMP, G_PASSIVE))
print('  self-oscillation onset  : A = %.2f   (register limit 1.99)'
      % (1.0 / G_WITH_AMP))
print('-' * 68)
if 1.0 / G_WITH_AMP < 1.99:
    print('  REACHABLE - the instability region can appear in the Step 8c map.')
else:
    print('  NOT reachable: %.0f dB short. A second amplifier in the same path'
          % (20 * np.log10((1.0 / 1.99) / G_WITH_AMP)))
    print('  would close it; the map will otherwise be complete but with no')
    print('  black region.')

# ---- the one case that must abort, not adopt --------------------------------
#  DRIVE_MAX was derived from the DECLARED gain. If the measured gain is well
#  above it, the ceiling every measurement so far ran under was too permissive,
#  and that has to be re-run rather than patched. The protection against an
#  amplifier that is fitted but NOT declared is preflight_drive(), which measures
#  the actual input level and does not care what anyone declared.
if EXT_GAIN_DB > 0.0 and AMP_GAIN_DB > EXT_GAIN_DB + 3.0:
    raise RuntimeError(
        'EXT_GAIN_DB declares %.1f dB but the ladder measured %.1f dB. DRIVE_MAX '
        'was derived from the declared figure, so it is %.1f dB too permissive '
        'and every step above ran under a ceiling that was too high. Put %.1f in '
        'the configuration cell and re-run from the top.'
        % (EXT_GAIN_DB, AMP_GAIN_DB, AMP_GAIN_DB - EXT_GAIN_DB, AMP_GAIN_DB))

# ---- adopt --------------------------------------------------------------
#  Adoption may TIGHTEN a ceiling freely, but it must not loosen one on the
#  strength of a measurement that might be noise. A measured gain far BELOW the
#  declared one means the amplifier is absent, unpowered, or the signal never
#  got out of the noise floor - and all three are safe under the declared
#  ceiling and unsafe under the looser measured one.
print('')
if not AUTO_CALIBRATE:
    print('  AUTO_CALIBRATE is off: nothing adopted. EXT_GAIN_DB stays at %.1f dB'
          % EXT_GAIN_DB)
elif EXT_GAIN_DB > 0.0 and AMP_GAIN_DB < EXT_GAIN_DB - 3.0:
    print('  *** DECLARED %.1f dB, MEASURED %.1f dB - %.1f dB MISSING.'
          % (EXT_GAIN_DB, AMP_GAIN_DB, EXT_GAIN_DB - AMP_GAIN_DB))
    print('  NOT ADOPTED. The declared ceiling is the stricter of the two and is')
    print('  kept. The usual causes are an amplifier that is not fitted or not')
    print('  powered, a cable off, or a signal small enough that this ladder just')
    print('  measured noise - and the tighter limit is safe under all of them.')
    print('  Fix the chain, or set EXT_GAIN_DB = 0.0 if there really is no')
    print('  amplifier, and re-run from the top.')
elif EXT_GAIN_DB <= 0.0:
    if abs(AMP_GAIN_DB) <= 3.0:
        #  Nothing is fitted, so the ladder's round trip IS the passive transfer.
        #  This is how that calibration gets measured instead of remembered.
        G_PASSIVE = G_WITH_AMP
        print('  no amplifier declared and none measured: G_PASSIVE re-measured')
        print('  as %.5f and adopted.' % G_PASSIVE)
    else:
        print('  *** %.1f dB of gain measured but EXT_GAIN_DB says none is fitted.'
              % AMP_GAIN_DB)
        print('  Either G_PASSIVE (%.5f) is stale, or an amplifier is fitted and'
              % G_PASSIVE)
        print('  undeclared - in which case DRIVE_MAX is far too permissive.')
        print('  Resolve this before running any further step.')
else:
    EXT_GAIN_DB = AMP_GAIN_DB
    DRIVE_MAX, DRIVE_LINEAR_MAX = derive_limits(
        EXT_GAIN_DB, EXT_PAD_DB, AMP_IN_MAX_DBM, AMP_OUT_MAX_DBM)
    print('  ADOPTED: EXT_GAIN_DB = %.1f dB (measured), DRIVE_MAX = %.5f,'
          % (EXT_GAIN_DB, DRIVE_MAX))
    print('           DRIVE_LINEAR_MAX = %.5f. Step 4b scans inside these.'
          % DRIVE_LINEAR_MAX)
    if DRIVE_FRAC > DRIVE_MAX:
        DRIVE_FRAC = 0.5 * DRIVE_MAX
        print('  DRIVE_FRAC exceeded the new ceiling and has been cut to %.5f.'
              % DRIVE_FRAC)


In [ ]:
# --- Step 4a picture: the ladder --------------------------------------------
#  Axis limits come from the ladder itself, so the same cell is readable whatever
#  is in the amplifier box.
fig, (a1, a2) = plt.subplots(2, 1, figsize=(7.4, 5.4), sharex=True)

a1.loglog(amp_D, amp_G, 'o-', color=CLR_BLUE, label='measured transfer')
a1.loglog(amp_D[_lin_mask], amp_G[_lin_mask], 'o', ms=8, mfc='none',
          mec=CLR_RED, mew=1.4, label='linear region, averaged')
a1.axhline(G_PASSIVE, ls=':', lw=1.0, color=CLR_GREY, label='passive (no amplifier)')
a1.set_ylabel('round-trip transfer')
a1.set_title('Step 4a - gain %+.1f dB, onset A = %.2f  (%s)'
             % (AMP_GAIN_DB, 1.0 / G_WITH_AMP, AMP_STOP_REASON))
a1.legend(fontsize=8)

a2.semilogx(amp_D, amp_dbm, 's-', color=CLR_ORANGE, label='inferred amplifier output')
a2.axhline(AMP_OUT_MAX_DBM, ls='--', lw=1.1, color=CLR_RED,
           label='AMP_OUT_MAX_DBM (%+.1f dBm)' % AMP_OUT_MAX_DBM)
a2.set_xlabel('DRIVE_FRAC')
a2.set_ylabel('amplifier output / dBm')
a2.legend(fontsize=8)
plt.show()

print('Top: the transfer is flat while the amplifier is linear and falls when it')
print('compresses. Bottom: the ladder stops before the next rung would cross the')
print('rating - it infers the output from what arrives at the input, so it never')
print('needs to be told the gain.')


---
# Step 4b — The drive window

Steps 5 to 8 all assume the fit is describing the coil. Two things break that, pulling opposite
ways: **too little drive** leaves the fitted peak mostly receiver noise floor with the DAC out of
counts, and **too much** compresses the amplifier so the centre and width move. Nudging cannot
find the window between them, because from inside each failure looks like the other.

**The emulator settles it.** Enabled, it replaces the input with a resonator of known Q driven
through the same chain as the coil, so the drive at which the recovered Q leaves its known value
is the smallest signal at which *this* receiver and *this* fitter can be trusted — a property of
the instrument, not of the coil. The same ladder then runs on the coil: rungs clearing that
threshold are noise-safe, rungs where Q has left its plateau are compressed, and the window is
what is left.

**The recommendation is applied, not printed.** `DRIVE_FRAC` becomes the top of the window and
Step 3's refine-and-fit is re-run at it, so `F0`, the linewidth and `Q` all belong to the drive
Steps 5 to 8 actually use. A recommendation that has to be typed back into the configuration cell
is a recommendation that gets forgotten.


In [ ]:
# =============================================================================
#  STEP 4b - THE DRIVE WINDOW, AND ADOPTING IT
# =============================================================================
print('=' * 68)
print('STEP 4b - where is the fit describing the coil and not the noise?')
print('=' * 68)

_hi = min(DRIVE_LINEAR_MAX, DRIVE_MAX)
scan_D = _hi * np.logspace(-1.5, 0.0, DRIVE_SCAN_N)

# ---- part 1: what does a trustworthy fit need, in receiver counts? ----------
#  The emulator is a known answer measured through the identical chain, so this
#  threshold is a property of the instrument and not of the coil.
print('part 1 - calibrating the fitter against the emulator (known Q = %.0f)'
      % SELFTEST_Q)
print('%10s %8s %11s %10s' % ('drive', 'set_amp', 'amplitude', 'Q error'))
cal_amp, cal_err = [], []
try:
    f0q, qq = emu.config(SELFTEST_F0, SELFTEST_Q, delay=13, oshift=7)
    lwq = f0q / qq
    emu.clear(); emu.enable()
    acq.start(DEC_USE)
    gcal = f0q + np.linspace(-2.5, 2.5, 25) * lwq
    for D in scan_D:
        try:
            fc, zc = sweep(gcal, drive_frac=D, navg=N_AVG, level_guard=0.95)
            ft = fit_lorentzian(fc, np.abs(zc) ** 2)
            amp = float(np.sqrt(max(ft['peak'], 0.0)))
            err = abs(ft['Q'] / qq - 1.0)
            cal_amp.append(amp); cal_err.append(err)
            print('%10.5f %8d %11.4g %9.2f%%'
                  % (D, int(round(D * 8192)), amp, 100 * err))
        except Exception as exc:
            print('%10.5f %8d   fit failed (%s)'
                  % (D, int(round(D * 8192)), str(exc)[:34]))
finally:
    emu.disable(); gen.off()

_ok = [a for a, e in zip(cal_amp, cal_err) if e <= FIT_TRUST_TOL]
if _ok:
    MIN_AMP_FOR_FIT = float(min(_ok))
    print('  -> a fit is good to %.0f%% once the peak reaches %.4g counts'
          % (100 * FIT_TRUST_TOL, MIN_AMP_FOR_FIT))
else:
    MIN_AMP_FOR_FIT = float('inf')
    print('  -> the emulator was never recovered to %.0f%%. The receiver cannot'
          % (100 * FIT_TRUST_TOL))
    print('     fit anything at these drives; nothing below is meaningful.')

# ---- part 2: the same ladder on the coil -----------------------------------
print('')
print('part 2 - the coil')
print('%10s %8s %11s %9s %10s %8s'
      % ('drive', 'set_amp', 'amplitude', 'Q', 'floor/peak', 'resid'))
sc_D, sc_amp, sc_Q, sc_fl = [], [], [], []
try:
    acq.start(DEC_USE)
    gco = F0 + np.linspace(-2.5, 2.5, 41) * LW
    for D in scan_D:
        try:
            fk, zk = sweep(gco, drive_frac=D, navg=N_AVG, level_guard=0.95)
            ft = fit_lorentzian(fk, np.abs(zk) ** 2)
            amp = float(np.sqrt(max(ft['peak'], 0.0)))
            fl = float(np.sqrt(max(ft['base'], 0.0)) / max(amp, 1e-30))
            sc_D.append(D); sc_amp.append(amp); sc_Q.append(ft['Q']); sc_fl.append(fl)
            print('%10.5f %8d %11.4g %9.1f %10.3f %8.4f'
                  % (D, int(round(D * 8192)), amp, ft['Q'], fl, ft['resid']))
        except Exception as exc:
            print('%10.5f %8d   fit failed (%s)'
                  % (D, int(round(D * 8192)), str(exc)[:34]))
finally:
    gen.off()

if len(sc_D) < 3:
    #  Say WHY, because the two causes need opposite responses and the message
    #  is the only thing standing between them.
    _thr = ('the %.4g counts a trustworthy fit needs' % MIN_AMP_FOR_FIT
            if np.isfinite(MIN_AMP_FOR_FIT)
            else 'a threshold the emulator never reached either')
    raise RuntimeError(
        'only %d of %d rungs produced a fit, so there is no window to find. '
        'Largest peak %.4g counts, against %s. The receiver is not seeing the '
        'coil at any drive this chain allows: DRIVE_MAX is %.5f with %.1f dB of '
        'external gain declared, and Step 4a measured %+.1f dB. Fix the chain or '
        'the coupling; there is nothing here to fit.'
        % (len(sc_D), len(scan_D), max(sc_amp) if sc_amp else 0.0, _thr,
           DRIVE_MAX, EXT_GAIN_DB, AMP_GAIN_DB))
sc_D = np.array(sc_D); sc_amp = np.array(sc_amp); sc_Q = np.array(sc_Q)

# ---- the window ------------------------------------------------------------
_noise_ok = sc_amp >= MIN_AMP_FOR_FIT
_reg_ok = np.array([int(round(d * 8192)) >= 30 for d in sc_D])
DRIVE_BEST = float('nan')
if not np.any(_noise_ok & _reg_ok):
    DRIVE_WINDOW = (float('nan'), float('nan'))
    print('')
    print('  NO USABLE WINDOW at any drive this amplifier allows.')
    print('  The largest peak reached was %.4g against the %.4g a trustworthy fit'
          % (sc_amp.max(), MIN_AMP_FOR_FIT))
    print('  needs. Raise N_AVG, or accept that Q is the only number this rig can')
    print('  give and treat the drive-dependence tests as unavailable.')
else:
    _cand = sc_Q[_noise_ok & _reg_ok]
    _plateau = float(np.median(_cand))
    _flat = np.abs(sc_Q / _plateau - 1.0) < 0.03
    _good = _noise_ok & _reg_ok & _flat
    DRIVE_WINDOW = (float(sc_D[_good].min()), float(sc_D[_good].max()))
    DRIVE_BEST = float(sc_D[_good].max())
    print('')
    print('  noise-limited below : %.5f' % (sc_D[~_noise_ok].max()
                                            if np.any(~_noise_ok) else 0.0))
    print('  compressed above    : %s'
          % ('%.5f' % sc_D[_good].max() if np.any(~_flat & _noise_ok) else 'not reached'))
    print('  USABLE WINDOW       : %.5f to %.5f' % DRIVE_WINDOW)
    print('  RECOMMENDED         : DRIVE_FRAC = %.5f  (set_amp %d)'
          % (DRIVE_BEST, int(round(DRIVE_BEST * 8192))))
    print('  quality factor there: %.1f  (plateau %.1f)'
          % (sc_Q[sc_D == DRIVE_BEST][0], _plateau))
    if DRIVE_WINDOW[0] == DRIVE_WINDOW[1]:
        print('  Only one rung qualifies - the window is marginal. More averaging')
        print('  (N_AVG) would widen it from the bottom.')

# ---- adopt it, and re-measure the resonance there ---------------------------
#  A recommendation that has to be typed back into the configuration cell is a
#  recommendation that gets forgotten, and every number in Steps 5 to 8 would
#  then belong to a different drive from the one that produced F0 and LW.
DRIVE_ADOPTED = False
print('')
if not AUTO_CALIBRATE:
    print('AUTO_CALIBRATE is off: DRIVE_FRAC stays at %.5f.' % DRIVE_FRAC)
elif not np.isfinite(DRIVE_BEST):
    print('No window to adopt - DRIVE_FRAC stays at %.5f.' % DRIVE_FRAC)
elif abs(DRIVE_BEST / max(DRIVE_FRAC, 1e-12) - 1.0) < 0.05:
    print('DRIVE_FRAC %.5f is already at the top of the window - nothing to change.'
          % DRIVE_FRAC)
else:
    _old_drive, _old_f0 = DRIVE_FRAC, F0
    _old_lw, _old_q, _old_fl = LW, Q_SWEEP, FLOOR_FRAC
    DRIVE_FRAC = DRIVE_BEST
    DRIVE_ADOPTED = True
    print('ADOPTING DRIVE_FRAC = %.5f (was %.5f) and re-measuring the resonance'
          % (DRIVE_FRAC, _old_drive))
    DEC_USE, f3, p3, FIT = refine_and_fit(F0, LW, DRIVE_FRAC)
    F0 = FIT['f0']; LW = FIT['lw']; Q_SWEEP = FIT['Q']
    FLOOR_FRAC = float(np.sqrt(FIT['base'] / max(FIT['peak'], 1e-300)))
    print('')
    print('  %-20s %14s %14s' % ('', 'before', 'after'))
    print('  %-20s %14.6f %14.6f' % ('centre / MHz', _old_f0 / 1e6, F0 / 1e6))
    print('  %-20s %14.2f %14.2f' % ('linewidth / kHz', _old_lw / 1e3, LW / 1e3))
    print('  %-20s %14.1f %14.1f' % ('quality factor', _old_q, Q_SWEEP))
    print('  %-20s %14.3f %14.3f' % ('floor / peak', _old_fl, FLOOR_FRAC))
    _dq = abs(Q_SWEEP / max(_old_q, 1e-30) - 1.0)
    if _dq > 0.05:
        print('  Q moved %.1f%% with the drive - which is exactly what this step'
              % (100 * _dq))
        print('  exists to catch. The earlier value was noise-limited or')
        print('  compressed; everything below uses the new one.')
    else:
        print('  Q moved %.1f%% - the earlier drive was already inside the window.'
              % (100 * _dq))

#  The last word on whether this is a resonance at all. Step 3 only warned,
#  because its fit was taken before the drive was chosen by measurement; there is
#  no such excuse now, and Steps 5 to 8 would otherwise be fitting noise.
print('')
print('final fit at the working drive: residual %.4f of peak' % FIT['resid'])
if FIT['resid'] > SEARCH_FIT_MAX_RESID:
    raise RuntimeError(
        'residual %.3f of peak at the working drive %.5f, against a %.2f limit: '
        'this curve is not a Lorentzian, so it is not the coil. Raise N_AVG '
        '(currently %d) to buy signal-to-noise, improve the coupling, or lower '
        'SEARCH_STEP (%.2f kHz was used) if the coarse pass may have landed '
        'beside the real line.'
        % (FIT['resid'], DRIVE_FRAC, SEARCH_FIT_MAX_RESID, N_AVG,
           SEARCH_STEP_USED / 1e3))


In [ ]:
# --- Step 4b pictures --------------------------------------------------------
if len(sc_D) >= 3:
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(7.6, 5.6), sharex=True)
    a1.loglog(sc_D, sc_amp, 'o-', color=CLR_BLUE, label='fitted peak')
    if np.isfinite(MIN_AMP_FOR_FIT):
        a1.axhline(MIN_AMP_FOR_FIT, ls='--', color=CLR_RED,
                   label='fit trustworthy above this')
    a1.set_ylabel('peak amplitude / counts')
    a1.set_title('Step 4b - the drive window')
    a1.legend(fontsize=8)
    a2.semilogx(sc_D, sc_Q, 'o-', color=CLR_ORANGE)
    if np.all(np.isfinite(DRIVE_WINDOW)):
        a2.axvspan(DRIVE_WINDOW[0], DRIVE_WINDOW[1], color=CLR_GREEN, alpha=0.15,
                   label='usable window')
    a2.axvline(DRIVE_FRAC, ls='-', lw=1.4, color=CLR_RED,
               label='adopted DRIVE_FRAC = %.5f' % DRIVE_FRAC)
    a2.legend(fontsize=8)
    a2.set_xlabel('DRIVE_FRAC')
    a2.set_ylabel('fitted Q')
    plt.show()
    print('Top: the fit needs the peak above the dashed line. Bottom: Q must be on')
    print('its plateau. The window is where both hold - too little drive is noise,')
    print('too much is compression, and nudging cannot find it because from inside')
    print('each looks like the other.')

# The re-measured line, at the drive everything below actually uses.
if DRIVE_ADOPTED:
    _fine = np.linspace(f3.min(), f3.max(), 600)
    fig, ax = plt.subplots(figsize=(7.4, 3.4))
    ax.plot((f3 - F0) / 1e3, p3, 'o', ms=3.5, color=CLR_BLUE, label='measured')
    ax.plot((_fine - F0) / 1e3, lorentzian(_fine, FIT), '-', lw=1.4,
            color=CLR_ORANGE, label='Lorentzian fit')
    ax.axhline(FIT['base'], ls=':', lw=1.0, color=CLR_GREY, label='fitted floor')
    ax.set_xlabel('offset from centre / kHz')
    ax.set_ylabel('power (arb.)')
    ax.set_title('re-measured at the adopted drive %.5f - f0 = %.6f MHz, '
                 'lw = %.2f kHz, Q = %.1f' % (DRIVE_FRAC, F0 / 1e6, LW / 1e3, Q_SWEEP))
    ax.legend(fontsize=8)
    plt.show()


---
# Step 4c — What the coupling costs

Two numbers, one predicted and one measured. Where they differ, the difference is telling you
about the wiring rather than the coil.

**The prediction.** The pickup capacitor and everything after it — cable, connector, board input
— form a capacitive divider. The 13 kΩ reactance of 0.3 pF at 40 MHz is the *good* half of the
trade: almost nothing is drawn from the tank, so the Q measured above is close to the coil's own.

**The measurement.** How much signal actually arrived, against the floor from Step 2 — which
decides whether an amplifier is needed.

**The conclusion, in advance:** *driven* measurements work without one, because you can drive
harder. What needs an amplifier is anything driven by the coil's **own thermal noise**, which is
how the real experiment works. The last block estimates that directly.


In [ ]:
# =============================================================================
#  STEP 4c - WHAT THE COUPLING COSTS
# =============================================================================
print('=' * 68)
print('STEP 4c - the signal budget, at the adopted drive %.5f' % DRIVE_FRAC)
print('=' * 68)

w0 = 2.0 * np.pi * F0
Xc = 1.0 / (w0 * C_COUPLE)
divider = C_COUPLE / (C_COUPLE + C_STRAY_EST)

print('PREDICTED, from the values in the configuration cell')
print('  reactance of %.2f pF at %.3f MHz : %.1f kohm'
      % (C_COUPLE * 1e12, F0 / 1e6, Xc / 1e3))
print('  capacitive divider %.2f pF / (%.2f + %.1f) pF : %.4f  (%.1f dB)'
      % (C_COUPLE * 1e12, C_COUPLE * 1e12, C_STRAY_EST * 1e12,
         divider, 20 * np.log10(divider)))
print('  -> every 10 mV across the tank arrives as %.3f mV at the board'
      % (10.0 * divider))
# A series capacitor into a resistor R looks, from the tank, like a shunt
# resistance R (1 + (Xc/R)^2). With R = 1 Mohm and Xc = 13 kohm that is
# essentially R itself - the path is capacitive, not lossy.
r_shunt = R_IN_BOARD * (1.0 + (Xc / R_IN_BOARD) ** 2)
print('  equivalent shunt resistance across the tank : %.2f Mohm' % (r_shunt / 1e6))

print('')
print('MEASURED')
peak_counts = float(np.sqrt(FIT['peak']))
base_counts = float(np.sqrt(max(FIT['base'], 1e-300)))
print('  peak response at resonance : %.1f counts (arb. units of the receiver)'
      % peak_counts)
print('  off-resonance baseline     : %.1f counts' % base_counts)
if base_counts > 0:
    print('  resonant contrast          : %.1f dB'
          % (20 * np.log10(peak_counts / base_counts)))
else:
    print('  resonant contrast          : no floor resolved by the fit')

raw_peak, TONE_PEAK, G_FWD = measure_transfer(F0, DRIVE_FRAC)
print('  raw input at resonance     : %.4f of full scale = %.2f mV peak'
      % (raw_peak, 1e3 * raw_peak * V_FULLSCALE))
print('  the tone alone             : %.4f of full scale = %.2f mV peak'
      % (TONE_PEAK, 1e3 * TONE_PEAK * V_FULLSCALE))
print('  peak / tone                : %.2fx  (the excess is DC offset and noise)'
      % (raw_peak / max(TONE_PEAK, 1e-12)))
print('  one-way transfer DAC->ADC  : %.5f  (%.1f dB) at resonance'
      % (G_FWD, 20 * np.log10(max(G_FWD, 1e-12))))
if raw_peak > 3.0 * TONE_PEAK:
    print('    -> most of the input excursion is NOT the signal. Harmless for')
    print('       clipping, but check Step 2 for what else is arriving.')

# ---- the off-resonance floor ------------------------------------------------
#  Drive well away from the line. Whatever still arrives is a floor: crosstalk,
#  stray coupling, noise. It costs precision but does not bias the width.
_far = np.array([F0 - 40.0 * LW, F0 + 40.0 * LW])
_far = _far[(_far > 1.0e6) & (_far < 0.45 * FS_CLK)]
_ff, _zf = sweep(_far, drive_frac=DRIVE_FRAC, navg=N_AVG, level_guard=0.95)
FLOOR_DIRECT = float(np.mean(np.abs(_zf)))
_on = float(np.sqrt(FIT['peak'] + FIT['base']))

print('')
print('OFF-RESONANCE FLOOR')
print('  measured +-40 linewidths out  : %.1f counts' % FLOOR_DIRECT)
print('  on-resonance response         : %.1f counts' % _on)
print('  ratio                         : %.3f' % (FLOOR_DIRECT / max(_on, 1e-300)))
print('  same ratio from the fit       : %.3f' % FLOOR_FRAC)

# Feedback can only act on the part of the response that came from the coil.
G_FWD_RES = G_FWD * float(np.sqrt(FIT['peak'] / max(FIT['peak'] + FIT['base'], 1e-300)))
print('')
print('  transfer via the coil         : %.5f  (%.1f dB)  <- Step 8 uses this'
      % (G_FWD_RES, 20 * np.log10(max(G_FWD_RES, 1e-12))))

print('')
NOISE_RATIO = float('nan')
if L_EST is None:
    print('THERMAL NOISE: skipped. Set L_EST in the configuration cell.')
else:
    kB = 1.380649e-23
    r_par = Q_SWEEP * w0 * L_EST                 # tank equivalent parallel R
    E_N_COIL = np.sqrt(4.0 * kB * T_COIL * r_par)      # V per root hertz
    E_N_DIV = E_N_COIL * divider                       # ...after the divider
    v_lsb = V_FULLSCALE / 8192.0
    # 14-bit quantisation noise density over the first Nyquist zone
    E_BOARD = (v_lsb / np.sqrt(12.0)) / np.sqrt(FS_CLK / 2.0)
    NOISE_RATIO = float(E_N_DIV / E_BOARD)
    print('THERMAL NOISE, using L = %.3f uH' % (L_EST * 1e6))
    print('  tank equivalent parallel resistance : %.1f kohm' % (r_par / 1e3))
    print('  coil Johnson noise at resonance     : %.2f nV/rtHz' % (E_N_COIL * 1e9))
    print('  after the %.4f divider              : %.2f nV/rtHz'
          % (divider, E_N_DIV * 1e9))
    print('  board quantisation floor            : %.2f nV/rtHz (ideal; the real'
          % (E_BOARD * 1e9))
    print('                                         floor is higher - Step 2)')
    print('  ratio                               : %.3f  (coil noise is %.0fx'
          % (NOISE_RATIO, 1.0 / max(NOISE_RATIO, 1e-9)))
    print('                                         below the board floor)')
    if NOISE_RATIO < 3.0:
        print('  -> the coil\'s own noise is AT OR BELOW the board floor. Driven')
        print('     measurements are fine; noise-driven ones need an amplifier.')
    else:
        print('  -> the coil\'s own noise should be visible without an amplifier.')


---
# Step 4d — The coil against its own inductance and capacitance

The strongest test in the notebook, because `L` and `C` were measured *by a different instrument
on a different day*. `f₀ = 1/(2π√(LC))` over the coil's own `C_coil`, the `C_parallel` fitted to
emulate the trap, and the two coupling capacitors each in series with their cable. Nothing is
adjustable.

**Why the 39 pF matters more than it looks.** Alone, 423.5 nH with 1.59 pF resonates near
165 MHz — above Nyquist, invisible to this board at any setting. It is what puts the resonance in
the band, and at 25× the coil's own capacitance it dominates the error budget: a 5% part puts
±2.4% on the predicted centre against `C_coil`'s ±0.07%. **If prediction and measurement
disagree, suspect the capacitor's tolerance before the coil.**

**The inversion is the deliverable.** Take the *measured* `f₀` and the *known* `L`, solve for the
capacitance actually across the tank, subtract what is accounted for, and what remains is the
stray capacitance of the real assembly — holder, leads, connectors. Nothing else here can measure
it, and it is the number worth carrying to the trap. Stray can only *add*, so a negative result
means the model is too big.

**And the losses.** With `L` known, `R_series = ω₀L/Q` and `R_parallel = Q·√(L/C)` follow, and the
latter sets both the signal budget above and the loop gain of Step 8. Adding capacitance lowers
`f₀` and the characteristic impedance together, so it costs Q and signal without changing the
coil — which is what makes this comparable with a bare-coil measurement taken before the
capacitor went on.


In [ ]:
# =============================================================================
#  STEP 4d - THE COIL AGAINST ITS OWN L AND C
# =============================================================================
#  The only independent check in this notebook. L and C came from elsewhere; f0
#  and Q came from Steps 3 and 4b. Nothing here is fitted to anything above.
# =============================================================================
print('=' * 68)
print('STEP 4d - measured resonance against the known L and C')
print('=' * 68)

# ---- the capacitance actually across the tank -------------------------------
#  Each coupling capacitor sits in series with everything after it, so what the
#  tank sees from each port is the SERIES pair - and 0.3 pF in series with 25 pF
#  is 0.297 pF, i.e. the small capacitor wins almost completely. Two ports.
C_SER_PORT = C_COUPLE * C_STRAY_EST / (C_COUPLE + C_STRAY_EST)
C_PORTS = 2.0 * C_SER_PORT
C_MODEL = C_COIL + C_PARALLEL + C_PORTS
C_MODEL_ERR = np.sqrt(C_COIL_ERR ** 2 + C_PARALLEL_ERR ** 2)


def _f_of(L, Cq):
    return 1.0 / (2.0 * np.pi * np.sqrt(L * Cq))


F0_PRED = _f_of(L_COIL, C_MODEL)
# f0 goes as (LC)^-1/2, so a fractional error in L or C becomes HALF of one in f0
F0_PRED_ERR = F0_PRED * 0.5 * np.hypot(L_COIL_ERR / L_COIL, C_MODEL_ERR / C_MODEL)

print('capacitance across the tank')
print('  coil self-capacitance      : %6.2f +/- %.2f pF'
      % (C_COIL * 1e12, C_COIL_ERR * 1e12))
print('  fitted parallel capacitor  : %6.2f +/- %.2f pF'
      % (C_PARALLEL * 1e12, C_PARALLEL_ERR * 1e12))
print('  two coupling ports         : %6.2f pF  (%.3f pF each: %.1f pF in series '
      'with %.0f pF)'
      % (C_PORTS * 1e12, C_SER_PORT * 1e12, C_COUPLE * 1e12, C_STRAY_EST * 1e12))
print('  ---------------------------------------')
print('  total, modelled            : %6.2f +/- %.2f pF'
      % (C_MODEL * 1e12, C_MODEL_ERR * 1e12))
print('')

# ---- would this even be visible without the added capacitor? ----------------
F0_BARE = _f_of(L_COIL, C_COIL + C_PORTS)
print('without the %.0f pF the coil would sit at %.1f MHz'
      % (C_PARALLEL * 1e12, F0_BARE / 1e6))
if F0_BARE > 0.5 * FS_CLK:
    print('  -> above the %.1f MHz Nyquist limit. The added capacitance is what'
          % (0.5 * FS_CLK / 1e6))
    print('     makes this measurable at all; it is not a correction.')
print('')

# ---- prediction against measurement -----------------------------------------
_dev = F0 - F0_PRED
_sig = abs(_dev) / F0_PRED_ERR if F0_PRED_ERR > 0 else float('inf')
print('centre frequency')
print('  predicted from L and C     : %.3f +/- %.3f MHz'
      % (F0_PRED / 1e6, F0_PRED_ERR / 1e6))
print('  measured                   : %.3f MHz' % (F0 / 1e6))
print('  difference                 : %+.3f MHz  (%+.2f%%, %.1f sigma)'
      % (_dev / 1e6, 100 * _dev / F0_PRED, _sig))
if _sig <= 2.0 and abs(_dev) / F0_PRED <= 0.02:
    print('  AGREE - two independent routes to the same resonance')
elif _sig <= 2.0:
    print('  WITHIN THE ERROR BAR, BUT ONLY BECAUSE THE ERROR BAR IS WIDE.')
    print('  %+.1f%% is a real offset; it passes only because the %.0f pF part is'
          % (100 * _dev / F0_PRED, C_PARALLEL * 1e12))
    print('  assumed to +-%.0f%%. Measure that capacitor and re-run - the'
          % (100 * C_PARALLEL_ERR / C_PARALLEL))
    print('  inversion below is the sharper test.')
elif _sig <= 4.0:
    print('  MARGINAL. Before blaming the coil, check the tolerance of the %.0f pF'
          % (C_PARALLEL * 1e12))
    print('  part: it is %.0fx the coil capacitance and dominates the prediction.'
          % (C_PARALLEL / C_COIL))
else:
    print('  DISAGREE. In order of likelihood: the parallel capacitor is not the')
    print('  value marked on it; there is more stray capacitance in the assembly')
    print('  than C_STRAY_EST allows; L was measured at a very different frequency')
    print('  from %.1f MHz and the coil is not a simple inductor here.' % (F0 / 1e6))
print('')

# ---- the inversion: what capacitance is REALLY across the tank ---------------
#  f0 and L are both known, so C follows, and whatever it exceeds the model by is
#  stray capacitance in the real assembly - holder, leads, connectors - which
#  nothing else here can measure. This is the number to carry to the trap.
C_IMPLIED = 1.0 / (L_COIL * (2.0 * np.pi * F0) ** 2)
C_IMPLIED_ERR = C_IMPLIED * (L_COIL_ERR / L_COIL)
C_UNACCOUNTED = C_IMPLIED - C_MODEL
print('inverting: total capacitance implied by the measured f0 and the known L')
print('  implied C_total            : %6.2f +/- %.2f pF'
      % (C_IMPLIED * 1e12, C_IMPLIED_ERR * 1e12))
print('  modelled C_total           : %6.2f +/- %.2f pF'
      % (C_MODEL * 1e12, C_MODEL_ERR * 1e12))
print('  unaccounted for            : %+6.2f pF' % (C_UNACCOUNTED * 1e12))
if C_UNACCOUNTED > 2.0 * C_MODEL_ERR:
    print('  -> real stray capacitance in the assembly. Carry it to the trap: the')
    print('     holder and leads contribute %.1f pF and the trap build will see'
          % (C_UNACCOUNTED * 1e12))
    print('     something similar.')
elif C_UNACCOUNTED < -0.5e-12:
    print('  -> NEGATIVE BY %.2f pF, WHICH IS NOT PHYSICAL. Stray capacitance can'
          % abs(C_UNACCOUNTED * 1e12))
    print('     only add, so something in the model is too big. The %.0f pF part is'
          % (C_PARALLEL * 1e12))
    print('     %.0fx the coil capacitance and is the first suspect: measure it.'
          % (C_PARALLEL / C_COIL))
    print('     A high reading of L is the other. Do not carry the implied total')
    print('     forward until this is resolved.')
else:
    print('  -> consistent with zero. The assembly adds no measurable stray.')
print('')

# ---- losses, now that L is a number -----------------------------------------
W0 = 2.0 * np.pi * F0
Z0_TANK = np.sqrt(L_COIL / C_IMPLIED)
R_SERIES = W0 * L_COIL / Q_SWEEP
R_PARALLEL = Q_SWEEP * W0 * L_COIL
print('losses, from Q = %.1f and L = %.1f nH' % (Q_SWEEP, L_COIL * 1e9))
print('  characteristic impedance   : %.1f ohm   (sqrt(L/C))' % Z0_TANK)
print('  series resistance (ESR)    : %.3f ohm' % R_SERIES)
print('  parallel resistance        : %.1f kohm  (the tank at resonance)'
      % (R_PARALLEL / 1e3))
print('')

# ---- the cost of the added capacitor ----------------------------------------
#  The capacitor does not change the series resistance, and Q = w0 L / Rs, so Q
#  simply follows f0 down. That is what makes this run comparable with any
#  measurement taken before the capacitor went on.
Q_BARE_IMPLIED = Q_SWEEP * F0_BARE / F0
print('what the %.0f pF costs' % (C_PARALLEL * 1e12))
print('  Q here                     : %.1f at %.2f MHz' % (Q_SWEEP, F0 / 1e6))
print('  implied bare-coil Q        : %.0f at %.1f MHz  (same ESR, Q scales with f0)'
      % (Q_BARE_IMPLIED, F0_BARE / 1e6))
print('  characteristic impedance   : %.0f -> %.0f ohm'
      % (np.sqrt(L_COIL / (C_COIL + C_PORTS)), Z0_TANK))
print('  -> a factor %.1f in both Q and tank impedance. Compare against an earlier'
      % (F0_BARE / F0))
print('     bare-coil measurement using the implied value, not the one here.')
print('')

# ---- does the measured transfer match the model? ----------------------------
#  A measured value well ABOVE the prediction means signal is reaching the input
#  by some route other than the pickup capacitor - the two coax runs coupling to
#  each other is the usual one - and that route has its own phase, which would
#  corrupt Step 8's axis.
_Xc = 1.0 / (W0 * C_COUPLE)
_v_tank = R_PARALLEL / np.hypot(_Xc, R_PARALLEL)
_div = C_COUPLE / (C_COUPLE + C_STRAY_EST)
G_FWD_PRED = _v_tank * _div
print('forward transfer DAC -> coil -> ADC')
print('  drive side: %.1f kohm of reactance into a %.1f kohm tank -> %.3f'
      % (_Xc / 1e3, R_PARALLEL / 1e3, _v_tank))
print('  pickup divider                                          -> %.5f' % _div)
print('  predicted G_FWD            : %.5f  (%.1f dB)'
      % (G_FWD_PRED, 20 * np.log10(max(G_FWD_PRED, 1e-12))))
print('  measured G_FWD (Step 4c)   : %.5f  (%.1f dB)'
      % (G_FWD, 20 * np.log10(max(G_FWD, 1e-12))))
print('  measured / predicted       : %.2fx' % (G_FWD / max(G_FWD_PRED, 1e-12)))
#  The prediction is PASSIVE, so the amplifier has to come out before the ratio
#  means anything. Comparing them directly is how a working amplifier gets
#  mistaken for a stray coupling path.
AMP_LIN = 10.0 ** (max(EXT_GAIN_DB, 0.0) / 20.0)
G_FWD_PASSIVE = G_FWD / AMP_LIN
if EXT_GAIN_DB > 0.0:
    print('  less the %.1f dB amplifier  : %.5f  (a factor of %.0f)'
          % (EXT_GAIN_DB, G_FWD_PASSIVE, AMP_LIN))
_resid_x = G_FWD_PASSIVE / max(G_FWD_PRED, 1e-12)
print('  passive measured / predicted: %.2fx' % _resid_x)
if _resid_x > 3.0:
    print('  -> More signal arrives than this model predicts. That is a statement')
    print('     about the MODEL first: C_STRAY_EST and the coupling capacitor')
    print('     values are estimates and the transfer depends on both. Measure the')
    print('     parts and the cable capacitance before assuming a stray path. It')
    print('     only becomes a measurement problem if the lineshape is distorted')
    print('     with it - which Step 5 tests.')
elif _resid_x < 0.33:
    print('  -> LESS signal than predicted. Suspect the coupling capacitors being')
    print('     smaller than marked, or more cable capacitance than C_STRAY_EST.')
else:
    print('  -> consistent with coupling through the capacitors alone.')

# ---- what these measurements do and do not determine ------------------------
print('')
print('WHAT IS ACTUALLY PINNED DOWN')
_sum_model = (C_COUPLE + C_STRAY_EST)
_sum_meas = C_COUPLE * (_v_tank / max(G_FWD_PASSIVE, 1e-12))
print('  C_COUPLE + stray, from the measured transfer : %.1f pF' % (_sum_meas * 1e12))
print('  C_COUPLE + stray, from the config values     : %.1f pF' % (_sum_model * 1e12))
print('  -> the SUM is determined; the split rests on the marked %.2f pF and'
      % (C_COUPLE * 1e12))
print('     scales with it.')
# The degeneracy breaks if the pickup capacitor is taken out of the path: the
# tank then sees the whole input capacitance instead of ~C_COUPLE, and the
# frequency shift gives it from L and f0 alone - no capacitor value assumed.
print('  TO SETTLE IT: bypass the pickup cap and read the new f0 - C_in then')
print('  follows from L and f0 alone, no capacitor value assumed:')
for _ci in (10e-12, 20e-12, 30e-12):
    _fn = 1.0 / (2.0 * np.pi * np.sqrt(L_COIL * (C_IMPLIED + _ci - C_COUPLE)))
    print('     C_in = %2.0f pF -> f0 %.2f MHz' % (_ci * 1e12, _fn / 1e6))


---
# Step 5 — A second, independent width

The discipline that replaces a known answer is **measuring the same quantity two ways that fail
differently**.

**Why it is no longer the phase.** `sweep()` retunes two oscillators per point with two separate
register writes, and between them `Δφ = 2π(f_new − f_old)·Δt` opens up and accumulates —
radians per point, indistinguishable from a transport delay of tens of microseconds. `Δt` jitters,
so it cannot be calibrated out. Every fit here uses `|z|²` only.

**What replaces it: a different receiver.** The decimation sets the receiver's clean window, and a
window that is not small compared with the linewidth broadens what it measures. Repeat the fit at
twice the decimation — half the window — and a real width does not move. One that does was being
set by the instrument, not by the coil.


In [ ]:
# =============================================================================
#  STEP 5 - THE SAME WIDTH, THROUGH A DIFFERENT RECEIVER
# =============================================================================
print('=' * 68)
print('STEP 5 - a second, independent width')
print('=' * 68)

DEC_ALT = int(min(12, DEC_USE + 1))
if DEC_ALT == DEC_USE:
    DEC_ALT = int(max(8, DEC_USE - 1))

for _tag, _d in (('primary  ', DEC_USE), ('alternate', DEC_ALT)):
    print('%s : R = %5d, clean window +-%6.1f kHz  (%.1f%% of a linewidth)'
          % (_tag, 1 << _d, 0.10 * FS_CLK / (1 << _d) / 1e3,
             100 * 0.10 * FS_CLK / (1 << _d) / LW))
print('')

acq.start(DEC_ALT)
grid5 = F0 + np.linspace(-2.5, 2.5, 61) * LW
f5, z5 = sweep(grid5, drive_frac=DRIVE_FRAC, navg=N_AVG, level_guard=0.95)
gen.amp(0.0)
p5 = np.abs(z5) ** 2
acq.start(DEC_USE)                       # put the receiver back

try:
    FIT_ALT = fit_lorentzian(f5, p5)
    LW_ALT = FIT_ALT['lw']
    Q_ALT = FIT_ALT['Q']
    _rel = LW_ALT / LW - 1.0
    print('  width at R = %-5d : %.2f kHz, Q = %.1f' % (1 << DEC_USE, LW / 1e3, Q_SWEEP))
    print('  width at R = %-5d : %.2f kHz, Q = %.1f' % (1 << DEC_ALT, LW_ALT / 1e3, Q_ALT))
    print('  centre moved by    : %+.2f kHz  (%+.3f linewidths)'
          % ((FIT_ALT['f0'] - F0) / 1e3, (FIT_ALT['f0'] - F0) / LW))
    print('  disagreement       : %+.1f%%' % (100 * _rel))
    print('-' * 68)
    if abs(_rel) <= 0.10:
        print('  AGREE - the width is the coil\'s, not the receiver\'s')
    else:
        print('  DISAGREE by more than 10%. The receiver window is shaping the')
        print('  measurement. Raise the decimation until the two agree, and re-run')
        print('  Step 3 before trusting Q.')
except Exception as exc:
    FIT_ALT = None
    LW_ALT = Q_ALT = None
    print('  the alternate-decimation fit refused: %s' % exc)
    print('  No second width. Treat Q as unconfirmed.')


In [ ]:
# --- Step 5 picture: the two receivers on one axis ---------------------------
#  Both curves normalised to their own fitted peak, so what is being compared is
#  the SHAPE. Axis limits from the fits, so this is readable at any Q.
if FIT_ALT is not None:
    _span = 2.5 * max(LW, LW_ALT)
    _fine = np.linspace(-_span, _span, 600)
    fig, ax = plt.subplots(figsize=(7.6, 3.8))
    for _f, _p, _ft, _d, _c in ((f3, p3, FIT, DEC_USE, CLR_BLUE),
                                (f5, p5, FIT_ALT, DEC_ALT, CLR_ORANGE)):
        _n = max(_ft['peak'], 1e-300)
        ax.plot((_f - _ft['f0']) / 1e3, (_p - _ft['base']) / _n, 'o', ms=3.5,
                color=_c, alpha=0.7,
                label='R = %d,  lw %.2f kHz,  Q %.1f' % (1 << _d, _ft['lw'] / 1e3, _ft['Q']))
        ax.plot(_fine / 1e3, 1.0 / (1.0 + (2.0 * _fine / _ft['lw']) ** 2), '-',
                lw=1.3, color=_c)
    ax.set_xlim(-_span / 1e3, _span / 1e3)
    ax.set_xlabel('offset from that fit\'s own centre / kHz')
    ax.set_ylabel('power, baseline removed and peak normalised')
    ax.set_title('Step 5 - the same line through two receivers  (%+.1f%% apart)'
                 % (100 * (LW_ALT / LW - 1.0)))
    ax.legend(fontsize=8)
    plt.show()
    print('If the two shapes lie on top of each other, the width belongs to the')
    print('coil. If the wider receiver gives the wider line, it belongs to the')
    print('receiver.')


---
# Step 6 — Is the measurement linear?

The definitive test for "am I loading the coil?" is to weaken the coupling and watch Q stop
changing. With the capacitors soldered on that is unavailable, so this is the test that is.

Drive at three levels and fit each. A **linear** system returns the same centre and width at all
three; only the height changes. If either moves, something is compressing — the input stage, the
coil, or the amplifier — and every number taken at the higher level is suspect.

The scaling column (peak power against drive squared) is the least reliable of the three and
fails for reasons that are not the coil: a fitted floor comparable with the peak, or a DAC
amplitude register down at a few dozen counts, where its own quantisation costs several percent
of the fundamental. **Lowering the drive makes that worse, not better** — which is why Step 4b
finds the window by measurement instead.


In [ ]:
# =============================================================================
#  STEP 6 - LINEARITY
# =============================================================================
print('=' * 68)
print('STEP 6 - linearity against drive level')
print('=' * 68)

acq.start(DEC_USE)
grid6 = F0 + np.linspace(-3.0, 3.0, 41) * LW

lin_rows = []
lin_floor, lin_reg = [], []
for _fr in DRIVE_LIN_FRACS:
    frac = _fr * DRIVE_FRAC
    f6, z6 = sweep(grid6, drive_frac=frac, navg=N_AVG, level_guard=0.95)
    gen.amp(0.0)
    fit6 = fit_lorentzian(f6, np.abs(z6) ** 2)
    lin_rows.append((frac, fit6))
    _reg = int(round(frac * 8192))
    _floor = fit6['base'] / max(fit6['peak'], 1e-300)
    lin_floor.append(_floor); lin_reg.append(_reg)
    print('  drive %8.5f (%+5.1f dB, set_amp %4d) : f0 = %.6f MHz, lw = %6.2f kHz, '
          'Q = %6.1f, floor/peak = %.2f'
          % (frac, 20 * np.log10(_fr / DRIVE_LIN_FRACS[0]), _reg,
             fit6['f0'] / 1e6, fit6['lw'] / 1e3, fit6['Q'], _floor))

lin_f0 = np.array([r[1]['f0'] for r in lin_rows])
lin_q = np.array([r[1]['Q'] for r in lin_rows])
lin_pk = np.array([r[1]['peak'] for r in lin_rows])
lin_d = np.array([r[0] for r in lin_rows])

# NOT d_f0: Step 7 uses that name for its array of fitted centres.
d_f0_spread = (lin_f0.max() - lin_f0.min()) / LW
d_q = (lin_q.max() - lin_q.min()) / lin_q.mean()
# a linear system's peak power scales as the square of the drive
scale_err = lin_pk / (lin_pk[0] * (lin_d / lin_d[0]) ** 2) - 1.0

print('')
print('  centre spread across the ladder : %.3f linewidths' % d_f0_spread)
print('  Q spread across the ladder      : %.1f%%' % (100 * d_q))
print('  peak power vs drive-squared     : %s'
      % ', '.join('%+.1f%%' % (100 * e) for e in scale_err))
print('-' * 68)
_shape_ok = (d_f0_spread < 0.10) and (d_q < 0.05)
_scale_ok = np.max(np.abs(scale_err)) < 0.15
LINEAR_OK = _shape_ok and _scale_ok

# Is the test even answerable at this signal level? Two ways it is not: the
# fitted floor swamping the peak, and the DAC running out of counts.
_floor_bad = max(lin_floor) > 0.30 if lin_floor else False
_reg_bad = min(lin_reg) < 100 if lin_reg else False
print('  smallest set_amp used  : %d counts%s'
      % (min(lin_reg) if lin_reg else 0,
         '   <- too coarse for a scaling test' if _reg_bad else ''))
print('  worst floor / peak     : %.2f%s'
      % (max(lin_floor) if lin_floor else 0.0,
         '   <- the fit is measuring floor, not signal' if _floor_bad else ''))
print('-' * 68)
if LINEAR_OK:
    print('PASS - centre, width and scaling are all drive-independent')
elif _shape_ok and (_floor_bad or _reg_bad):
    LINEAR_OK = True
    print('INCONCLUSIVE ON SCALING, BUT SHAPE IS CLEAN.')
    print('  Centre and width are drive-independent (%.3f linewidths, %.1f%%),'
          % (d_f0_spread, 100 * d_q))
    print('  which is what non-linearity would move first. Only the power-versus-')
    print('  drive-squared column fails, and it cannot be trusted here:')
    if _floor_bad:
        print('    - the fitted floor is %.0f%% of the peak, so "peak" is mostly floor'
              % (100 * max(lin_floor)))
    if _reg_bad:
        print('    - the DAC amplitude register runs down to %d counts, where its'
              % min(lin_reg))
        print('      own quantisation costs several percent of the fundamental')
    print('  LOWERING THE DRIVE MAKES THIS WORSE. Step 4b already found the usable')
    print('  window by measurement and adopted the top of it.')
else:
    print('CHECK - the centre or width moves with drive level. That is real')
    print('        non-linearity. Set DRIVE_FRAC to the LOWER end of the Step 4b')
    print('        window, turn AUTO_CALIBRATE off, and repeat Steps 3 and 5.')


In [ ]:
# --- Step 6 picture ----------------------------------------------------------
fig, (a1, a2, a3) = plt.subplots(3, 1, figsize=(7.2, 6.4), sharex=True)

a1.semilogx(lin_d, (lin_f0 - F0) / LW, 'o-', color=CLR_BLUE)
a1.axhspan(-0.10, 0.10, color=CLR_GREEN, alpha=0.15, label='drive-independent')
a1.set_ylabel('centre / linewidths')
a1.set_title('Step 6 - linearity  (%s)' % ('PASS' if LINEAR_OK else 'CHECK'))
a1.legend(fontsize=8)

a2.semilogx(lin_d, lin_q, 'o-', color=CLR_ORANGE)
a2.axhline(Q_SWEEP, ls=':', lw=1.0, color=CLR_GREY, label='Q from Step 4b')
a2.set_ylabel('fitted Q')
a2.legend(fontsize=8)

a3.loglog(lin_d, lin_pk, 'o', color=CLR_GREEN, label='fitted peak power')
a3.loglog(lin_d, lin_pk[0] * (lin_d / lin_d[0]) ** 2, '--', lw=1.1, color=CLR_GREY,
          label='drive squared')
a3.set_xlabel('drive (fraction of full scale)')
a3.set_ylabel('peak power')
a3.legend(fontsize=8)
plt.show()

print('Top two panels are the test that matters: a linear system moves neither.')
print('The bottom panel fails for reasons that are not the coil - a fitted floor')
print('comparable with the peak, or a DAC register down at a few dozen counts.')


---
# Step 7 — Does anything move while you are measuring?

The emulated coil was exactly constant. A real one answers to room temperature, to airflow and to
anything mechanical — and if the centre moves during a feedback map, the map smears in a way that
looks like real structure.

Two quantities, easy to conflate: **drift** is a slow one-way trend that biases a long scan, and
**scatter** is point-to-point wobble that sets the floor on any single measurement. Both are
reported in **linewidths**, the unit in which they either matter or do not, and the verdict is
computed against the actual cost of the Step 8c map at its configured size.


In [ ]:
# =============================================================================
#  STEP 7 - DRIFT AND SCATTER
# =============================================================================
print('=' * 68)
print('STEP 7 - drift over %.0f minutes, sampled every %.0f s'
      % (DRIFT_MINUTES, DRIFT_PERIOD))
print('=' * 68)

acq.start(DEC_USE)
grid7 = F0 + np.linspace(-2.0, 2.0, 21) * LW      # short: this runs many times

t_start = time.time()
d_t, d_f0, d_q = [], [], []

while time.time() - t_start < DRIFT_MINUTES * 60.0:
    t_point = time.time()
    try:
        f7, z7 = sweep(grid7, drive_frac=DRIVE_FRAC, navg=32, level_guard=0.95)
        gen.amp(0.0)
        fit7 = fit_lorentzian(f7, np.abs(z7) ** 2)
        d_t.append(t_point - t_start)
        d_f0.append(fit7['f0'])
        d_q.append(fit7['Q'])
        print('  %6.1f s : f0 %+9.1f Hz from start, Q = %6.1f'
              % (d_t[-1], d_f0[-1] - d_f0[0], d_q[-1]))
    except Exception as exc:
        # One bad fit must not end a twenty-minute run. Record and continue.
        print('  %6.1f s : fit failed (%s)' % (time.time() - t_start, exc))
    time.sleep(max(0.0, DRIFT_PERIOD - (time.time() - t_point)))

gen.off()
d_t = np.array(d_t); d_f0 = np.array(d_f0); d_q = np.array(d_q)

if len(d_t) < 4:
    # Do not leave the summary printing nan. If the fits failed this often the
    # cause is upstream and saying so is more use than a blank.
    DRIFT_HZ_PER_MIN = SCATTER_HZ = float('nan')
    print('ONLY %d USABLE POINTS OUT OF %d ATTEMPTS - drift not characterised.'
          % (len(d_t), max(1, int(DRIFT_MINUTES * 60.0 / max(DRIFT_PERIOD, 1e-9)))))
    print('The fits themselves are failing, not the coil. Check the Step 3')
    print('residual and the Step 4b drive window before re-running.')
else:
    slope, icept = np.polyfit(d_t, d_f0, 1)
    resid = d_f0 - (slope * d_t + icept)
    DRIFT_HZ_PER_MIN = slope * 60.0
    SCATTER_HZ = float(np.std(resid))
    total = slope * d_t[-1]
    print('')
    print('  drift    : %+.1f Hz/min  =  %+.4f linewidths/min'
          % (DRIFT_HZ_PER_MIN, DRIFT_HZ_PER_MIN / LW))
    print('  over the whole run: %+.0f Hz = %+.3f linewidths' % (total, total / LW))
    print('  scatter  : %.1f Hz  =  %.4f linewidths' % (SCATTER_HZ, SCATTER_HZ / LW))
    print('  Q scatter: %.1f%%' % (100 * np.std(d_q) / np.mean(d_q)))
    print('-' * 68)
    # A feedback map takes minutes; if the centre moves an appreciable fraction
    # of a linewidth in that time the map is of several different systems. The
    # cost is computed from the map's CONFIGURED size, not from a remembered
    # figure, so changing FB_MAP_* changes this verdict with it.
    map_minutes = (FB_MAP_PHASES * FB_MAP_GAINS
                   * (FB_MAP_NPTS * (FB_MAP_SETTLE
                                     + (2 * FB_MAP_NAVG + GUARD_NSAMP) * 1.0e-5)
                      + 0.05)) / 60.0
    move = abs(DRIFT_HZ_PER_MIN) * map_minutes / LW
    print('  the Step 8c map as configured takes about %.1f minutes, over which the'
          % map_minutes)
    print('  centre would move %.3f linewidths' % move)
    if move > 0.10:
        print('  -> enough to smear the map. Either shorten the scan (fewer phases')
        print('     or gains), run a servo during it, or re-centre between rows.')
    else:
        print('  -> negligible. The scan can run open-loop.')

    fig, (b1, b2) = plt.subplots(2, 1, figsize=(7.5, 4.6), sharex=True)
    b1.plot(d_t / 60.0, (d_f0 - d_f0[0]) / 1e3, 'o-', ms=3.5, lw=1.0)
    b1.plot(d_t / 60.0, (slope * d_t + icept - d_f0[0]) / 1e3, '-', lw=1.2,
            color=CLR_ORANGE, label='linear drift')
    b1.set_ylabel('centre shift / kHz')
    b1.legend(fontsize=8)
    b1.set_title('Step 7 - drift %+.1f Hz/min, scatter %.1f Hz'
                 % (DRIFT_HZ_PER_MIN, SCATTER_HZ))
    b2.plot(d_t / 60.0, d_q, 'o-', ms=3.5, lw=1.0, color=CLR_GREEN)
    b2.set_xlabel('time / minutes')
    b2.set_ylabel('Q')
    plt.show()


---
# Step 8 — Feedback against a real coil

Feedback takes the coil's own signal, shifts its phase, scales it and adds it back, changing
**Q** — feedback aligned with the coil's motion cancels part of its loss — and the **centre**,
where feedback aligned with its displacement acts like retuning.

Five things differ from every previous stage, and all are handled by measuring, not assuming.

**1. The phase axis has no absolute meaning.** Cables, connectors and the coil add delay the
latency correction cannot know about, so the two special phases are *found* and everything is
reported relative to them. A fixed offset is expected.

**2. There is no overload flag.** The emulator's sticky clip bit has been the guard for ten
stages and it is gone. Every point checks the input level and abandons rather than fitting
nonsense.

**3. The loop may not have enough gain, and that is a result.** The feedback crosses both
coupling capacitors, carrying the divider *twice*. If `K = A × G_fwd` cannot reach a percent or
so, no phase looks different from any other — and the notebook says so up front rather than
letting an `argmax` invent a narrowing phase out of fit noise.

**4. Everything is measured differentially.** Feedback is antisymmetric in phase — what narrows
the line at φ broadens it at φ+180 — while drift, baseline bias and a sagging drive are not.
Subtracting the opposite phase cancels those, doubles the signal, and the result is checked
against the measurement's own scatter before any phase is named.

**5. Self-oscillation is approached, not entered.** Past a certain gain the coil sustains itself.
On the emulator that was safe; here it saturates the input, and a saturated input still looks
vaguely like a resonance. The boundary is **extrapolated** from the linear region.


In [ ]:
# =============================================================================
#  STEP 8a - THE PHASE AXIS: find the two special phases
# =============================================================================
print('=' * 68)
print('STEP 8 - feedback on the real coil')
print('=' * 68)

acq.start(DEC_USE)
whole, frac = ps.config_delay(F0)
print('quarter-period delay line: %d whole clocks + %.3f' % (whole, frac))
print('')

# =============================================================================
#  LOOP-GAIN BUDGET
# =============================================================================
#  The feedback path crosses BOTH coupling capacitors, so the round trip carries
#  the divider twice. Step 4c measured the one-way half as G_FWD, so K = A x G_FWD
#  and K = 1 is self-oscillation.
# =============================================================================
_HEADROOM = 0.95 - DRIVE_FRAC
if _HEADROOM <= 0.0:
    raise RuntimeError('DRIVE_FRAC = %.2f leaves no converter headroom for any '
                       'feedback at all. Lower it before running Step 8.'
                       % DRIVE_FRAC)
#  CLOSED-LOOP. Feedback multiplies the converter signal by 1/(1-K), so dividing
#  the headroom by the OPEN-loop level under-estimates it badly once K is large.
#     DAC:  A <= (H-D) / (D*r + G*(H-D))       ADC:  A <= (1 - D*r/guard) / G
_r = raw_peak / max(DRIVE_FRAC, 1e-9)          # peak-based transfer
_H = 0.95
_a_dac = (_H - DRIVE_FRAC) / max(DRIVE_FRAC * _r + G_FWD_RES * (_H - DRIVE_FRAC), 1e-12)
_a_adc = (1.0 - DRIVE_FRAC * _r / FB_LEVEL_GUARD) / max(G_FWD_RES, 1e-12)
FB_GAIN_CEIL = float(max(0.0, min(FB_GAIN_MAX, _a_dac, _a_adc)))
#  Keep the probe clear of the boundary. FB_GAIN_CEIL can sit at K = 1 exactly
#  (the closed-loop headroom formula stops there), and probing ON the boundary
#  makes half the phases oscillate, leaves the cosine fit with too few points,
#  and hands Step 8b a NaN phase.
FB_PROBE_USE = float(min(FB_GAIN_PROBE, FB_GAIN_CEIL, 0.6 / max(G_FWD_RES, 1e-9)))
FB_GAINS_USE = [round(fr * FB_GAIN_CEIL, 4) for fr in FB_GAIN_FRACS]

print('  gain ceiling: %.2f from converter headroom, %.2f from the input guard'
      % (_a_dac, _a_adc))
if FB_GAIN_CEIL < 0.5 * FB_GAIN_MAX:
    _d_ok = 0.9 * (_H - 0.0) / (1.0 + FB_GAIN_MAX * _r / max(1.0 - FB_GAIN_MAX * G_FWD_RES, 0.05))
    print('  *** THE DRIVE IS EATING THE FEEDBACK RANGE: at DRIVE_FRAC = %.2f'
          % DRIVE_FRAC)
    print('  only A = %.2f fits (K = %.3f). Lower it to about %.2f and re-run.'
          % (FB_GAIN_CEIL, FB_GAIN_CEIL * G_FWD_RES, max(0.05, min(0.3, _d_ok))))

#  G_FWD_RES, not G_FWD: feedback acts only on the part of the returning signal
#  that came out of the resonator. The off-resonance floor is fed back too, but
#  it never touches the coil, so counting it over-predicts the loop gain.
K_MAX_POSSIBLE = FB_GAIN_CEIL * G_FWD_RES
K_AT_PROBE = FB_PROBE_USE * G_FWD_RES
print('loop-gain budget')
print('  one-way transfer, all of it    : %.5f  (%.1f dB)'
      % (G_FWD, 20 * np.log10(max(G_FWD, 1e-12))))
print('  ...of which went via the coil  : %.5f  (%.1f dB)   <- what feedback acts on'
      % (G_FWD_RES, 20 * np.log10(max(G_FWD_RES, 1e-12))))
print('  ...the rest is the floor       : %.0f%% of the returning amplitude'
      % (100 * (1.0 - G_FWD_RES / max(G_FWD, 1e-12))))
print('  converter headroom for feedback: %.3f of full scale' % _HEADROOM)
print('  largest gain that fits in it   : %.2f' % FB_GAIN_CEIL)
print('  gain ceiling actually used     : %.2f%s'
      % (FB_GAIN_CEIL,
         '  (capped by headroom, not by FB_GAIN_MAX = %.2f)' % FB_GAIN_MAX
         if FB_GAIN_CEIL < FB_GAIN_MAX else '  (the register ceiling)'))
print('  round trip at that ceiling     : K = %.4f' % K_MAX_POSSIBLE)
print('  Q change that would produce    : %+.1f%%'
      % (100 * (1.0 / (1.0 - min(K_MAX_POSSIBLE, 0.99)) - 1.0)))
print('  K at the probe gain %.2f       : %.4f  (Q %+.1f%%)'
      % (FB_PROBE_USE, K_AT_PROBE,
         100 * (1.0 / (1.0 - min(K_AT_PROBE, 0.99)) - 1.0)))
print('  drive + feedback at the DAC    : %.3f of full scale'
      % (DRIVE_FRAC + FB_GAIN_CEIL * raw_peak))

if K_MAX_POSSIBLE < 0.01:
    print('')
    print('  *** THIS SWEEP CANNOT SHOW A FEEDBACK EFFECT. ***')
    print('  Even at the register ceiling the loop gain is %.4f, which moves Q by'
          % K_MAX_POSSIBLE)
    print('  %.1f%% - below what a Lorentzian fit resolves. The sweep will run and'
          % (100 * K_MAX_POSSIBLE))
    print('  return flat Q against phase. That IS the measurement: with both ports')
    print('  coupled through %.1f pF there is not enough gain in the loop to move a'
          % (C_COUPLE * 1e12))
    print('  real coil, and an amplifier in the pickup path is what changes it.')
elif K_MAX_POSSIBLE < 0.05:
    print('')
    print('  Marginal but workable: %.1f%% of Q at the ceiling. The differential'
          % (100 * K_MAX_POSSIBLE))
    print('  analysis below is what makes an effect this small readable.')
else:
    print('')
    print('  Comfortable: %.1f%% of Q at the ceiling.' % (100 * K_MAX_POSSIBLE))
print('')
print('probing %d phases at gain %.2f, %d averages per point'
      % (len(FB_PHASES), FB_PROBE_USE, FB_NAVG))
print('')

# +-4 linewidths, and wider than it looks it needs to be. Feedback both narrows
# the line and shifts it, and a grid that the shifted line has walked out of
# fails to fit at exactly the phases that matter most.
grid8 = F0 + np.linspace(-4.0, 4.0, 33) * LW

fb_phi, fb_q, fb_df = [], [], []
try:
    for phi in FB_PHASES:
        ps.set(FB_PROBE_USE, phi)
        time.sleep(SETTLE)
        gen.freq(F0); gen.amp(DRIVE_FRAC); gen.run(); time.sleep(0.02)
        lvl = sc.peak_frac(GUARD_NSAMP)
        gen.amp(0.0)
        if lvl > FB_LEVEL_GUARD:
            print('  %4d deg : ABANDONED, input at %.2f of full scale' % (phi, lvl))
            fb_phi.append(phi); fb_q.append(np.nan); fb_df.append(np.nan)
            continue
        try:
            f8, z8 = sweep(grid8, drive_frac=DRIVE_FRAC, navg=FB_NAVG,
                           level_guard=FB_LEVEL_GUARD)
            gen.amp(0.0)
            fit8 = fit_lorentzian(f8, np.abs(z8) ** 2)
            # A saturated or oscillating loop still returns a "fit", and it can
            # be absurd. Anything this far from the unfed line is not a
            # measurement of the coil.
            if not (0.02 * LW < fit8['lw'] < 50.0 * LW):
                raise ValueError('width %.1f kHz is nowhere near the %.1f kHz '
                                 'line - loop saturated?'
                                 % (fit8['lw'] / 1e3, LW / 1e3))
            fb_phi.append(phi); fb_q.append(fit8['Q']); fb_df.append(fit8['f0'] - F0)
            print('  %4d deg : Q = %7.1f  (x%.2f),  centre %+8.1f Hz (%+.3f lw)'
                  % (phi, fit8['Q'], fit8['Q'] / Q_SWEEP, fb_df[-1], fb_df[-1] / LW))
        except Exception as exc:
            fb_phi.append(phi); fb_q.append(np.nan); fb_df.append(np.nan)
            print('  %4d deg : fit failed (%s)' % (phi, exc))
finally:
    # The feedback loop is closed while this runs. Any exception, or
    # a Ctrl-C between two phases, must not leave it that way.
    ps.off()
    gen.off()

fb_phi = np.array(fb_phi, dtype=float)
fb_q = np.array(fb_q, dtype=float)
fb_df = np.array(fb_df, dtype=float)

if np.all(~np.isfinite(fb_q)):
    raise RuntimeError('every feedback point failed - check levels and coupling')

# =============================================================================
#  DIFFERENTIAL ANALYSIS - phi against phi + 180
# =============================================================================
#  Feedback is antisymmetric in phase; drift, baseline bias and a sagging drive
#  are not. Subtracting the opposite phase cancels those and doubles the signal.
# =============================================================================
half = len(fb_phi) // 2
_opp = np.array([int(np.argmin(np.abs(((fb_phi - (p + 180.0)) + 180.0) % 360.0 - 180.0)))
                 for p in fb_phi[:half]])
dq = fb_q[:half] - fb_q[_opp]            # antisymmetric: the feedback signature
ddf = fb_df[:half] - fb_df[_opp]

# ---- fit a cosine, do not take an argmax ------------------------------------
#  Theory says the differential IS a cosine. An argmax returns a grid point,
#  is biased to the ends of the half-turn, and discards eight of nine points.
def _fit_cos(phi_deg, y):
    """Least-squares fit of y = D cos(phi - phi0). Returns (D, phi0_deg)."""
    good = np.isfinite(y)
    if int(np.count_nonzero(good)) < 3:
        return float('nan'), float('nan')
    p = np.radians(np.asarray(phi_deg, dtype=float)[good])
    basis = np.column_stack([np.cos(p), np.sin(p)])
    coef, _, _, _ = np.linalg.lstsq(basis, np.asarray(y, dtype=float)[good], rcond=-1)
    return float(np.hypot(coef[0], coef[1])), float(np.degrees(np.arctan2(coef[1], coef[0])))

_n_osc = int(np.sum(~np.isfinite(fb_q)))
if _n_osc:
    print('  %d of %d phases gave no fit (oscillating or level-guarded)'
          % (_n_osc, len(fb_q)))
DQ_AMP, PHI_NARROW = _fit_cos(fb_phi[:half], dq)
DDF_AMP, PHI_SHIFT = _fit_cos(fb_phi[:half], ddf)
PHI_NARROW = float(PHI_NARROW % 360.0)
PHI_SHIFT = float(PHI_SHIFT % 360.0)
PHI_BROAD = (PHI_NARROW + 180.0) % 360.0

# Noise from successive differences along a curve that should be smooth: the
# measurement declaring its own error bar rather than us assuming one.
_fin = fb_q[np.isfinite(fb_q)]
Q_NOISE = float(np.std(np.diff(_fin)) / np.sqrt(2.0)) if len(_fin) > 2 else float('nan')
FB_DETECTED = bool(np.isfinite(Q_NOISE) and DQ_AMP > 3.0 * Q_NOISE)

# Q+ = Q0/(1-K) and Q- = Q0/(1+K), so the differential amplitude is
#     m = DQ_AMP / (2 Q0) = K / (1 - K^2)
# which is K only for small K - it over-reads by 25% at K = 0.5 and diverges at
# K = 1. Invert it exactly instead: K = (-1 + sqrt(1 + 4 m^2)) / (2 m).
#  (Q+ - Q-)/(Q+ + Q-) = K exactly, for every pair, with Q0 cancelled. The
#  cosine amplitude is only used to locate the phase: at large K the curve is
#  Q0/(1 - K cos phi), which is peaked rather than sinusoidal, so its amplitude
#  is not a good measure of K even though its symmetry still fixes the phase.
_sq = fb_q[:half] + fb_q[_opp]
with np.errstate(invalid='ignore', divide='ignore'):
    _kp = dq / _sq
K_MEASURED = float(np.nanmax(_kp)) if np.any(np.isfinite(_kp)) else 0.0

print('')
print('  differential amplitude       : %.2f%% of Q0' % (100 * DQ_AMP / Q_SWEEP))
print('  MEASURED K                   : %.4f' % K_MEASURED)
print('  predicted K, resonant only   : %.4f' % K_AT_PROBE)
print('  (counting the floor as signal would predict %.4f)'
      % (FB_PROBE_USE * G_FWD))
print('  point-to-point Q noise       : %.2f%% of Q0' % (100 * Q_NOISE / Q_SWEEP))
print('  signal-to-noise              : %.1f' % (DQ_AMP / max(Q_NOISE, 1e-12)))
print('')

if not FB_DETECTED:
    print('  NO FEEDBACK EFFECT DETECTED - and the phases below are meaningless.')
    print('  The modulation is under three times the noise, so the "narrowing')
    print('  phase" is just wherever the noise peaked. Do NOT read it as physics,')
    print('  and do not read Step 8b\'s K either.')
    print('')
    if K_MAX_POSSIBLE < 0.01:
        print('  This agrees with the budget above: the loop simply has no gain.')
        print('  The fix is not in this notebook. Either put a low-noise amplifier')
        print('  in the pickup path before IN1, or couple more strongly than')
        print('  %.1f pF - both of which change the divider, and with it every' % (C_COUPLE * 1e12))
        print('  number Step 4c predicted.')
    else:
        print('  The budget said an effect SHOULD be visible, so this disagreement')
        print('  is worth chasing: check the delay line configured above, check')
        print('  that the phase shifter is not bypassed, and re-run Step 4c to')
        print('  confirm the coupling has not changed.')
else:
    print('  narrowing phase (cosine fit)   : %.1f deg' % PHI_NARROW)
    print('  broadening phase (opposite)    : %.1f deg' % PHI_BROAD)
    print('  largest centre shift at        : %.1f deg' % PHI_SHIFT)
    _sep = (PHI_SHIFT - PHI_NARROW) % 360.0
    print('  separation between them        : %.1f deg  (theory: 90 or 270)' % _sep)
    _off = min(abs(_sep - 90.0), abs(_sep - 270.0))
    if _off > 35.0:
        print('  WARNING: the two effects should be a quarter turn apart and are')
        print('  %.0f degrees off it. At a signal-to-noise of %.1f that is unlikely'
              % (_off, DQ_AMP / max(Q_NOISE, 1e-12)))
        print('  to be scatter - suspect the loop compressing, or a second path')
        print('  with its own phase (Step 4d compares the measured transfer against the model).')
    else:
        print('  -> a quarter turn, as theory demands. The phase axis is sound.')

    _ratio = K_MEASURED / max(FB_PROBE_USE * G_FWD_RES, 1e-12)
    print('')
    print('  measured K / predicted K : %.2fx' % _ratio)
    print('  round-trip transfer from the loop : %.5f  (%.1f dB)'
          % (K_MEASURED / FB_PROBE_USE,
             20 * np.log10(max(K_MEASURED / FB_PROBE_USE, 1e-12))))
    print('  The measured value is the measurement; the budget rests on estimated')
    print('  capacitances, so a factor of two either way means little.')
    if not (0.33 < _ratio < 3.0):
        print('  -> over 3x apart: compare the transfer above with Step 4b.')


In [ ]:
# --- Step 8a picture ---------------------------------------------------------
#  The differential curves are the measurement; the raw ones are shown faint
#  behind them so a common-mode drift is visible for what it is.
fig, (c1, c2) = plt.subplots(2, 1, figsize=(7.5, 5.4), sharex=True)

_rel = (fb_phi[:half] - PHI_NARROW + 180.0) % 360.0 - 180.0
_o = np.argsort(_rel)
_relall = (fb_phi - PHI_NARROW + 180.0) % 360.0 - 180.0
_oa = np.argsort(_relall)

c1.plot(_relall[_oa], (fb_q[_oa] - np.nanmean(fb_q)) / Q_SWEEP * 100.0, '.',
        ms=4, color='0.75', label='raw, mean removed')
c1.plot(_rel[_o], dq[_o] / Q_SWEEP * 100.0, 'o-', ms=4, lw=1.1,
        color=CLR_BLUE, label='differential, phi - (phi+180)')
c1.axhline(0.0, ls=':', lw=1.0, color=CLR_GREY)
if np.isfinite(Q_NOISE):
    c1.axhspan(-300.0 * Q_NOISE / Q_SWEEP, 300.0 * Q_NOISE / Q_SWEEP,
               color=CLR_RED, alpha=0.12, label='3x noise')
c1.set_ylabel('Q change / % of Q0')
c1.set_title('Step 8 - feedback at gain %.2f   [%s]'
             % (FB_PROBE_USE,
                'effect detected' if FB_DETECTED else 'NO effect above the noise'))
c1.legend(fontsize=8)

c2.plot(_relall[_oa], (fb_df[_oa] - np.nanmean(fb_df)) / LW, '.', ms=4, color='0.75')
c2.plot(_rel[_o], ddf[_o] / LW, 'o-', ms=4, lw=1.1, color=CLR_ORANGE)
c2.axhline(0.0, ls=':', lw=1.0, color=CLR_GREY)
c2.set_xlabel('feedback phase relative to the narrowing phase / degrees')
c2.set_ylabel('centre shift / linewidths')
plt.show()

if FB_DETECTED:
    print('The two curves should be a quarter turn apart: where the Q difference')
    print('peaks the centre-shift difference crosses zero, and vice versa. The SIGN')
    print('of the lower curve depends on how the feedback reaches the coil and is')
    print('not expected to match the paper - their loop couples through a')
    print('capacitor, this one sums into the drive.')
else:
    print('Both curves are consistent with zero. The shaded band is three times the')
    print('measurement\'s own point-to-point noise; a real feedback effect would')
    print('leave it. Nothing here is a statement about the coil - it is a statement')
    print('about how much gain survives two 0.3 pF capacitors.')


In [ ]:
# =============================================================================
#  STEP 8b - HOW THE EFFECT SCALES WITH GAIN, and where the boundary is
# =============================================================================
#  Define K from the measured quality factor by Q_eff = Q0 / (1 - K). K is then
#  the fraction of the coil's loss the feedback is cancelling, it should be
#  proportional to the gain, and K = 1 is the self-oscillation boundary.
#
#  The boundary is EXTRAPOLATED from the linear region, never approached. See
#  the note above this step.
# =============================================================================
print('')
print('STEP 8b - K against gain at the narrowing phase')
print('-' * 68)

#  Measure each gain at BOTH phi_narrow and phi_narrow + 180. With
#  Q+ = Q0/(1-K) and Q- = Q0/(1+K),
#      (Q+ - Q-) / (Q+ + Q-) = K
#  exactly, with Q0 cancelled - no reference measurement, no drift term.
def _q_at(phi, A):
    """Quality factor with the loop closed at gain A and phase phi."""
    ps.set(A, phi)
    time.sleep(SETTLE)
    gen.freq(F0); gen.amp(DRIVE_FRAC); gen.run(); time.sleep(0.02)
    lvl = sc.peak_frac(GUARD_NSAMP)
    gen.amp(0.0)
    if lvl > FB_LEVEL_GUARD:
        raise RuntimeError('input at %.2f of full scale' % lvl)
    fq, zq = sweep(grid8, drive_frac=DRIVE_FRAC, navg=FB_NAVG,
                   level_guard=FB_LEVEL_GUARD)
    _f = fit_lorentzian(fq, np.abs(zq) ** 2)
    if not (0.02 * LW < _f['lw'] < 50.0 * LW):
        raise ValueError('width %.1f kHz is nowhere near the %.1f kHz line'
                         % (_f['lw'] / 1e3, LW / 1e3))
    return _f


FB_8B_OK = bool(np.isfinite(PHI_NARROW) and FB_DETECTED)
if not FB_8B_OK:
    # Skip rather than halt: Step 8c is still worth running, and a Lorentzian
    # fitted at a phase that means nothing is worse than no number at all.
    k_slope = A_ONSET = k_free = k_int = float('nan')
    gains_ok, ks = [], []
    print('  SKIPPED - Step 8a found no usable narrowing phase (effect detected'
          ' = %s).' % FB_DETECTED)
    print('  If the probe gain sat on the oscillation boundary, lower')
    print('  FB_GAIN_PROBE; if the loop simply has no gain, this is the answer.')

gains_ok, ks = [], []
if FB_8B_OK:
  try:
      for A in FB_GAINS_USE:
          try:
              fplus = _q_at(PHI_NARROW, A)
              fminus = _q_at(PHI_BROAD, A)
              qp, qm = fplus['Q'], fminus['Q']
              K = (qp - qm) / (qp + qm)
              gains_ok.append(A); ks.append(K)
              print('  gain %.2f : Q+ = %7.1f, Q- = %7.1f  ->  K = %.4f  '
                    '(centre split %+.0f Hz)'
                    % (A, qp, qm, K, fplus['f0'] - fminus['f0']))
          except Exception as exc:
              print('  gain %.2f : abandoned (%s)' % (A, exc))
  finally:
      # Highest gains in the notebook are applied here. Open the loop
      # and kill the drive on every exit path, including Ctrl-C.
      ps.off()
      gen.off()

if len(gains_ok) >= 2:
    gains_ok = np.array(gains_ok); ks = np.array(ks)
    # forced through the origin: zero gain must mean zero effect
    k_slope = float(np.sum(gains_ok * ks) / np.sum(gains_ok ** 2))
    # ALSO fit with a free intercept: anything gain-independent otherwise gets
    # absorbed into the origin-forced slope and biases the loop gain.
    k_free = k_int = float('nan')
    if len(gains_ok) >= 3:
        k_free, k_int = np.polyfit(gains_ok, ks, 1)
        k_free = float(k_free); k_int = float(k_int)
    A_ONSET = 1.0 / k_slope if k_slope > 0 else float('nan')
    pred = k_slope * gains_ok
    lin_err = float(np.max(np.abs(ks - pred))) if len(ks) else float('nan')
    print('')
    print('  K = %.4f x gain          (forced through the origin)' % k_slope)
    if np.isfinite(k_free):
        print('  K = %.4f + %.4f x gain  (free intercept)' % (k_int, k_free))
        if abs(k_int) > 0.25 * float(np.max(np.abs(ks))):
            print('  *** THE INTERCEPT IS NOT NEGLIGIBLE: %.0f%% of the largest K'
                  % (100 * abs(k_int) / float(np.max(np.abs(ks)))))
            print('  measured. K must be zero at zero gain, so something')
            print('  gain-independent is leaking into it - drift between the two')
            print('  phases, or an error in the phase axis making the pair')
            print('  cancel imperfectly. TAKE THE FREE-FIT SLOPE as the loop gain')
            print('  and add more gain points before quoting either.')
    print('  worst departure from proportional : %.3f' % lin_err)
    print('  predicted from the budget         : K = %.4f x gain' % G_FWD_RES)
    print('  extrapolated self-oscillation onset at gain %.2f' % A_ONSET)
    if not FB_DETECTED:
        print('')
        print('  DO NOT READ THE NUMBERS ABOVE AS PHYSICS. Step 8a found no effect')
        print('  above the noise, so this fit is a straight line through scatter.')
        print('  K, its slope and the onset gain are all meaningless here.')
    elif A_ONSET >= 1.99:
        print('')
        print('  The onset is beyond the largest coefficient the register can hold')
        print('  (1.99), so self-oscillation is UNREACHABLE with this coupling. That')
        print('  is the honest result for a passively coupled coil rather than a')
        print('  shortcoming of the measurement: the round trip crosses %.1f pF'
              % (C_COUPLE * 1e12))
        print('  twice and loses about %.0f dB, and no setting recovers it.'
              % abs(20 * np.log10(max(G_FWD_RES, 1e-12))))
        print('  An amplifier in the pickup path is the change that would.')
    elif A_ONSET < max(FB_GAINS_USE) * 1.3:
        print('  NOTE: that is close to the gains already used. Do not raise')
        print('  FB_GAIN_FRACS further without watching the input level.')
    print('')
    print('  NOT MEASURED ON PURPOSE: the boundary itself. On the emulator,')
    print('  bisecting into oscillation was safe and informative. Here it')
    print('  saturates the input, and a saturated input still looks vaguely like')
    print('  a resonance. Make that a separate, deliberate experiment.')
else:
    A_ONSET = float('nan')
    k_slope = float('nan')
    print('  too few usable gains to fit K')


---
# Step 8c — The gain/phase map

Herkenhoff *et al.*'s resonator modification scan: Q and detuning over feedback **gain** against
feedback **phase**, unstable region left blank, 1D cuts beneath. Theirs is 128×128; each pixel
here costs a sweep and a fit, so the default is 24×12 — the same object in minutes rather than
days.

**The grid tracks the line**, because feedback narrows the resonance by 1/(1−K) and a span chosen
for the unfed line ends up with barely one point across it.

**Self-oscillation is detected, not approached.** Before each pixel the drive is switched off and
the input read: an unstable loop produces a large signal with no drive at all. Those pixels are
drawn black, as in the paper, and the feedback is opened immediately. Whether any black appears
depends on whether the loop can reach K = 1 — and, with an external amplifier fitted, on whether
doing so is safe.


In [ ]:
# =============================================================================
#  STEP 8c - GAIN / PHASE MAP  (Herkenhoff et al., resonator modification scan)
# =============================================================================
print('=' * 68)
print('STEP 8c - mapping Q and detuning over gain and phase')
print('=' * 68)

map_phi = np.linspace(0.0, 360.0, FB_MAP_PHASES, endpoint=False)
#  The gain axis deliberately goes PAST the boundary, because the black region
#  is the point of the figure. FB_GAIN_CEIL stops just short of K = 1 by
#  construction (the closed-loop signal diverges there), so the map uses its own
#  ceiling with the oscillation check as the safety net.
#
#  UNLESS an external amplifier makes that unsafe. An oscillating loop drives the
#  DAC to full scale, and DRIVE_MAX does not bound the feedback path, so the
#  amplifier would see 10 - EXT_PAD_DB dBm.
#  Two constraints on reaching the boundary, and they pull against each other:
#    SAFETY       an oscillating loop drives the DAC to full scale, so the
#                 amplifier sees 10 - EXT_PAD_DB dBm, which must clear
#                 AMP_IN_MAX_DBM. Wants a LARGE pad.
#    REACHABILITY the round trip must reach K = 1 inside the register.
#                 Wants a SMALL pad.
_pad_needed = max(0.0, 10.0 - AMP_IN_MAX_DBM)
_gain_needed = 20.0 * np.log10(1.0 / (FB_GAIN_MAX * max(G_PASSIVE, 1e-12)))
print('SELF-OSCILLATION REGION')
print('  pad needed for safety        : %2.0f dB  (you have %.0f)'
      % (_pad_needed, EXT_PAD_DB))
print('  net gain needed to reach K=1 : %.1f dB (you have %.1f)'
      % (_gain_needed, EXT_GAIN_DB - EXT_PAD_DB))
print('  amplifier gain satisfying both : %.1f dB' % (_gain_needed + _pad_needed))
if EXT_GAIN_DB > 0.0 and EXT_GAIN_DB < _gain_needed + _pad_needed - 0.5:
    print('  -> %.1f dB short. Terminating the amplifier output in 50 ohm raises'
          % (_gain_needed + _pad_needed - EXT_GAIN_DB))
    print('     AMP_IN_MAX_DBM from -15 to +5 dBm and cuts the pad requirement by')
    print('     20 dB; the second amplifier supplies the rest.')
print('')
_osc_gain = 1.6 / max(G_FWD_RES, 1e-9)
MAP_CAN_OSCILLATE = True
if EXT_GAIN_DB > 0.0 and (10.0 - EXT_PAD_DB) > AMP_IN_MAX_DBM:
    MAP_CAN_OSCILLATE = False
    _osc_gain = FB_GAIN_CEIL
    print('NOT exploring past the oscillation boundary: an oscillating loop would')
    print('put %+.1f dBm into the amplifier against a %+.1f dBm rating. Fit at'
          % (10.0 - EXT_PAD_DB, AMP_IN_MAX_DBM))
    print('least %.0f dB of pad between OUT1 and the amplifier to allow it.'
          % (10.0 - AMP_IN_MAX_DBM))
    print('')
#  LOG-spaced gains: the thesis plots |G| in dB, so the samples must be even in
#  dB, not in amplitude. Spans 40 dB below the ceiling.
_gmax = float(min(FB_GAIN_MAX, _osc_gain))
map_gain = _gmax * np.logspace(-2.0, 0.0, FB_MAP_GAINS)
map_gain_db = 20.0 * np.log10(map_gain)
MAP_Q = np.full((FB_MAP_GAINS, FB_MAP_PHASES), np.nan)
MAP_DF = np.full((FB_MAP_GAINS, FB_MAP_PHASES), np.nan)
MAP_OSC = np.zeros((FB_MAP_GAINS, FB_MAP_PHASES), dtype=bool)

# settle + the averaged reads + the per-point level guard, which is the
# biggest single term at GUARD_NSAMP single-word reads per frequency.
_per = FB_MAP_NPTS * (FB_MAP_SETTLE + (2 * FB_MAP_NAVG + GUARD_NSAMP) * 1.0e-5) + 0.05
print('grid %d phases x %d gains = %d pixels, roughly %.0f s'
      % (FB_MAP_PHASES, FB_MAP_GAINS, FB_MAP_PHASES * FB_MAP_GAINS,
         FB_MAP_PHASES * FB_MAP_GAINS * _per))
print('gain axis %.3f to %.2f  (%.0f to %+.0f dB), K up to %.3f'
      % (map_gain[0], map_gain[-1], map_gain_db[0], map_gain_db[-1],
         map_gain[-1] * G_FWD_RES))
print('')

acq.start(DEC_USE)
ps.config_delay(F0)
_t0 = time.time()
try:
    for gi, A in enumerate(map_gain):
        f0_t, lw_t = F0, LW                      # tracker, reset each row
        for pj, phi in enumerate(map_phi):
            ps.set(A, phi)
            time.sleep(SETTLE)

            # --- is the loop oscillating? drive OFF, look for signal anyway ---
            gen.amp(0.0)
            time.sleep(0.005)
            if sc.peak_frac(GUARD_NSAMP) > FB_OSC_LEVEL:
                ps.off()                         # open the loop immediately
                MAP_OSC[gi, pj] = True
                f0_t, lw_t = F0, LW
                continue

            try:
                span = 2.5 * max(lw_t, 0.15 * LW)
                grid = f0_t + np.linspace(-span, span, FB_MAP_NPTS)
                fm, zm = sweep(grid, drive_frac=DRIVE_FRAC, navg=FB_MAP_NAVG,
                               settle=FB_MAP_SETTLE, level_guard=FB_LEVEL_GUARD)
                fit = fit_lorentzian(fm, np.abs(zm) ** 2)
                if not (0.05 * LW < fit['lw'] < 20.0 * LW):
                    raise ValueError('width out of range')
                MAP_Q[gi, pj] = fit['Q']
                MAP_DF[gi, pj] = fit['f0'] - F0
                f0_t, lw_t = fit['f0'], fit['lw']        # track for the next pixel
            except Exception:
                f0_t, lw_t = F0, LW                      # lost it - start over
        print('  gain %5.3f  (K = %.3f) : %2d fitted, %2d oscillating'
              % (A, A * G_FWD_RES, int(np.sum(np.isfinite(MAP_Q[gi]))),
                 int(np.sum(MAP_OSC[gi]))))
finally:
    ps.off()
    gen.off()

print('')
print('map complete in %.0f s' % (time.time() - _t0))
print('  Q range      : %.1f to %.1f  (no feedback: %.1f)'
      % (np.nanmin(MAP_Q), np.nanmax(MAP_Q), Q_SWEEP))
print('  detuning     : %+.2f to %+.2f kHz'
      % (np.nanmin(MAP_DF) / 1e3, np.nanmax(MAP_DF) / 1e3))
print('  oscillating  : %d of %d pixels'
      % (int(np.sum(MAP_OSC)), MAP_OSC.size))
if not np.any(MAP_OSC):
    if not MAP_CAN_OSCILLATE:
        print('  -> the gain axis was capped short of the boundary on purpose.')
    else:
        print('  -> no instability region: K = 1 needs a transfer above %.2f and'
              % (1.0 / max(FB_GAIN_MAX, 1e-9)))
        print('     you have %.3f.' % G_FWD_RES)


In [ ]:
# --- Step 8c picture: the resonator modification scan ------------------------
#  Laid out as Herkenhoff's thesis does it: phase on -180..+180, gain in dB,
#  the quality factor on a LOGARITHMIC colour scale, the detuning on a symmetric
#  one, and the unstable region left black.
_rel = ((map_phi - PHI_NARROW + 180.0) % 360.0) - 180.0
_o = np.argsort(_rel)
_Q = np.ma.masked_invalid(MAP_Q[:, _o])
_D = np.ma.masked_invalid(MAP_DF[:, _o] / 1e3)
_cq, _cd = CMAP_SEQ, CMAP_DIV
_ext = [_rel[_o][0], _rel[_o][-1], map_gain_db[0], map_gain_db[-1]]
_dlim = float(np.nanmax(np.abs(MAP_DF / 1e3))) or 1.0

# cuts: one across phase at high gain, one across gain at the narrowing phase
_cg = int(np.argmin(np.abs(map_gain - 0.8 * map_gain[-1])))
_cp = int(np.argmin(np.abs(_rel[_o])))

fig = plt.figure(figsize=(9.6, 7.4))
for k, (dat, cmap, ttl, lab, norm) in enumerate([
        (_Q, _cq, '(a)  quality factor', 'Q',
         matplotlib.colors.LogNorm(vmin=max(np.nanmin(MAP_Q), 1.0),
                                   vmax=np.nanmax(MAP_Q))),
        (_D, _cd, '(b)  detuning of the resonance', r'$\Delta\nu_\mathrm{res}$ / kHz',
         matplotlib.colors.Normalize(vmin=-_dlim, vmax=_dlim))]):
    ax = fig.add_subplot(2, 2, k + 1)
    ax.grid(False)
    im = ax.imshow(dat, origin='lower', aspect='auto', extent=_ext,
                   cmap=cmap, norm=norm, interpolation='nearest')
    ax.axhline(map_gain_db[_cg], ls='--', lw=1.0, color='w')
    ax.axvline(_rel[_o][_cp], ls='--', lw=1.0, color='w')
    ax.set_xticks([-180, -90, 0, 90, 180])
    ax.set_xlabel(r'$\arg\{A\}$ rel. to narrowing / $^\circ$')
    ax.set_ylabel(r'$|A|$ / dB')
    ax.set_title(ttl)
    fig.colorbar(im, ax=ax).set_label(lab)

    ax2 = fig.add_subplot(2, 2, k + 3)
    ax2.plot(_rel[_o], dat[_cg, :], 'o-', color=CLR_BLUE,
             label='across phase, %+.0f dB' % map_gain_db[_cg])
    ax2.set_xticks([-180, -90, 0, 90, 180])
    ax2.set_xlabel(r'$\arg\{A\}$ rel. to narrowing / $^\circ$')
    ax2.set_ylabel(lab)
    if k == 0:
        ax2.set_yscale('log')
        ax2.axhline(Q_SWEEP, ls=':', color=CLR_GREY, label='no feedback')
    else:
        ax2.axhline(0.0, ls=':', color=CLR_GREY)
    ax3 = ax2.twiny()
    ax3.plot(map_gain_db, dat[:, _cp], 's-', color=CLR_ORANGE,
             label='across gain, at the narrowing phase')
    ax3.set_xlabel(r'$|A|$ / dB')
    ax3.grid(False)
    _h1, _l1 = ax2.get_legend_handles_labels()
    _h2, _l2 = ax3.get_legend_handles_labels()
    ax2.legend(_h1 + _h2, _l1 + _l2, loc='best')
plt.show()

print('THE TWO PANELS BENEATH ARE CUTS THROUGH THE MAPS, along the dashed lines.')
print('  blue  - across PHASE at fixed gain: Q should peak at 0 (the narrowing')
print('          phase) and dip a half turn away; the detuning should cross zero')
print('          where Q peaks and be extreme a QUARTER turn from it.')
print('  amber - across GAIN at the narrowing phase: Q should rise as Q0/(1-K)')
print('          with K proportional to gain, so it runs away towards the')
print('          boundary; the detuning stays near zero, because at the')
print('          narrowing phase feedback changes damping and not frequency.')
print('Measured: Q %.0f -> %.0f (x%.1f), detuning %+.0f to %+.0f kHz.'
      % (Q_SWEEP, np.nanmax(MAP_Q), np.nanmax(MAP_Q) / Q_SWEEP,
         np.nanmin(MAP_DF) / 1e3, np.nanmax(MAP_DF) / 1e3))


---
# Step 8d — The lineshape family

Herkenhoff's Fig. 6.3 shows the resonator's **noise** spectrum at a series of feedback gains: the
line sharpens and rises with positive feedback, flattens with negative.

**The noise version is not obtainable here** — Step 4c measures the coil's Johnson noise arriving
far below the converter floor, where Herkenhoff's resonator is cryogenic with a low-noise
amplifier at the tank. The **driven** equivalent carries the same information.

**Both signs are available.** The coefficient register cannot go negative, but negative feedback
is simply the opposite phase, `A` at `φ_narrow + 180°` — exactly what the thesis's negative `G̃`
curves are.

**Each setting gets its own sweep span.** Q here spans an order of magnitude, so a fixed span puts
three points across the narrowest line and the Lorentzian stops looking like one. The span is
predicted from the Step 8b slope, then refined by a second pass around what the first found, and
the figure's limits follow: panel (a) from the widest fitted line, panel (b) zoomed onto the
narrowest. Nothing is set by hand, so the same cell works at any Q.


In [ ]:
# =============================================================================
#  STEP 8d - LINESHAPE AT A FAMILY OF FEEDBACK GAINS  (thesis Fig. 6.3)
# =============================================================================
#  Each setting is swept on ITS OWN grid. Feedback narrows the line by 1/(1-K)
#  and shifts it, so a span fixed at the unfed linewidth puts three points across
#  the narrowest curve - and three points do not look like a Lorentzian. Two
#  passes per setting: the first span is predicted from the Step 8b slope, the
#  second is set by what the first actually found. Nothing is entered by hand, so
#  the same cell works on a coil of any Q, warm or cold.
# =============================================================================
print('=' * 68)
print('STEP 8d - lineshape versus feedback gain, both signs')
print('=' * 68)
FB_8D_OK = bool(FB_8B_OK)
if not FB_8D_OK:
    print('SKIPPED - Step 8a found no usable narrowing phase, so there is no')
    print('meaningful axis to sweep along. Not a fault in this step.')

spec_f, spec_p, spec_lbl, spec_fit, spec_A = [], [], [], [], []
if FB_8D_OK:
    acq.start(DEC_USE)
    ps.config_delay(F0)
    #  K per unit gain, for predicting the first span. The Step 8b slope is the
    #  measured one; the budget's G_FWD_RES is the fallback if 8b could not fit.
    K_PER_GAIN = k_slope if np.isfinite(k_slope) and k_slope > 0 else G_FWD_RES

    def _lineshape(f_hint, lw_hint):
        """Locate on a span wide enough to catch a shifted line, then resweep
        tight around the fit. Returns (f, p, fit)."""
        out = None
        for _pass in (0, 1):
            span = FB_SPEC_HALFSPAN * max(lw_hint, 0.02 * LW)
            if _pass == 0:
                span = max(span, 1.5 * LW)     # must not miss a shifted line
            grid = f_hint + np.linspace(-span, span, FB_SPEC_NPTS)
            fs_, zs_ = sweep(grid, drive_frac=DRIVE_FRAC, navg=FB_NAVG,
                             level_guard=FB_LEVEL_GUARD)
            ft = fit_lorentzian(fs_, np.abs(zs_) ** 2)
            if not (0.005 * LW < ft['lw'] < 50.0 * LW):
                raise ValueError('width %.1f kHz is nowhere near the %.1f kHz line'
                                 % (ft['lw'] / 1e3, LW / 1e3))
            f_hint, lw_hint = ft['f0'], ft['lw']
            out = (fs_, np.abs(zs_) ** 2, ft)
        return out

    print('%8s %7s %9s %10s %11s %11s %9s'
          % ('gain/', 'A', 'predicted', 'Q', 'f_res', 'linewidth', 'K'))
    print('%8s %7s %9s %10s %11s %11s %9s'
          % ('ceiling', '', 'lw / kHz', '', 'MHz', 'kHz', ''))
    try:
        for fr in FB_SPEC_FRACS:
            A = abs(fr) * FB_GAIN_CEIL
            phi = PHI_NARROW if fr >= 0 else PHI_BROAD
            #  K = K_PER_GAIN x A, and lw = lw0 (1 - K), so the first span is
            #  already close instead of an order of magnitude too wide.
            _kp = float(np.clip(K_PER_GAIN * A * (1.0 if fr >= 0 else -1.0),
                                -8.0, 0.95))
            _lw_hint = max(LW * (1.0 - _kp), 0.02 * LW)
            ps.set(A, phi)
            time.sleep(SETTLE)
            gen.amp(0.0); time.sleep(0.005)
            if sc.peak_frac(GUARD_NSAMP) > FB_OSC_LEVEL:
                ps.off()
                print('%+8.2f %7.2f %9.1f    oscillating, skipped' % (fr, A, _lw_hint / 1e3))
                continue
            try:
                fs_, pp_, ft = _lineshape(F0, _lw_hint)
                spec_f.append(fs_); spec_p.append(pp_)
                spec_lbl.append(fr); spec_fit.append(ft); spec_A.append(A)
                print('%+8.2f %7.2f %9.1f %10.1f %11.6f %11.2f %+9.3f'
                      % (fr, A, _lw_hint / 1e3, ft['Q'], ft['f0'] / 1e6,
                         ft['lw'] / 1e3, 1.0 - Q_SWEEP / ft['Q']))
            except Exception as exc:
                print('%+8.2f %7.2f %9.1f    %s' % (fr, A, _lw_hint / 1e3, exc))
    finally:
        ps.off(); gen.off()

    if len(spec_p) < 2:
        raise RuntimeError('too few usable spectra')
    _lws = np.array([f['lw'] for f in spec_fit])
    _qs = np.array([f['Q'] for f in spec_fit])
    print('')
    print('  Q spans %.0f to %.0f  (no feedback %.0f)' % (_qs.min(), _qs.max(), Q_SWEEP))
    print('  linewidth spans %.1f to %.1f kHz - a factor of %.1f, which is why each'
          % (_lws.min() / 1e3, _lws.max() / 1e3, _lws.max() / max(_lws.min(), 1e-30)))
    print('  curve needs its own span and the figure needs two x-scales.')


In [ ]:
# --- Step 8d picture: thesis Fig. 6.3 layout, auto-zoomed -------------------
#  Orange for positive feedback, teal for negative, black for none. Panel (a) is
#  scaled by the WIDEST fitted line so nothing is cropped; panel (b) is centred
#  and scaled on the UNFED line - the one the whole family is measured against,
#  and the only curve in the set that is the coil on its own. Both limits come
#  from this run's fits, so the zoom follows the resonator into a trap or a
#  cryostat without anyone editing a number.
if not spec_p or len(spec_p) < 2:
    print('Step 8d produced no data - nothing to plot.')
else:
    _ref = None
    for _lb, _p in zip(spec_lbl, spec_p):
        if _lb == 0.0:
            _ref = float(np.median(_p))
    _ref = _ref or float(np.median(spec_p[0]))

    _lws = np.array([f['lw'] for f in spec_fit])
    _f0s = np.array([f['f0'] for f in spec_fit])
    #  The unfed curve: the fraction nearest zero, which is the setting with the
    #  feedback coefficient at zero. It is the reference the rest of the family
    #  is quoted against, so it is what the zoom is built around.
    _unfed = int(np.argmin(np.abs(np.asarray(spec_lbl, dtype=float))))
    _xa = FB_SPEC_HALFSPAN * _lws.max()
    _xb_c = _f0s[_unfed] - F0
    _xb_w = FB_SPEC_ZOOM_LW * _lws[_unfed]
    _order = sorted(range(len(spec_lbl)), key=lambda i: spec_lbl[i])

    fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.9))
    _panels = ((axes[0], -_xa, _xa, '(a)  the whole family'),
               (axes[1], _xb_c - _xb_w, _xb_c + _xb_w,
                '(b)  zoom on the unfed line, Q = %.0f, lw = %.1f kHz'
                % (spec_fit[_unfed]['Q'], _lws[_unfed] / 1e3)))
    for ax, _lo, _hi, _ttl in _panels:
        for i in _order:
            _lb = spec_lbl[i]; _ft = spec_fit[i]
            if _lb > 0:     col, lwd = '#E8820C', 1.4
            elif _lb < 0:   col, lwd = '#12A5A5', 1.4
            else:           col, lwd = 'k', 1.8
            ax.plot((spec_f[i] - F0) / 1e3,
                    10 * np.log10(np.maximum(spec_p[i], 1e-30) / _ref),
                    '-', lw=lwd, color=col,
                    label='%+5.2f    %6.0f    %10.5f    %7.1f'
                          % (_lb, _ft['Q'], _ft['f0'] / 1e6, _ft['lw'] / 1e3))
        ax.set_xlim(_lo / 1e3, _hi / 1e3)
        ax.set_xlabel(r'$\nu - \nu_\mathrm{res,0}$ / kHz')
        ax.set_ylabel('magnitude / dB')
        ax.set_title(_ttl)
    axes[0].legend(fontsize=7, loc='best', labelspacing=0.25,
                   title=r'$\tilde{G}$        Q       $\nu_\mathrm{res}$/MHz   lw/kHz')
    plt.show()

    # The same numbers as a table, because a legend is for identifying curves and
    # a table is for reading values off.
    print('')
    print('  RESONATOR MODIFICATION - each curve in the figure above')
    print('  %8s %7s %9s %8s %14s %12s %9s'
          % ('G/ceil', 'A', 'phase', 'Q', 'f_res / MHz', 'lw / kHz', 'K'))
    print('  ' + '-' * 72)
    for i in _order:
        _ft = spec_fit[i]
        print('  %+8.2f %7.2f %9s %8.1f %14.6f %12.2f %+9.3f'
              % (spec_lbl[i], spec_A[i],
                 'narrow' if spec_lbl[i] >= 0 else 'broaden',
                 _ft['Q'], _ft['f0'] / 1e6, _ft['lw'] / 1e3,
                 1.0 - Q_SWEEP / _ft['Q']))
    print('  ' + '-' * 72)
    print('  %8s %7s %9s %8.1f %14.6f %12.2f' % ('unfed', 0.0, '-', Q_SWEEP,
                                                 F0 / 1e6, LW / 1e3))
    print('')
    print('Orange is positive feedback (narrowing, taller), teal is negative')
    print('(broadening, flatter), black is none. Panel (b) is +-%.1f linewidths of'
          % FB_SPEC_ZOOM_LW)
    print('the UNFED line (%.2f kHz), which is the curve everything else is'
          % (_lws[_unfed] / 1e3))
    print('measured against; the narrowed curves stand up inside it.')
    if np.isfinite(NOISE_RATIO):
        print('This is the DRIVEN equivalent of thesis Fig. 6.3 - the noise version')
        print('needs the coil noise above the converter floor, and here it is %.0fx'
              % (1.0 / max(NOISE_RATIO, 1e-9)))
        print('below it (Step 4c).')


---
# Step 8e — How far, and how accurately, the resonance can be moved

Thesis Fig. 6.2 shows the resonance pushed aside by feedback, decoupling the ion from the
detection system.

**How far?** Feedback in quadrature with the coil's motion shifts the centre rather than the
damping, so the sweep runs at `φ_shift` — a quarter turn from the narrowing phase — and at the
opposite phase for the other direction. Reported in **linewidths**, the only fair comparison
against a resonator of different Q.

**How accurately?** The shift-versus-gain curve is fitted, inverted to request a set of target
shifts, and each one applied and re-measured. The residual is the accuracy with which a wanted
detuning can actually be set.


In [ ]:
# =============================================================================
#  STEP 8e - FREQUENCY SHIFT: RANGE AND ACCURACY
# =============================================================================
print('=' * 68)
print('STEP 8e - how far and how accurately the resonance moves')
print('=' * 68)
FB_8E_OK = bool(FB_8B_OK)
if not FB_8E_OK:
    print('SKIPPED - Step 8a found no usable narrowing phase, so there is no')
    print('meaningful axis to sweep along. Not a fault in this step.')

if FB_8E_OK:
    acq.start(DEC_USE)
    ps.config_delay(F0)
    PHI_SHIFT_OPP = (PHI_SHIFT + 180.0) % 360.0
    sh_gain, sh_df = [], []

    def _shift_at(A, phi):
        """Centre shift with the loop closed at (A, phi). Returns Hz, or None."""
        ps.set(A, phi)
        time.sleep(SETTLE)
        gen.amp(0.0); time.sleep(0.005)
        if sc.peak_frac(GUARD_NSAMP) > FB_OSC_LEVEL:
            ps.off(); return None
        fq, zq = sweep(F0 + np.linspace(-4.0, 4.0, 33) * LW, drive_frac=DRIVE_FRAC,
                       navg=FB_NAVG, level_guard=FB_LEVEL_GUARD)
        ft = fit_lorentzian(fq, np.abs(zq) ** 2)
        if not (0.02 * LW < ft['lw'] < 50.0 * LW):
            return None
        return ft['f0'] - F0

    try:
        for phi, sgn in ((PHI_SHIFT, +1), (PHI_SHIFT_OPP, -1)):
            for fr in FB_SHIFT_FRACS:
                A = fr * FB_GAIN_CEIL
                try:
                    d = _shift_at(A, phi)
                except Exception:
                    d = None
                if d is None:
                    continue
                sh_gain.append(sgn * A); sh_df.append(d)
                print('  A = %5.2f at %5.1f deg : shift %+9.1f Hz (%+.3f linewidths)'
                      % (A, phi, d, d / LW))
    finally:
        ps.off(); gen.off()

    if len(sh_df) < 4:
        raise RuntimeError('too few usable shift points')
    sh_gain = np.array(sh_gain); sh_df = np.array(sh_df)
    SHIFT_MAX = float(np.max(np.abs(sh_df)))
    _ord = np.argsort(sh_gain)
    _slope = float(np.polyfit(sh_gain, sh_df, 1)[0])
    print('')
    print('  RANGE   : %+.1f to %+.1f Hz  =  %+.2f to %+.2f linewidths'
          % (sh_df.min(), sh_df.max(), sh_df.min() / LW, sh_df.max() / LW))
    print('            %.3f%% of the centre frequency' % (100 * SHIFT_MAX / F0))
    print('  slope   : %.0f Hz per unit gain' % _slope)

    # ---- accuracy: request a shift, apply it, measure what you got --------------
    print('')
    print('  ACCURACY - requesting a shift and measuring the result')
    _targets = np.linspace(0.35, 0.85, FB_SHIFT_TESTS) * SHIFT_MAX
    _err = []
    try:
        for tgt in _targets:
            A_req = float(np.clip(abs(tgt / _slope), 0.0, FB_GAIN_CEIL))
            got = _shift_at(A_req, PHI_SHIFT if _slope > 0 else PHI_SHIFT_OPP)
            if got is None:
                print('    target %+8.1f Hz : abandoned' % tgt); continue
            _err.append(got - tgt)
            print('    target %+8.1f Hz -> A = %.2f -> got %+8.1f Hz  (error %+7.1f Hz'
                  ', %+.3f lw)' % (tgt, A_req, got, got - tgt, (got - tgt) / LW))
    finally:
        ps.off(); gen.off()

    if _err:
        SHIFT_RMS_ERR = float(np.sqrt(np.mean(np.square(_err))))
        print('')
        print('  rms error : %.1f Hz = %.4f linewidths = %.1f ppm of f0'
              % (SHIFT_RMS_ERR, SHIFT_RMS_ERR / LW, 1e6 * SHIFT_RMS_ERR / F0))
        _drift = globals().get('DRIFT_HZ_PER_MIN', float('nan'))
        if np.isfinite(_drift):
            print('  Step 7 measured the coil drifting %.1f Hz/min unaided, so a shift'
                  % abs(_drift))
            print('  can be set %s accurately as the coil holds still.'
                  % ('more' if SHIFT_RMS_ERR < abs(_drift) else 'less'))
    else:
        SHIFT_RMS_ERR = float('nan')


In [ ]:
# --- Step 8e picture --------------------------------------------------------
if globals().get('sh_df') is None or len(sh_df) < 2:
    print('Step 8d/8e produced no data - nothing to plot.')
else:
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(9.6, 4.0))
    a1.plot(sh_gain[_ord], sh_df[_ord] / 1e3, 'o-', color=CLR_BLUE)
    a1.plot(sh_gain[_ord], _slope * sh_gain[_ord] / 1e3, '--', color=CLR_GREY,
            label='%.0f Hz per unit gain' % _slope)
    a1.axhline(0.0, ls=':', color=CLR_GREY)
    a1.set_xlabel('signed feedback gain (negative = opposite phase)')
    a1.set_ylabel('centre shift / kHz')
    a1.set_title('(a)  range')
    a1.legend()

    _lwax = sh_df[_ord] / LW
    a2.plot(sh_gain[_ord], _lwax, 'o-', color=CLR_ORANGE)
    a2.axhline(0.0, ls=':', color=CLR_GREY)
    a2.set_xlabel('signed feedback gain')
    a2.set_ylabel('centre shift / linewidths')
    a2.set_title('(b)  in linewidths - the portable number')
    plt.show()

    print('Panel (b) is the fair comparison against a resonator of any Q: a shift of')
    print('one linewidth decouples as effectively at 40 MHz as at 740 kHz.')


---
# Step 9 — Close the loop on the instrument, and write down what was learned

**The self-test runs again.** If it passed at the start and passes now, nothing about the
instrument changed while the coil was being measured, so the numbers are about the coil. If it
passed then and fails now, the reason is in the board, not the physics.

**Then everything measured is written to a file**, so the next notebook does not have to be told
the answer by hand.


In [ ]:
# =============================================================================
#  STEP 9a - THE SELF-TEST, AGAIN
# =============================================================================
print('=' * 68)
print('STEP 9 - closing checks')
print('=' * 68)
print('re-running the instrument self-test...')
print('')
st1_after = instrument_selftest()
print('')
if st1_before['ok'] and st1_after['ok']:
    print('The instrument returned a known answer BEFORE and AFTER the session.')
    print('Nothing about the board changed while the coil was being measured.')
elif st1_before['ok'] and not st1_after['ok']:
    print('*** The self-test passed at the start and FAILS now. ***')
    print('Something changed during the session - a reload, a thermal problem, a')
    print('register left in an odd state. Treat every coil number below as')
    print('provisional until this passes again.')
else:
    print('*** The self-test was already failing. Nothing below is trustworthy. ***')


In [ ]:
# =============================================================================
#  STEP 9b - SUMMARY
# =============================================================================
def _g(name, default=None):
    """Fetch a result that an earlier step may have skipped."""
    v = globals().get(name, default)
    return default if v is None else v


_qa = globals().get('Q_ALT', None)

print('')
print('=' * 68)
print('BENCH COIL - SUMMARY')
print('=' * 68)
print('source                    : %s' % SOURCE_NOTE)
print('measured                  : %s' % time.strftime('%Y-%m-%d %H:%M:%S'))
print('bitstream build id        : 0x%08X' % acq.build_id())
print('')
print('centre frequency          : %.6f MHz' % (F0 / 1e6))
print('linewidth                 : %.3f kHz' % (LW / 1e3))
print('quality factor, swept     : %.1f' % Q_SWEEP)
print('quality factor, alt. dec. : %s' % ('%.1f' % _qa if _qa else 'not obtained'))
print('off-resonance floor       : %.3f of the peak amplitude' % FLOOR_FRAC)
print('')
print('CALIBRATIONS MEASURED THIS RUN (nothing below was carried over)')
print('  coarse search step      : %.1f kHz%s'
      % (SEARCH_STEP_USED / 1e3,
         '' if SEARCH_STEP_USED == SEARCH_STEP else '  (halved from %.1f)' % (SEARCH_STEP / 1e3)))
print('  amplifier gain          : %+.1f dB  (%s)'
      % (_g('AMP_GAIN_DB', float('nan')),
         'adopted' if AUTO_CALIBRATE else 'measured, not adopted'))
print('  drive window            : %.5f to %.5f' % _g('DRIVE_WINDOW', (float('nan'),) * 2))
print('  drive used              : %.5f%s'
      % (DRIVE_FRAC, '  (adopted from the window)' if _g('DRIVE_ADOPTED', False) else ''))
print('  linear at that drive    : %s' % globals().get('LINEAR_OK', 'not tested'))
print('  decimation              : R = %d (log2 = %d), clean window +-%.1f kHz'
      % (1 << DEC_USE, DEC_USE, 0.10 * FS_CLK / (1 << DEC_USE) / 1e3))
print('')
print('drift                     : %+.1f Hz/min (%.4f lw/min)'
      % (_g('DRIFT_HZ_PER_MIN', float('nan')),
         _g('DRIFT_HZ_PER_MIN', float('nan')) / LW))
print('scatter                   : %.1f Hz' % _g('SCATTER_HZ', float('nan')))
print('')
print('narrowing phase           : %.0f deg (loop-dependent, not portable)'
      % _g('PHI_NARROW', float('nan')))
print('K per unit gain           : %.3f' % _g('k_slope', float('nan')))
print('extrapolated onset gain   : %.2f' % _g('A_ONSET', float('nan')))
print('=' * 68)

# --- the agreement table: the closest thing to a known answer available here --
print('')
print('AGREEMENT BETWEEN INDEPENDENT METHODS')
print('  method            Q        vs swept')
print('  swept amplitude   %7.1f      -' % Q_SWEEP)
if _qa:
    print('  alt. decimation   %7.1f   %+6.1f%%' % (_qa, 100 * (_qa / Q_SWEEP - 1)))
_spread = [q for q in (Q_SWEEP, _qa) if q]
if len(_spread) >= 2:
    _sp = (max(_spread) - min(_spread)) / np.mean(_spread)
    print('  spread            %.1f%%' % (100 * _sp))
    print('')
    print('  %s' % ('CONSISTENT - take the swept value as the answer' if _sp < 0.15
                    else 'INCONSISTENT - do not build on these numbers yet'))
else:
    print('')
    print('  Only one method succeeded. That is the situation this notebook exists')
    print('  to avoid - fix the failing method before going further.')


In [ ]:
# =============================================================================
#  STEP 9c - WRITE THE RESULTS, AND LEAVE THE HARDWARE SAFE
# =============================================================================
results = {
    'source_note': SOURCE_NOTE,
    'measured_utc': time.strftime('%Y-%m-%dT%H:%M:%S', time.gmtime()),
    'build_id': '0x%08X' % acq.build_id(),
    'coil': {
        'f0_hz': float(F0),
        'linewidth_hz': float(LW),
        'q_swept': float(Q_SWEEP),
        'q_alt_dec': (float(_qa) if _qa else None),
        'floor_frac_fit': float(FLOOR_FRAC),
        'floor_frac_direct': float(FLOOR_DIRECT
                                   / max(np.sqrt(FIT['peak'] + FIT['base']), 1e-300)),
    },
    'calibration': {
        'auto': bool(AUTO_CALIBRATE),
        'search_step_hz': float(SEARCH_STEP_USED),
        'search_step_halvings': int(round(np.log2(SEARCH_STEP / max(SEARCH_STEP_USED, 1e-30)))),
        'amp_gain_db_measured': float(_g('AMP_GAIN_DB', float('nan'))),
        'ext_gain_db_used': float(EXT_GAIN_DB),
        'g_passive': float(G_PASSIVE),
        'drive_max': float(DRIVE_MAX),
        'drive_linear_max': float(DRIVE_LINEAR_MAX),
        'drive_window': [float(x) for x in _g('DRIVE_WINDOW', (float('nan'),) * 2)],
        'drive_adopted': bool(_g('DRIVE_ADOPTED', False)),
        'min_amp_for_fit': float(_g('MIN_AMP_FOR_FIT', float('nan'))),
    },
    'lc_crosscheck': {
        'l_coil_h': L_COIL, 'l_coil_err_h': L_COIL_ERR,
        'c_coil_f': C_COIL, 'c_coil_err_f': C_COIL_ERR,
        'c_parallel_f': C_PARALLEL, 'c_parallel_err_f': C_PARALLEL_ERR,
        'c_model_total_f': float(C_MODEL),
        'c_implied_total_f': float(C_IMPLIED),
        'c_unaccounted_f': float(C_UNACCOUNTED),
        'f0_predicted_hz': float(F0_PRED),
        'f0_predicted_err_hz': float(F0_PRED_ERR),
        'f0_deviation_sigma': float(abs(F0 - F0_PRED) / F0_PRED_ERR),
        'f0_bare_coil_hz': float(F0_BARE),
        'z0_tank_ohm': float(Z0_TANK),
        'r_series_ohm': float(R_SERIES),
        'r_parallel_ohm': float(R_PARALLEL),
        'q_bare_coil_implied': float(Q_BARE_IMPLIED),
        'g_fwd_measured': float(G_FWD),
        'g_fwd_passive': float(G_FWD_PASSIVE),
        'g_fwd_predicted': float(G_FWD_PRED),
    },
    'coupling': {
        'c_couple_f': C_COUPLE,
        'c_stray_est_f': C_STRAY_EST,
        'predicted_divider': float(C_COUPLE / (C_COUPLE + C_STRAY_EST)),
        'xc_ohm_at_f0': float(1.0 / (2 * np.pi * F0 * C_COUPLE)),
        'noise_ratio_coil_over_board': float(_g('NOISE_RATIO', float('nan'))),
    },
    'drive': {
        'frac': float(DRIVE_FRAC),
        'linear': bool(globals().get('LINEAR_OK', False)),
    },
    'receiver': {
        'dec_log2': int(DEC_USE),
        'output_rate_hz': float(FS_CLK / (1 << DEC_USE)),
        'clean_halfwidth_hz': float(0.10 * FS_CLK / (1 << DEC_USE)),
        'n_avg': N_AVG,
    },
    'stability': {
        'drift_hz_per_min': float(_g('DRIFT_HZ_PER_MIN', float('nan'))),
        'scatter_hz': float(_g('SCATTER_HZ', float('nan'))),
    },
    'feedback': {
        'phi_narrow_deg': float(_g('PHI_NARROW', float('nan'))),
        'phi_shift_deg': float(_g('PHI_SHIFT', float('nan'))),
        'k_per_gain': float(_g('k_slope', float('nan'))),
        'onset_gain_extrapolated': float(_g('A_ONSET', float('nan'))),
        'q_max_mapped': (float(np.nanmax(MAP_Q)) if 'MAP_Q' in globals() else None),
        'shift_max_hz': float(_g('SHIFT_MAX', float('nan'))),
        'shift_rms_err_hz': float(_g('SHIFT_RMS_ERR', float('nan'))),
        'note': ('phase axis is loop-dependent: cables, coil and any amplifier '
                 'shift it. Only phases RELATIVE to phi_narrow are portable.'),
    },
    'selftest': {
        'before_ok': bool(st1_before['ok']),
        'after_ok': bool(st1_after['ok']),
        'q_error_before_frac': float(st1_before['err_q_frac']),
        'q_error_after_frac': float(st1_after['err_q_frac']),
    },
}

with open(RESULTS_JSON, 'w') as fh:
    json.dump(results, fh, indent=2, sort_keys=True)
print('results written to %s' % os.path.abspath(RESULTS_JSON))

# --- leave everything in the state it was found in ---------------------------
gen.off()
ps.off()
emu.disable()
acq.stop()
print('')
print('generator off, feedback off, emulator disabled, acquisition stopped.')
print('The board is back to its power-on behaviour.')


---
# What to do with this

**If the two quality factors agree**, you have a characterised coil and `bench_coil_results.json`
holds everything a later notebook needs — Stage 10's atlas can be pointed at it with
`EMULATION = False`.

**If they disagree**, in order of how often each is the culprit: the **fitted baseline** (if the
floor in the Step 3 plot is a sizeable fraction of the peak the width is inflated — more drive or
a shorter pickup lead fixes it); **something in the band** (compare the Step 2 survey with and
without the coil); **compression**, which Step 6 tests directly; and **the drive**, where Step 4b
says which values give a trustworthy fit and raising `N_AVG` widens the window from the bottom.

**Things deliberately not done here**, and why:

| | |
|---|---|
| Walking into self-oscillation | Saturates the input, and a saturated input still looks vaguely like a resonance. Worth doing, but as its own experiment with the level watched continuously |
| Varying the coupling | The capacitors are soldered on. Step 6's ladder is the substitute; the real test needs a second pair |
| Anything needing an ion | The dip, resistive cooling, ion temperature and the phase-sensitive technique all wait for the trap |
| A servo during the scan | Step 7 says whether one is needed |

**Run the self-test cell whenever a result surprises you.** Ten seconds, no recabling, and it
separates "the coil did something interesting" from "the instrument moved".
